In [ ]:
%%bash
echo test

ps aux | grep -E 'python|jupyter|postgres|psql' | grep -v grep

pkill -f psql || true
pkill -f postgres || true

test
root          68  3.2  0.0      0     0 ?        Z    17:08   0:21 [python3] <defunct>
root          69  0.4  0.5  99384 76076 ?        S    17:08   0:03 python3 /usr/local/bin/colab-fileshim.py
root          90  0.9  1.1 412372 155356 ?       Sl   17:08   0:06 /usr/bin/python3 /usr/local/bin/jupyter-server --debug --transport="ipc" --ip=172.28.0.12 --ServerApp.token= --port=9000 --FileContentsManager.root_dir=/ --FileContentsManager.allow_hidden=True --ServerApp.log_format="|%(levelname)s|%(message)s" --ServerApp.iopub_data_rate_limit=1e10 --MappingKernelManager.root_dir=/content
root        2750 93.0  0.7 664140 103160 ?       Ssl  17:20   0:01 /usr/bin/python3 -m colab_kernel_launcher -f /root/.local/share/jupyter/runtime/kernel-9352c6ad-182c-411c-871d-e58df5cfd46c.json


In [ ]:
%%bash
set -e

# 1) PostgreSQL install
apt-get update -y
apt-get install -y postgresql postgresql-contrib

# 2) Start cluster (Colab’ta systemd yok; pg_ctlcluster ile başlatıyoruz)
pg_ctlcluster 14 main start || true

# 3) Quick health check
sudo -u postgres psql -c "select version();"
sudo -u postgres psql -c "show data_directory;"
sudo -u postgres psql -c "select now();"

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 https://cli.github.com/packages stable InRelease
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,844 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,622 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,310 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,219 kB]
Get:14 http

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
from google.colab import drive
import os
import shutil

drive.mount('/content/drive')

DATA_DIR = "/content/data"
DRIVE_DIR = "/content/drive/MyDrive/retail_dw_data"

os.makedirs(DATA_DIR, exist_ok=True)
print("CSV dosyaları Colab yerel diskine kopyalanıyor...")
shutil.copy(f"{DRIVE_DIR}/01_empty_95_off.csv", DATA_DIR)
shutil.copy(f"{DRIVE_DIR}/02_empty_95_on.csv", DATA_DIR)
print("Kopyalama tamamlandı! Dosyalar hazır.")

Mounted at /content/drive
CSV dosyaları Colab yerel diskine kopyalanıyor...
Kopyalama tamamlandı! Dosyalar hazır.


In [ ]:
%%bash
set -e

# 1) DB oluştur (varsa hata vermesin)
sudo -u postgres psql -tc "SELECT 1 FROM pg_database WHERE datname='retail_dw'" | grep -q 1 \
  || sudo -u postgres psql -c "CREATE DATABASE retail_dw;"

# 2) retail_dw içinde extension + server + schema kur (-v ON_ERROR_STOP=1 eklendi)
sudo -u postgres psql -d retail_dw -v ON_ERROR_STOP=1 <<'SQL'
-- Extensions
CREATE EXTENSION IF NOT EXISTS file_fdw;
CREATE EXTENSION IF NOT EXISTS "uuid-ossp";

-- Foreign server (csv_server)
DO $$
BEGIN
  IF NOT EXISTS (SELECT 1 FROM pg_foreign_server WHERE srvname = 'csv_server') THEN
    CREATE SERVER csv_server FOREIGN DATA WRAPPER file_fdw;
  END IF;
END $$;

-- Schemas
CREATE SCHEMA IF NOT EXISTS sa_online_retail;
CREATE SCHEMA IF NOT EXISTS sa_offline_retail;
CREATE SCHEMA IF NOT EXISTS bl_cl;
CREATE SCHEMA IF NOT EXISTS bl_3nf;
CREATE SCHEMA IF NOT EXISTS bl_dm;


SQL

CREATE DATABASE
CREATE EXTENSION
CREATE EXTENSION
DO
CREATE SCHEMA
CREATE SCHEMA
CREATE SCHEMA
CREATE SCHEMA
CREATE SCHEMA


In [ ]:
%%bash
sudo chmod o+rx /content
sudo chmod o+rx /content/data
sudo chmod o+r /content/data/01_empty_95_off.csv
sudo chmod o+r /content/data/02_empty_95_on.csv
sudo chmod o+r /content/data/03_empty_5_off.csv
sudo chmod o+r /content/data/04_empty_5_on.csv



In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

SQL

In [ ]:
%%bash
set -e
DB="retail_dw"

sudo chmod o+rx /content
sudo chmod o+rx /content/data
sudo chmod o+r /content/data/01_empty_95_off.csv
sudo chmod o+r /content/data/02_empty_95_on.csv

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

DROP FOREIGN TABLE IF EXISTS sa_offline_retail.ext_offline_retail;
DROP FOREIGN TABLE IF EXISTS sa_online_retail.ext_online_retail;

CREATE FOREIGN TABLE sa_offline_retail.ext_offline_retail (
  customer_id              VARCHAR(1000),
  gender                   VARCHAR(1000),
  marital_status           VARCHAR(1000),
  transaction_id           VARCHAR(1000),
  transaction_date         VARCHAR(1000),
  product_id               VARCHAR(1000),
  product_category         VARCHAR(1000),
  quantity                 VARCHAR(1000),
  unit_price               VARCHAR(1000),
  discount_applied         VARCHAR(1000),
  day_of_week              VARCHAR(1000),
  week_of_year             VARCHAR(1000),
  month_of_year            VARCHAR(1000),
  product_name             VARCHAR(1000),
  product_brand            VARCHAR(1000),
  product_stock            VARCHAR(1000),
  product_material         VARCHAR(1000),
  promotion_id             VARCHAR(1000),
  promotion_type           VARCHAR(1000),
  promotion_start_date     VARCHAR(1000),
  promotion_end_date       VARCHAR(1000),
  customer_zip_code        VARCHAR(1000),
  customer_city            VARCHAR(1000),
  customer_state           VARCHAR(1000),
  store_zip_code           VARCHAR(1000),
  store_city               VARCHAR(1000),
  store_state              VARCHAR(1000),
  date_of_birth            VARCHAR(1000),
  payment_method           VARCHAR(1000),
  delivery_id              VARCHAR(1000),
  delivery_type            VARCHAR(1000),
  delivery_status          VARCHAR(1000),
  shipping_partner         VARCHAR(1000),
  employee_salary          VARCHAR(1000),
  membership_date          VARCHAR(1000),
  store_location           VARCHAR(1000),
  last_purchase_date       VARCHAR(1000),
  total_sales              VARCHAR(1000),
  product_manufacture_date VARCHAR(1000),
  product_expiry_date      VARCHAR(1000),
  promotion_channel        VARCHAR(1000),
  employee_name            VARCHAR(1000),
  employee_position        VARCHAR(1000),
  employee_hire_date       VARCHAR(1000)
)
SERVER csv_server
OPTIONS (
  filename  '/content/data/01_empty_95_off.csv', -- burası silinmeli mi? Altta alter table ile zaten isim girmiyor muyuz? Aslında aşağıdaki kısım incremental olan 5 olan olmayacak mıydı?
  format    'csv',
  header    'true',
  delimiter ',',
  quote     '"',
  escape    '"'
);

CREATE FOREIGN TABLE sa_online_retail.ext_online_retail (
  customer_id              VARCHAR(1000),
  gender                   VARCHAR(1000),
  marital_status           VARCHAR(1000),
  transaction_id           VARCHAR(1000),
  transaction_date         VARCHAR(1000),
  product_id               VARCHAR(1000),
  product_category         VARCHAR(1000),
  quantity                 VARCHAR(1000),
  unit_price               VARCHAR(1000),
  discount_applied         VARCHAR(1000),
  day_of_week              VARCHAR(1000),
  week_of_year             VARCHAR(1000),
  month_of_year            VARCHAR(1000),
  product_name             VARCHAR(1000),
  product_brand            VARCHAR(1000),
  product_stock            VARCHAR(1000),
  product_material         VARCHAR(1000),
  promotion_id             VARCHAR(1000),
  promotion_type           VARCHAR(1000),
  promotion_start_date     VARCHAR(1000),
  promotion_end_date       VARCHAR(1000),
  customer_zip_code        VARCHAR(1000),
  customer_city            VARCHAR(1000),
  customer_state           VARCHAR(1000),
  customer_support_calls   VARCHAR(1000),
  date_of_birth            VARCHAR(1000),
  payment_method           VARCHAR(1000),
  delivery_id              VARCHAR(1000),
  delivery_type            VARCHAR(1000),
  delivery_status          VARCHAR(1000),
  shipping_partner         VARCHAR(1000),
  membership_date          VARCHAR(1000),
  website_address          VARCHAR(1000),
  order_channel            VARCHAR(1000),
  customer_support_method  VARCHAR(1000),
  issue_status             VARCHAR(1000),
  product_manufacture_date VARCHAR(1000),
  product_expiry_date      VARCHAR(1000),
  total_sales              VARCHAR(1000),
  promotion_channel        VARCHAR(1000),
  last_purchase_date       VARCHAR(1000),
  app_usage                VARCHAR(1000),
  website_visits           VARCHAR(1000),
  social_media_engagement  VARCHAR(1000),
  engagement_id            VARCHAR(1000)
)
SERVER csv_server
OPTIONS (
  filename  '/content/data/02_empty_95_on.csv',
  format    'csv',
  header    'true',
  delimiter ',',
  quote     '"',
  escape    '"'
);



SQL

DROP FOREIGN TABLE
DROP FOREIGN TABLE
CREATE FOREIGN TABLE
CREATE FOREIGN TABLE
DROP TABLE
CREATE TABLE
DROP TABLE
CREATE TABLE
TRUNCATE TABLE
INSERT 0 475000
TRUNCATE TABLE
INSERT 0 475000


NOTICE:  foreign table "ext_offline_retail" does not exist, skipping
NOTICE:  foreign table "ext_online_retail" does not exist, skipping
NOTICE:  table "src_offline_retail_raw" does not exist, skipping
NOTICE:  table "src_online_retail_raw" does not exist, skipping


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

/* =========================================================
   1) FOREIGN TABLE FILE POINTERS -> 95% FILES
   ========================================================= */
ALTER FOREIGN TABLE sa_offline_retail.ext_offline_retail
  OPTIONS (SET filename '/content/data/01_empty_95_off.csv');

ALTER FOREIGN TABLE sa_online_retail.ext_online_retail
  OPTIONS (SET filename '/content/data/02_empty_95_on.csv');


/* =========================================================
   2) RAW TABLES CLEAN START FOR BULK DEMO
   ========================================================= */
DROP FOREIGN TABLE IF EXISTS sa_offline_retail.src_offline_retail_raw;
CREATE TABLE sa_offline_retail.src_offline_retail_raw (
  customer_id              VARCHAR(1000),
  gender                   VARCHAR(1000),
  marital_status           VARCHAR(1000),
  transaction_id           VARCHAR(1000),
  transaction_date         VARCHAR(1000),
  product_id               VARCHAR(1000),
  product_category         VARCHAR(1000),
  quantity                 VARCHAR(1000),
  unit_price               VARCHAR(1000),
  discount_applied         VARCHAR(1000),
  day_of_week              VARCHAR(1000),
  week_of_year             VARCHAR(1000),
  month_of_year            VARCHAR(1000),
  product_name             VARCHAR(1000),
  product_brand            VARCHAR(1000),
  product_stock            VARCHAR(1000),
  product_material         VARCHAR(1000),
  promotion_id             VARCHAR(1000),
  promotion_type           VARCHAR(1000),
  promotion_start_date     VARCHAR(1000),
  promotion_end_date       VARCHAR(1000),
  customer_zip_code        VARCHAR(1000),
  customer_city            VARCHAR(1000),
  customer_state           VARCHAR(1000),
  store_zip_code           VARCHAR(1000),
  store_city               VARCHAR(1000),
  store_state              VARCHAR(1000),
  date_of_birth            VARCHAR(1000),
  payment_method           VARCHAR(1000),
  delivery_id              VARCHAR(1000),
  delivery_type            VARCHAR(1000),
  delivery_status          VARCHAR(1000),
  shipping_partner         VARCHAR(1000),
  employee_salary          VARCHAR(1000),
  membership_date          VARCHAR(1000),
  store_location           VARCHAR(1000),
  last_purchase_date       VARCHAR(1000),
  total_sales              VARCHAR(1000),
  product_manufacture_date VARCHAR(1000),
  product_expiry_date      VARCHAR(1000),
  promotion_channel        VARCHAR(1000),
  employee_name            VARCHAR(1000),
  employee_position        VARCHAR(1000),
  employee_hire_date       VARCHAR(1000),
  insert_dt                TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
  batch_id                 BIGINT,
  load_type                VARCHAR(20),
  source_file_name         VARCHAR(255),
  load_dts                 TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
  source_row_num           BIGINT
  );

DROP FOREIGN TABLE IF EXISTS sa_online_retail.src_online_retail_raw;
CREATE TABLE sa_online_retail.src_online_retail_raw (
  customer_id              VARCHAR(1000),
  gender                   VARCHAR(1000),
  marital_status           VARCHAR(1000),
  transaction_id           VARCHAR(1000),
  transaction_date         VARCHAR(1000),
  product_id               VARCHAR(1000),
  product_category         VARCHAR(1000),
  quantity                 VARCHAR(1000),
  unit_price               VARCHAR(1000),
  discount_applied         VARCHAR(1000),
  day_of_week              VARCHAR(1000),
  week_of_year             VARCHAR(1000),
  month_of_year            VARCHAR(1000),
  product_name             VARCHAR(1000),
  product_brand            VARCHAR(1000),
  product_stock            VARCHAR(1000),
  product_material         VARCHAR(1000),
  promotion_id             VARCHAR(1000),
  promotion_type           VARCHAR(1000),
  promotion_start_date     VARCHAR(1000),
  promotion_end_date       VARCHAR(1000),
  customer_zip_code        VARCHAR(1000),
  customer_city            VARCHAR(1000),
  customer_state           VARCHAR(1000),
  customer_support_calls   VARCHAR(1000),
  date_of_birth            VARCHAR(1000),
  payment_method           VARCHAR(1000),
  delivery_id              VARCHAR(1000),
  delivery_type            VARCHAR(1000),
  delivery_status          VARCHAR(1000),
  shipping_partner         VARCHAR(1000),
  membership_date          VARCHAR(1000),
  website_address          VARCHAR(1000),
  order_channel            VARCHAR(1000),
  customer_support_method  VARCHAR(1000),
  issue_status             VARCHAR(1000),
  product_manufacture_date VARCHAR(1000),
  product_expiry_date      VARCHAR(1000),
  total_sales              VARCHAR(1000),
  promotion_channel        VARCHAR(1000),
  last_purchase_date       VARCHAR(1000),
  app_usage                VARCHAR(1000),
  website_visits           VARCHAR(1000),
  social_media_engagement  VARCHAR(1000),
  engagement_id            VARCHAR(1000),
  insert_dt                TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
  batch_id                 BIGINT,
  load_type                VARCHAR(20),
  source_file_name         VARCHAR(255),
  load_dts                 TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
  source_row_num           BIGINT
  );

TRUNCATE TABLE sa_offline_retail.src_offline_retail_raw;
INSERT INTO sa_offline_retail.src_offline_retail_raw (
  customer_id, gender, marital_status, transaction_id, transaction_date,
  product_id, product_category, quantity, unit_price, discount_applied,
  day_of_week, week_of_year, month_of_year, product_name, product_brand,
  product_stock, product_material, promotion_id, promotion_type,
  promotion_start_date, promotion_end_date, customer_zip_code,
  customer_city, customer_state, store_zip_code, store_city, store_state,
  date_of_birth, payment_method, delivery_id, delivery_type,
  delivery_status, shipping_partner, employee_salary, membership_date,
  store_location, last_purchase_date, total_sales, product_manufacture_date,
  product_expiry_date, promotion_channel, employee_name, employee_position,
  employee_hire_date,
  batch_id, load_type, source_file_name, load_dts, source_row_num
)
SELECT
  customer_id, gender, marital_status, transaction_id, transaction_date,
  product_id, product_category, quantity, unit_price, discount_applied,
  day_of_week, week_of_year, month_of_year, product_name, product_brand,
  product_stock, product_material, promotion_id, promotion_type,
  promotion_start_date, promotion_end_date, customer_zip_code,
  customer_city, customer_state, store_zip_code, store_city, store_state,
  date_of_birth, payment_method, delivery_id, delivery_type,
  delivery_status, shipping_partner, employee_salary, membership_date,
  store_location, last_purchase_date, total_sales, product_manufacture_date,
  product_expiry_date, promotion_channel, employee_name, employee_position,
  employee_hire_date,
  1 AS batch_id,
  'BULK' AS load_type,
  '01_empty_95_off.csv' AS source_file_name,
  CURRENT_TIMESTAMP AS load_dts,
  ROW_NUMBER() OVER () AS source_row_num
FROM sa_offline_retail.ext_offline_retail;

TRUNCATE TABLE sa_online_retail.src_online_retail_raw;
INSERT INTO sa_online_retail.src_online_retail_raw (
  customer_id, gender, marital_status, transaction_id, transaction_date,
  product_id, product_category, quantity, unit_price, discount_applied,
  day_of_week, week_of_year, month_of_year, product_name, product_brand,
  product_stock, product_material, promotion_id, promotion_type,
  promotion_start_date, promotion_end_date, customer_zip_code,
  customer_city, customer_state, customer_support_calls, date_of_birth,
  payment_method, delivery_id, delivery_type, delivery_status,
  shipping_partner, membership_date, website_address, order_channel,
  customer_support_method, issue_status, product_manufacture_date,
  product_expiry_date, total_sales, promotion_channel, last_purchase_date,
  app_usage, website_visits, social_media_engagement, engagement_id,
  batch_id, load_type, source_file_name, load_dts, source_row_num
)
SELECT
  customer_id, gender, marital_status, transaction_id, transaction_date,
  product_id, product_category, quantity, unit_price, discount_applied,
  day_of_week, week_of_year, month_of_year, product_name, product_brand,
  product_stock, product_material, promotion_id, promotion_type,
  promotion_start_date, promotion_end_date, customer_zip_code,
  customer_city, customer_state, customer_support_calls, date_of_birth,
  payment_method, delivery_id, delivery_type, delivery_status,
  shipping_partner, membership_date, website_address, order_channel,
  customer_support_method, issue_status, product_manufacture_date,
  product_expiry_date, total_sales, promotion_channel, last_purchase_date,
  app_usage, website_visits, social_media_engagement, engagement_id,
      1 AS batch_id,
  'BULK' AS load_type,
  '02_empty_95_on.csv' AS source_file_name,
  CURRENT_TIMESTAMP AS load_dts,
  ROW_NUMBER() OVER () AS source_row_num
FROM sa_online_retail.ext_online_retail;

/* quick check */
SELECT 'offline_raw_bulk_95' AS check_name, COUNT(*) FROM sa_offline_retail.src_offline_retail_raw
UNION ALL
SELECT 'online_raw_bulk_95', COUNT(*) FROM sa_online_retail.src_online_retail_raw;

SQL

In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

/* =========================================================
   ONLINE CLEAN STAGING
   ========================================================= */

DROP TABLE IF EXISTS sa_online_retail.src_online_retail;

CREATE TABLE sa_online_retail.src_online_retail AS
SELECT
    /* ===================== CUSTOMER ===================== */
    COALESCE(NULLIF(LOWER(TRIM(customer_id)), ''), 'n.a.') AS customer_id,
    COALESCE(NULLIF(LOWER(TRIM(gender)), ''), 'n.a.') AS gender,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(marital_status),' ','_')), ''), 'n.a.') AS marital_status,

    CASE
        WHEN NULLIF(TRIM(date_of_birth), '') IS NOT NULL
        AND TRIM(date_of_birth) ~ '^\d{2}-\d{2}-\d{4}$'
        THEN TO_DATE(TRIM(date_of_birth), 'DD-MM-YYYY')
        WHEN NULLIF(TRIM(date_of_birth), '') IS NOT NULL
        AND TRIM(date_of_birth) ~ '^\d{2}/\d{2}/\d{4}$'
        THEN TO_DATE(TRIM(date_of_birth), 'DD/MM/YYYY')
    END AS birth_of_dt,

    CASE
        WHEN NULLIF(TRIM(membership_date), '') IS NOT NULL
        AND TRIM(membership_date) ~ '^\d{2}-\d{2}-\d{4}$'
        THEN TO_DATE(TRIM(membership_date), 'DD-MM-YYYY')
        WHEN NULLIF(TRIM(membership_date), '') IS NOT NULL
        AND TRIM(membership_date) ~ '^\d{2}/\d{2}/\d{4}$'
        THEN TO_DATE(TRIM(membership_date), 'DD/MM/YYYY')
    END AS membership_dt,

    CASE
        WHEN NULLIF(TRIM(last_purchase_date), '') IS NOT NULL
        AND TRIM(last_purchase_date) ~ '^\d{2}-\d{2}-\d{4} \d{2}:\d{2}$'
        THEN TO_TIMESTAMP(TRIM(last_purchase_date), 'DD-MM-YYYY HH24:MI')
        WHEN NULLIF(TRIM(last_purchase_date), '') IS NOT NULL
        AND TRIM(last_purchase_date) ~ '^\d{2}/\d{2}/\d{4} \d{2}:\d{2}$'
        THEN TO_TIMESTAMP(TRIM(last_purchase_date), 'DD/MM/YYYY HH24:MI')
    END AS last_purchase_dt,

    COALESCE(NULLIF(TRIM(customer_zip_code), ''), 'n.a.') AS customer_zip_code,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(customer_city),' ','_')), ''), 'n.a.') AS customer_city,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(customer_state),' ','_')), ''), 'n.a.') AS customer_state,


    /* ===================== TRANSACTION ===================== */
    COALESCE(NULLIF(LOWER(TRIM(transaction_id)), ''), 'n.a.') AS transaction_id,

    CASE
        WHEN NULLIF(TRIM(transaction_date), '') IS NOT NULL
        AND TRIM(transaction_date) ~ '^\d{2}-\d{2}-\d{4} \d{2}:\d{2}$'
        THEN TO_TIMESTAMP(TRIM(transaction_date), 'DD-MM-YYYY HH24:MI')
        WHEN NULLIF(TRIM(transaction_date), '') IS NOT NULL
        AND TRIM(transaction_date) ~ '^\d{2}/\d{2}/\d{4} \d{2}:\d{2}$'
        THEN TO_TIMESTAMP(TRIM(transaction_date), 'DD/MM/YYYY HH24:MI')
    END AS transaction_dt,

 /* ===================== FINANCIAL ===================== */

    CASE
      WHEN TRIM(quantity) ~ '^-?\d+$'
      THEN quantity::INT
    END  AS quantity,

      CASE
        WHEN TRIM(unit_price) ~ '^-?\d+(\.\d+)?$'
        THEN unit_price::NUMERIC(10,2)
      END AS unit_price,

    CASE
      WHEN TRIM(total_sales) ~ '^-?\d+(\.\d+)?$'
      THEN total_sales::NUMERIC(10,2)
    END AS total_sales,

    CASE
      WHEN TRIM(discount_applied) ~ '^-?\d+(\.\d+)?$'
      THEN discount_applied::NUMERIC(10,2)
    END AS discount_applied,

COALESCE(NULLIF(LOWER(REPLACE(TRIM(payment_method), ' ', '_')), ''), 'n.a.') AS payment_method,

    /* ===================== TIME ===================== */
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(day_of_week), ' ', '_')), ''), 'n.a.') AS day_of_week,

        CASE
            WHEN TRIM(week_of_year) ~ '^\d+$'
            THEN CAST(TRIM(week_of_year) AS INTEGER)
        END AS week_of_year,

        CASE
            WHEN TRIM(month_of_year) ~ '^\d+$'
            THEN CAST(TRIM(month_of_year) AS INTEGER)
        END AS month_of_year,


    /* ===================== PRODUCT ===================== */
    COALESCE(NULLIF(LOWER(TRIM(product_id)), ''), 'n.a.') AS product_id,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(product_category),' ','_')), ''), 'n.a.') AS product_category,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(product_name),' ','_')), ''), 'n.a.') AS product_name,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(product_brand),' ','_')), ''), 'n.a.') AS product_brand,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(product_material),' ','_')), ''), 'n.a.') AS product_material,

    CASE WHEN NULLIF(TRIM(product_stock), '') IS NOT NULL
         THEN product_stock::INT END AS product_stock,

    CASE
        WHEN NULLIF(TRIM(product_manufacture_date), '') IS NOT NULL
        AND TRIM(product_manufacture_date) ~ '^\d{2}-\d{2}-\d{4} \d{2}:\d{2}$'
        THEN TO_TIMESTAMP(TRIM(product_manufacture_date), 'DD-MM-YYYY HH24:MI')
        WHEN NULLIF(TRIM(product_manufacture_date), '') IS NOT NULL
        AND TRIM(product_manufacture_date) ~ '^\d{2}/\d{2}/\d{4} \d{2}:\d{2}$'
        THEN TO_TIMESTAMP(TRIM(product_manufacture_date), 'DD/MM/YYYY HH24:MI')
    END AS product_manufacture_dt,

    CASE
        WHEN NULLIF(TRIM(product_expiry_date), '') IS NOT NULL
        AND TRIM(product_expiry_date) ~ '^\d{2}-\d{2}-\d{4} \d{2}:\d{2}$'
        THEN TO_TIMESTAMP(TRIM(product_expiry_date), 'DD-MM-YYYY HH24:MI')
        WHEN NULLIF(TRIM(product_expiry_date), '') IS NOT NULL
        AND TRIM(product_expiry_date) ~ '^\d{2}/\d{2}/\d{4} \d{2}:\d{2}$'
        THEN TO_TIMESTAMP(TRIM(product_expiry_date), 'DD/MM/YYYY HH24:MI')
    END AS product_expiry_dt,

    /* ===================== PROMOTION ===================== */
    COALESCE(NULLIF(LOWER(TRIM(promotion_id)), ''), 'n.a.') AS promotion_id,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(promotion_type),' ','_')), ''), 'n.a.') AS promotion_type,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(promotion_channel),' ','_')), ''), 'n.a.') AS promotion_channel,

 CASE
    WHEN NULLIF(TRIM(promotion_start_date), '') IS NOT NULL
     AND TRIM(promotion_start_date) ~ '^\d{2}-\d{2}-\d{4} \d{2}:\d{2}$'
    THEN TO_TIMESTAMP(TRIM(promotion_start_date), 'DD-MM-YYYY HH24:MI')
    WHEN NULLIF(TRIM(promotion_start_date), '') IS NOT NULL
     AND TRIM(promotion_start_date) ~ '^\d{2}/\d{2}/\d{4} \d{2}:\d{2}$'
    THEN TO_TIMESTAMP(TRIM(promotion_start_date), 'DD/MM/YYYY HH24:MI')
END AS promotion_start_dt,

CASE
    WHEN NULLIF(TRIM(promotion_end_date), '') IS NOT NULL
     AND TRIM(promotion_end_date) ~ '^\d{2}-\d{2}-\d{4} \d{2}:\d{2}$'
    THEN TO_TIMESTAMP(TRIM(promotion_end_date), 'DD-MM-YYYY HH24:MI')
    WHEN NULLIF(TRIM(promotion_end_date), '') IS NOT NULL
     AND TRIM(promotion_end_date) ~ '^\d{2}/\d{2}/\d{4} \d{2}:\d{2}$'
    THEN TO_TIMESTAMP(TRIM(promotion_end_date), 'DD/MM/YYYY HH24:MI')
END AS promotion_end_dt,


    /* ===================== DELIVERY ===================== */
    COALESCE(NULLIF(LOWER(TRIM(delivery_id)), ''), 'n.a.') AS delivery_id,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(delivery_type), ' ', '_')), ''), 'n.a.') AS delivery_type,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(delivery_status), ' ', '_')), ''), 'n.a.') AS delivery_status,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(shipping_partner),' ','_')), ''), 'n.a.') AS shipping_partner,

   /* ===================== ENGAGEMENT ===================== */

    COALESCE(NULLIF(LOWER(TRIM(engagement_id)), ''), 'n.a.') AS engagement_id,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(website_address), ' ', '_')), ''), 'n.a.') AS website_address,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(order_channel), ' ', '_')), ''), 'n.a.') AS order_channel,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(customer_support_method), ' ', '_')), ''), 'n.a.') AS customer_support_method,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(issue_status), ' ', '_')), ''), 'n.a.') AS issue_status,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(app_usage), ' ', '_')), ''), 'n.a.') AS app_usage,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(social_media_engagement), ' ', '_')), ''), 'n.a.') AS social_media_engagement,

    /* SAFE INTEGER CAST */
    CASE
        WHEN TRIM(src.website_visits) ~ '^\d+$'
        THEN CAST(TRIM(src.website_visits) AS INTEGER)
    END AS website_visits,

    CASE
        WHEN TRIM(src.customer_support_calls) ~ '^\d+$'
        THEN CAST(TRIM(src.customer_support_calls) AS INTEGER)
    END AS customer_support_calls,

        /* ===================== META ===================== */
        insert_dt

    FROM sa_online_retail.src_online_retail_raw src;


/* =========================================================
   OFFLINE CLEAN STAGING
   ========================================================= */

DROP TABLE IF EXISTS sa_offline_retail.src_offline_retail;

CREATE TABLE sa_offline_retail.src_offline_retail AS
SELECT
    /* ===================== CUSTOMER ===================== */
    COALESCE(NULLIF(LOWER(TRIM(customer_id)), ''), 'n.a.') AS customer_id,
    COALESCE(NULLIF(LOWER(TRIM(gender)), ''), 'n.a.') AS gender,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(marital_status),' ','_')), ''), 'n.a.') AS marital_status,

    CASE
        WHEN NULLIF(TRIM(date_of_birth), '') IS NOT NULL
        AND TRIM(date_of_birth) ~ '^\d{2}-\d{2}-\d{4}$'
        THEN TO_DATE(TRIM(date_of_birth), 'DD-MM-YYYY')
        WHEN NULLIF(TRIM(date_of_birth), '') IS NOT NULL
        AND TRIM(date_of_birth) ~ '^\d{2}/\d{2}/\d{4}$'
        THEN TO_DATE(TRIM(date_of_birth), 'DD/MM/YYYY')
    END AS birth_of_dt,

    CASE
        WHEN NULLIF(TRIM(membership_date), '') IS NOT NULL
        AND TRIM(membership_date) ~ '^\d{2}-\d{2}-\d{4}$'
        THEN TO_DATE(TRIM(membership_date), 'DD-MM-YYYY')
        WHEN NULLIF(TRIM(membership_date), '') IS NOT NULL
        AND TRIM(membership_date) ~ '^\d{2}/\d{2}/\d{4}$'
        THEN TO_DATE(TRIM(membership_date), 'DD/MM/YYYY')
    END AS membership_dt,

    CASE
        WHEN NULLIF(TRIM(last_purchase_date), '') IS NOT NULL
        AND TRIM(last_purchase_date) ~ '^\d{2}-\d{2}-\d{4} \d{2}:\d{2}$'
        THEN TO_TIMESTAMP(TRIM(last_purchase_date), 'DD-MM-YYYY HH24:MI')
        WHEN NULLIF(TRIM(last_purchase_date), '') IS NOT NULL
        AND TRIM(last_purchase_date) ~ '^\d{2}/\d{2}/\d{4} \d{2}:\d{2}$'
        THEN TO_TIMESTAMP(TRIM(last_purchase_date), 'DD/MM/YYYY HH24:MI')
    END AS last_purchase_dt,

    COALESCE(NULLIF(TRIM(customer_zip_code), ''), 'n.a.') AS customer_zip_code,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(customer_city),' ','_')), ''), 'n.a.') AS customer_city,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(customer_state),' ','_')), ''), 'n.a.') AS customer_state,


    /* ===================== TRANSACTION ===================== */
    COALESCE(NULLIF(LOWER(TRIM(transaction_id)), ''), 'n.a.') AS transaction_id,

    CASE
        WHEN NULLIF(TRIM(transaction_date), '') IS NOT NULL
        AND TRIM(transaction_date) ~ '^\d{2}-\d{2}-\d{4} \d{2}:\d{2}$'
        THEN TO_TIMESTAMP(TRIM(transaction_date), 'DD-MM-YYYY HH24:MI')
        WHEN NULLIF(TRIM(transaction_date), '') IS NOT NULL
        AND TRIM(transaction_date) ~ '^\d{2}/\d{2}/\d{4} \d{2}:\d{2}$'
        THEN TO_TIMESTAMP(TRIM(transaction_date), 'DD/MM/YYYY HH24:MI')
    END AS transaction_dt,

/* ===================== FINANCIAL ===================== */

    CASE
      WHEN TRIM(quantity) ~ '^-?\d+$'
      THEN quantity::INT
    END  AS quantity,

      CASE
        WHEN TRIM(unit_price) ~ '^-?\d+(\.\d+)?$'
        THEN unit_price::NUMERIC(10,2)
      END AS unit_price,

    CASE
      WHEN TRIM(total_sales) ~ '^-?\d+(\.\d+)?$'
      THEN total_sales::NUMERIC(10,2)
    END AS total_sales,

    CASE
      WHEN TRIM(discount_applied) ~ '^-?\d+(\.\d+)?$'
      THEN discount_applied::NUMERIC(10,2)
    END AS discount_applied,

COALESCE(NULLIF(LOWER(REPLACE(TRIM(payment_method), ' ', '_')), ''), 'n.a.') AS payment_method,

   /* ===================== TIME ===================== */
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(day_of_week), ' ', '_')), ''), 'n.a.') AS day_of_week,

   CASE
    WHEN TRIM(week_of_year) ~ '^\d+$'
    THEN CAST(TRIM(week_of_year) AS INTEGER)
END AS week_of_year,

CASE
    WHEN TRIM(month_of_year) ~ '^\d+$'
    THEN CAST(TRIM(month_of_year) AS INTEGER)
END AS month_of_year,


    /* ===================== STORE ===================== */
    COALESCE(NULLIF(TRIM(store_zip_code), ''), 'n.a.') AS store_zip_code,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(store_city),' ','_')), ''), 'n.a.') AS store_city,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(store_state),' ','_')), ''), 'n.a.') AS store_state,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(store_location),' ','_')), ''), 'n.a.') AS store_location,

    /* ===================== EMPLOYEE ===================== */
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(employee_name),' ','_')), ''), 'n.a.') AS employee_name,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(employee_position),' ','_')), ''), 'n.a.') AS employee_position,

        CASE
            WHEN TRIM(employee_salary) ~ '^-?\d+(\.\d+)?$'
            THEN CAST(TRIM(employee_salary) AS NUMERIC(10,2))
        END AS employee_salary,
          CASE
              WHEN NULLIF(TRIM(employee_hire_date), '') IS NOT NULL
              AND TRIM(employee_hire_date) ~ '^\d{2}-\d{2}-\d{4}$'
              THEN TO_DATE(TRIM(employee_hire_date), 'DD-MM-YYYY')
              WHEN NULLIF(TRIM(employee_hire_date), '') IS NOT NULL
              AND TRIM(employee_hire_date) ~ '^\d{2}/\d{2}/\d{4}$'
              THEN TO_DATE(TRIM(employee_hire_date), 'DD/MM/YYYY')
          END AS employee_hire_date,

    /* ===================== PRODUCT ===================== */
    COALESCE(NULLIF(TRIM(product_id), ''), 'n.a.') AS product_id,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(product_category),' ','_')), ''), 'n.a.') AS product_category,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(product_name),' ','_')), ''), 'n.a.') AS product_name,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(product_brand),' ','_')), ''), 'n.a.') AS product_brand,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(product_material),' ','_')), ''), 'n.a.') AS product_material,

        CASE
            WHEN TRIM(product_stock) ~ '^\d+$'
            THEN CAST(TRIM(product_stock) AS INTEGER)
        END AS product_stock,
    CASE
        WHEN NULLIF(TRIM(product_manufacture_date), '') IS NOT NULL
        AND TRIM(product_manufacture_date) ~ '^\d{2}-\d{2}-\d{4} \d{2}:\d{2}$'
        THEN TO_TIMESTAMP(TRIM(product_manufacture_date), 'DD-MM-YYYY HH24:MI')
        WHEN NULLIF(TRIM(product_manufacture_date), '') IS NOT NULL
        AND TRIM(product_manufacture_date) ~ '^\d{2}/\d{2}/\d{4} \d{2}:\d{2}$'
        THEN TO_TIMESTAMP(TRIM(product_manufacture_date), 'DD/MM/YYYY HH24:MI')
    END AS product_manufacture_dt,

    CASE
        WHEN NULLIF(TRIM(product_expiry_date), '') IS NOT NULL
        AND TRIM(product_expiry_date) ~ '^\d{2}-\d{2}-\d{4} \d{2}:\d{2}$'
        THEN TO_TIMESTAMP(TRIM(product_expiry_date), 'DD-MM-YYYY HH24:MI')
        WHEN NULLIF(TRIM(product_expiry_date), '') IS NOT NULL
        AND TRIM(product_expiry_date) ~ '^\d{2}/\d{2}/\d{4} \d{2}:\d{2}$'
        THEN TO_TIMESTAMP(TRIM(product_expiry_date), 'DD/MM/YYYY HH24:MI')
    END AS product_expiry_dt,

    /* ===================== PROMOTION ===================== */
    COALESCE(NULLIF(TRIM(promotion_id), ''), 'n.a.') AS promotion_id,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(promotion_type),' ','_')), ''), 'n.a.') AS promotion_type,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(promotion_channel),' ','_')), ''), 'n.a.') AS promotion_channel,

 CASE
    WHEN NULLIF(TRIM(promotion_start_date), '') IS NOT NULL
     AND TRIM(promotion_start_date) ~ '^\d{2}-\d{2}-\d{4} \d{2}:\d{2}$'
    THEN TO_TIMESTAMP(TRIM(promotion_start_date), 'DD-MM-YYYY HH24:MI')
    WHEN NULLIF(TRIM(promotion_start_date), '') IS NOT NULL
     AND TRIM(promotion_start_date) ~ '^\d{2}/\d{2}/\d{4} \d{2}:\d{2}$'
    THEN TO_TIMESTAMP(TRIM(promotion_start_date), 'DD/MM/YYYY HH24:MI')
END AS promotion_start_dt,

CASE
    WHEN NULLIF(TRIM(promotion_end_date), '') IS NOT NULL
     AND TRIM(promotion_end_date) ~ '^\d{2}-\d{2}-\d{4} \d{2}:\d{2}$'
    THEN TO_TIMESTAMP(TRIM(promotion_end_date), 'DD-MM-YYYY HH24:MI')
    WHEN NULLIF(TRIM(promotion_end_date), '') IS NOT NULL
     AND TRIM(promotion_end_date) ~ '^\d{2}/\d{2}/\d{4} \d{2}:\d{2}$'
    THEN TO_TIMESTAMP(TRIM(promotion_end_date), 'DD/MM/YYYY HH24:MI')
END AS promotion_end_dt,

 /* ===================== DELIVERY ===================== */
    COALESCE(NULLIF(TRIM(delivery_id), ''), 'n.a.') AS delivery_id,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(delivery_type), ' ', '_')), ''), 'n.a.') AS delivery_type,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(delivery_status), ' ', '_')), ''), 'n.a.') AS delivery_status,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(shipping_partner),' ','_')), ''), 'n.a.') AS shipping_partner,


    /* ===================== META ===================== */
    insert_dt

FROM sa_offline_retail.src_offline_retail_raw;

SQL

DROP TABLE
SELECT 475000
DROP TABLE
SELECT 475000


NOTICE:  table "src_online_retail" does not exist, skipping
NOTICE:  table "src_offline_retail" does not exist, skipping


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

--GRAIN ANALYSIS / SOURCE BEHAVIOR
--TO DESIDE THE GRAIN. The source dataset is stored at transaction-line grain, where each row represents a single product line within a transaction,
-- and repeated transaction IDs indicate multi-product purchases rather than duplicate transactions.


/* =========================================================
   GRAIN ANALYSIS / SOURCE BEHAVIOR PROFILING
   Purpose:
   - understand whether the source is transaction-header or transaction-line grain
   - observe which identifiers behave as row-level unique keys
   - observe which identifiers repeat within transactions
   ========================================================= */


/* =========================================================
   1) HIGH-LEVEL UNIQUENESS PROFILE
   ========================================================= */

SELECT
    'offline' AS source_name,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT transaction_id) AS distinct_transaction_id,
    COUNT(DISTINCT transaction_dt) AS distinct_transaction_date,
    COUNT(DISTINCT product_id) AS distinct_product_id,
    COUNT(DISTINCT promotion_id) AS distinct_promotion_id,
    COUNT(DISTINCT customer_id) AS distinct_customer_id,
    COUNT(DISTINCT delivery_id) AS distinct_delivery_id
FROM sa_offline_retail.src_offline_retail
WHERE transaction_id <> 'n.a.'

UNION ALL

SELECT
    'online' AS source_name,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT transaction_id) AS distinct_transaction_id,
    COUNT(DISTINCT transaction_dt) AS distinct_transaction_date,
    COUNT(DISTINCT product_id) AS distinct_product_id,
    COUNT(DISTINCT promotion_id) AS distinct_promotion_id,
    COUNT(DISTINCT customer_id) AS distinct_customer_id,
    COUNT(DISTINCT delivery_id) AS distinct_delivery_id
FROM sa_online_retail.src_online_retail
WHERE transaction_id <> 'n.a.';


/* =========================================================
   2) DOES TRANSACTION_ID REPEAT?
   If yes, the dataset is not transaction-header grain.
   ========================================================= */

SELECT
    'offline' AS source_name,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT transaction_id) AS distinct_transaction_id,
    COUNT(*) - COUNT(DISTINCT transaction_id) AS repeated_transaction_rows
FROM sa_offline_retail.src_offline_retail
WHERE transaction_id <> 'n.a.'

UNION ALL

SELECT
    'online' AS source_name,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT transaction_id) AS distinct_transaction_id,
    COUNT(*) - COUNT(DISTINCT transaction_id) AS repeated_transaction_rows
FROM sa_online_retail.src_online_retail
WHERE transaction_id <> 'n.a.';


/* =========================================================
   3) HOW MANY TRANSACTIONS CONTAIN MULTIPLE PRODUCTS?
   Strong evidence for transaction-line grain.
   ========================================================= */

SELECT
    'offline' AS source_name,
    COUNT(*) AS transaction_count_with_multiple_products
FROM (
    SELECT transaction_id
    FROM sa_offline_retail.src_offline_retail
    WHERE transaction_id <> 'n.a.'
      AND product_id <> 'n.a.'
    GROUP BY transaction_id
    HAVING COUNT(DISTINCT product_id) > 1
) t

UNION ALL

SELECT
    'online' AS source_name,
    COUNT(*) AS transaction_count_with_multiple_products
FROM (
    SELECT transaction_id
    FROM sa_online_retail.src_online_retail
    WHERE transaction_id <> 'n.a.'
      AND product_id <> 'n.a.'
    GROUP BY transaction_id
    HAVING COUNT(DISTINCT product_id) > 1
) t;


/* =========================================================
   4) HOW MANY TRANSACTIONS CONTAIN MULTIPLE DELIVERIES?
   Helps determine whether delivery behaves at transaction level or line level.
   ========================================================= */

SELECT
    'offline' AS source_name,
    COUNT(*) AS transaction_count_with_multiple_delivery_ids
FROM (
    SELECT transaction_id
    FROM sa_offline_retail.src_offline_retail
    WHERE transaction_id <> 'n.a.'
      AND delivery_id <> 'n.a.'
    GROUP BY transaction_id
    HAVING COUNT(DISTINCT delivery_id) > 1
) t

UNION ALL

SELECT
    'online' AS source_name,
    COUNT(*) AS transaction_count_with_multiple_delivery_ids
FROM (
    SELECT transaction_id
    FROM sa_online_retail.src_online_retail
    WHERE transaction_id <> 'n.a.'
      AND delivery_id <> 'n.a.'
    GROUP BY transaction_id
    HAVING COUNT(DISTINCT delivery_id) > 1
) t;


/* =========================================================
   5) HOW MANY TRANSACTIONS CONTAIN MULTIPLE PROMOTIONS?
   Shows whether promotion is transaction-level or line-level in source behavior.
   ========================================================= */

SELECT
    'offline' AS source_name,
    COUNT(*) AS transaction_count_with_multiple_promotion_ids
FROM (
    SELECT transaction_id
    FROM sa_offline_retail.src_offline_retail
    WHERE transaction_id <> 'n.a.'
      AND promotion_id <> 'n.a.'
    GROUP BY transaction_id
    HAVING COUNT(DISTINCT promotion_id) > 1
) t

UNION ALL

SELECT
    'online' AS source_name,
    COUNT(*) AS transaction_count_with_multiple_promotion_ids
FROM (
    SELECT transaction_id
    FROM sa_online_retail.src_online_retail
    WHERE transaction_id <> 'n.a.'
      AND promotion_id <> 'n.a.'
    GROUP BY transaction_id
    HAVING COUNT(DISTINCT promotion_id) > 1
) t;


/* =========================================================
   6) ONLINE ONLY: HOW MANY TRANSACTIONS CONTAIN MULTIPLE ENGAGEMENT IDS?
   Helps determine whether engagement behaves as row-level or transaction-level.
   ========================================================= */

SELECT
    COUNT(*) AS transaction_count_with_multiple_engagement_ids
FROM (
    SELECT transaction_id
    FROM sa_online_retail.src_online_retail
    WHERE transaction_id <> 'n.a.'
      AND engagement_id <> 'n.a.'
    GROUP BY transaction_id
    HAVING COUNT(DISTINCT engagement_id) > 1
) t;


/* =========================================================
   7) SAMPLE OF REPEATED TRANSACTIONS
   Helps visually inspect line behavior inside the same transaction.
   ========================================================= */

SELECT
    'offline' AS source_name,
    transaction_id,
    COUNT(*) AS line_count,
    COUNT(DISTINCT product_id) AS distinct_product_count,
    COUNT(DISTINCT delivery_id) AS distinct_delivery_count,
    COUNT(DISTINCT promotion_id) AS distinct_promotion_count
FROM sa_offline_retail.src_offline_retail
WHERE transaction_id <> 'n.a.'
GROUP BY transaction_id
HAVING COUNT(*) > 1
ORDER BY line_count DESC, transaction_id
LIMIT 15;

SELECT
    'online' AS source_name,
    transaction_id,
    COUNT(*) AS line_count,
    COUNT(DISTINCT product_id) AS distinct_product_count,
    COUNT(DISTINCT delivery_id) AS distinct_delivery_count,
    COUNT(DISTINCT promotion_id) AS distinct_promotion_count,
    COUNT(DISTINCT engagement_id) AS distinct_engagement_count
FROM sa_online_retail.src_online_retail
WHERE transaction_id <> 'n.a.'
GROUP BY transaction_id
HAVING COUNT(*) > 1
ORDER BY line_count DESC, transaction_id
LIMIT 15;


/* =========================================================
   8) OPTIONAL INTERSECTION CHECK:
   rows where transaction_id repeats and at least one major entity also varies
   This helps explain line-level behavior in a compact way.
   ========================================================= */

SELECT
    'offline' AS source_name,
    COUNT(*) AS repeated_transactions_with_multi_product_and_multi_delivery
FROM (
    SELECT transaction_id
    FROM sa_offline_retail.src_offline_retail
    WHERE transaction_id <> 'n.a.'
    GROUP BY transaction_id
    HAVING COUNT(*) > 1
       AND COUNT(DISTINCT product_id) > 1
       AND COUNT(DISTINCT delivery_id) > 1
) t

UNION ALL

SELECT
    'online' AS source_name,
    COUNT(*) AS repeated_transactions_with_multi_product_and_multi_delivery
FROM (
    SELECT transaction_id
    FROM sa_online_retail.src_online_retail
    WHERE transaction_id <> 'n.a.'
    GROUP BY transaction_id
    HAVING COUNT(*) > 1
       AND COUNT(DISTINCT product_id) > 1
       AND COUNT(DISTINCT delivery_id) > 1
) t;

SQL



 source_name | total_rows | distinct_transaction_id | distinct_transaction_date | distinct_product_id | distinct_promotion_id | distinct_customer_id | distinct_delivery_id 
-------------+------------+-------------------------+---------------------------+---------------------+-----------------------+----------------------+----------------------
 offline     |     475000 |                  378278 |                    381983 |                9999 |                   999 |               475000 |               475000
 online      |     475000 |                  378155 |                    382343 |                9999 |                   999 |               475000 |               475000
(2 rows)

 source_name | total_rows | distinct_transaction_id | repeated_transaction_rows 
-------------+------------+-------------------------+---------------------------
 offline     |     475000 |                  378278 |                     96722
 online      |     475000 |                  378155 |     

In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

-- =========================================================
-- INTERPRETATION OF RESULT BLOCK 1: HIGH-LEVEL UNIQUENESS PROFILE
-- =========================================================
-- Both offline and online sources contain 475,000 rows, which confirms that the two channels were generated at the same scale.
-- Transaction identifiers are not unique at row level, since distinct_transaction_id is much lower than total_rows in both sources.
-- This means the dataset is not stored at transaction-header grain.
-- Product and promotion IDs are also clearly repeating across rows, which is expected because the same products and promotions are reused across many sales lines.
-- By contrast, customer_id and delivery_id are fully unique at row level in both sources, which indicates that these fields were synthetically generated almost as row-level identifiers rather than reusable business entities.
-- This is an important profiling finding: uniqueness in the raw source does not automatically mean true business-entity grain; it only describes source behavior.

-- =========================================================
-- INTERPRETATION OF RESULT BLOCK 2: REPEATED TRANSACTION IDS
-- =========================================================
-- In both channels, around 96,000 rows belong to repeated transaction IDs.
-- This confirms that the same transaction can appear on multiple rows. 475/96=5 prodycts by each transaction
-- Therefore, each row should not be interpreted as a complete transaction record.
-- Instead, the source behaves as a transaction-line dataset, where one transaction may contain multiple line items.

-- =========================================================
-- INTERPRETATION OF RESULT BLOCK 3: TRANSACTIONS WITH MULTIPLE PRODUCTS
-- =========================================================
-- More than 82,000 transactions in each source contain multiple distinct product IDs.  avg 475/96=5 prodycts by each transaction
-- This is strong evidence that the raw dataset is stored at transaction-line grain.
-- In practical terms, one purchase event may include several products, and each product is recorded on a separate line.
-- This behavior is consistent with order-line or basket-line modeling rather than transaction-header modeling.

-- =========================================================
-- INTERPRETATION OF RESULT BLOCK 4: TRANSACTIONS WITH MULTIPLE DELIVERY IDS
-- =========================================================
-- The number of transactions with multiple delivery IDs is almost the same as the number of transactions with multiple products.
-- This shows that delivery is not behaving as a transaction-level entity in the synthetic source.
-- Instead, delivery_id behaves as a line-level identifier, meaning separate delivery IDs are generated for separate rows within the same transaction.
-- For this reason, forcing a transaction-level delivery key would not reflect actual source behavior.

-- =========================================================
-- INTERPRETATION OF RESULT BLOCK 5: TRANSACTIONS WITH MULTIPLE PROMOTION IDS
-- =========================================================
-- Promotion behavior is also line-sensitive in the source.
-- More than 82,000 repeated transactions contain multiple distinct promotion IDs, which means promotions are not consistently assigned at full-transaction level.
-- In this dataset, promotion appears to vary across transaction lines, similarly to product and delivery.
-- This supports preserving source behavior first, then deciding integration rules in later layers.

-- =========================================================
-- INTERPRETATION OF RESULT BLOCK 6: ONLINE TRANSACTIONS WITH MULTIPLE ENGAGEMENT IDS
-- =========================================================
-- In the online source, the count of transactions with multiple engagement IDs is also very high and closely aligned with the multi-product pattern.
-- This indicates that engagement_id behaves as a line-level or row-level source identifier rather than a reusable transaction-level key.
-- Therefore, engagement should be interpreted according to observed source behavior, not according to a real-world expectation of one engagement per order.

-- =========================================================
-- INTERPRETATION OF RESULT BLOCK 7: SAMPLE OF REPEATED OFFLINE TRANSACTIONS
-- =========================================================
-- The sample shows repeated transaction IDs with line_count equal to the number of distinct products, deliveries, and promotions.
-- For example, some offline transactions contain 6 rows, 6 distinct products, 6 distinct deliveries, and 6 distinct promotions.
-- This is direct row-level evidence that the source is line-grain and that several identifiers vary together across lines inside the same transaction.
-- In other words, repeated transaction IDs in this dataset do not indicate duplicate errors; they indicate multi-line purchases.

-- =========================================================
-- INTERPRETATION OF RESULT BLOCK 8: SAMPLE OF REPEATED ONLINE TRANSACTIONS
-- =========================================================
-- The online sample confirms the same structural pattern as the offline source.
-- Repeated transactions contain multiple distinct products, deliveries, promotions, and engagement IDs at the same time.
-- In most examples, the counts move almost one-to-one with line_count, which means the synthetic source generates many operational identifiers at row level.
-- This reinforces the conclusion that the online dataset is also stored at transaction-line grain.

-- =========================================================
-- INTERPRETATION OF RESULT BLOCK 9: INTERSECTION CHECK
-- =========================================================
-- The intersection result shows that the repeated transactions with multiple products are effectively the same transactions that also contain multiple delivery IDs.
-- This is a very strong confirmation of line-level source behavior.
-- It means the repeated transaction pattern is not caused by accidental duplication, but by intentional multi-line generation in the synthetic dataset.
-- Therefore, for this source, transaction_id should be treated as a grouping identifier across lines, not as a unique row identifier.

-- FINAL GRAIN CONCLUSION:
-- The source dataset is stored at transaction-line grain, where each row represents a single product line within a transaction, and repeated transaction IDs reflect multi-line purchases rather than duplicate transaction records.



SQL


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CREATE SEQUENCE IF NOT EXISTS bl_cl.etl_batch_run_seq START 1;
CREATE SEQUENCE IF NOT EXISTS bl_cl.etl_step_run_seq START 1;
CREATE SEQUENCE IF NOT EXISTS bl_cl.etl_file_registry_seq START 1;

CREATE TABLE IF NOT EXISTS bl_cl.etl_batch_run (
    batch_id            BIGINT PRIMARY KEY DEFAULT nextval('bl_cl.etl_batch_run_seq'),
    pipeline_name       VARCHAR(200) NOT NULL,
    trigger_type        VARCHAR(50) NOT NULL DEFAULT 'MANUAL',   -- MANUAL / SCHEDULED
    status              VARCHAR(30) NOT NULL DEFAULT 'STARTED',  -- STARTED / SUCCESS / FAILED
    source_system       VARCHAR(100),
    initiated_by        VARCHAR(100) DEFAULT current_user,
    start_ts            TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
    end_ts              TIMESTAMP,
    file_count          INTEGER DEFAULT 0,
    total_rows_read     BIGINT DEFAULT 0,
    total_rows_loaded   BIGINT DEFAULT 0,
    error_message       TEXT
);

CREATE TABLE IF NOT EXISTS bl_cl.etl_file_registry (
    file_id             BIGINT PRIMARY KEY DEFAULT nextval('bl_cl.etl_file_registry_seq'),
    source_system       VARCHAR(100) NOT NULL,                   -- online / offline
    file_name           VARCHAR(500) NOT NULL,
    file_path           VARCHAR(1000) NOT NULL,
    file_period         DATE,                                    -- e.g. 2026-03-01 for monthly batch
    file_hash           VARCHAR(128),
    status              VARCHAR(30) NOT NULL DEFAULT 'NEW',      -- NEW / IN_PROGRESS / DONE / FAILED / SKIPPED
    arrival_ts          TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
    processed_ts        TIMESTAMP,
    batch_id            BIGINT,
    rows_read           BIGINT DEFAULT 0,
    rows_loaded         BIGINT DEFAULT 0,
    error_message       TEXT,
    CONSTRAINT uq_etl_file_registry UNIQUE (source_system, file_name)
);

CREATE TABLE IF NOT EXISTS bl_cl.etl_step_run (
    step_run_id         BIGINT PRIMARY KEY DEFAULT nextval('bl_cl.etl_step_run_seq'),
    batch_id            BIGINT NOT NULL,
    step_name           VARCHAR(200) NOT NULL,
    step_order          INTEGER,
    target_object       VARCHAR(200),
    status              VARCHAR(30) NOT NULL DEFAULT 'STARTED',  -- STARTED / SUCCESS / FAILED
    start_ts            TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
    end_ts              TIMESTAMP,
    rows_inserted       BIGINT DEFAULT 0,
    rows_updated        BIGINT DEFAULT 0,
    rows_deleted        BIGINT DEFAULT 0,
    rows_rejected       BIGINT DEFAULT 0,
    error_message       TEXT,
    CONSTRAINT fk_step_batch
        FOREIGN KEY (batch_id) REFERENCES bl_cl.etl_batch_run(batch_id)
);

SQL

In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CREATE SEQUENCE IF NOT EXISTS bl_cl.etl_log_seq
START WITH 1
INCREMENT BY 1;


CREATE TABLE IF NOT EXISTS bl_cl.etl_log (

    log_id BIGINT PRIMARY KEY
        DEFAULT nextval('bl_cl.etl_log_seq'),

    log_ts TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    log_schema_name TEXT DEFAULT 'BL_CL',
    procedure_name TEXT NOT NULL,
    table_name TEXT,
    rows_affected INTEGER DEFAULT 0,
    status VARCHAR(20) DEFAULT 'SUCCESS',
        -- SUCCESS
        -- NO_CHANGE
        -- FAILED
        -- STARTED
    log_type VARCHAR(20) DEFAULT 'INFO',
        -- INFO
        -- WARNING
        -- ERROR
    log_message TEXT,
    error_detail TEXT,
    inserted_rows INTEGER DEFAULT 0,
    updated_rows  INTEGER DEFAULT 0,
    deleted_rows  INTEGER DEFAULT 0,
    closed_rows   INTEGER DEFAULT 0,
    batch_id      BIGINT DEFAULT NULL

);


CREATE OR REPLACE FUNCTION bl_cl.log_etl_event(
    p_procedure_name TEXT,
    p_table_name TEXT,
    p_rows_affected INTEGER,
    p_status TEXT,
    p_log_message TEXT,
    p_error_detail TEXT DEFAULT NULL,
    p_log_type TEXT DEFAULT 'INFO',
    p_inserted_rows INTEGER DEFAULT 0,
    p_updated_rows  INTEGER DEFAULT 0,
    p_deleted_rows  INTEGER DEFAULT 0,
    p_closed_rows   INTEGER DEFAULT 0,
    p_batch_id      BIGINT DEFAULT NULL
)
RETURNS VOID
LANGUAGE plpgsql
AS $$
BEGIN
    INSERT INTO bl_cl.etl_log (
        procedure_name,
        table_name,
        rows_affected,
        status,
        log_message,
        error_detail,
        log_type,
        inserted_rows,
        updated_rows,
        deleted_rows,
        closed_rows,
        batch_id
    )
    VALUES (
        p_procedure_name,
        p_table_name,
        p_rows_affected,
        p_status,
        p_log_message,
        p_error_detail,
        p_log_type,
        p_inserted_rows,
        p_updated_rows,
        p_deleted_rows,
        p_closed_rows,
        p_batch_id
    );
END;
$$;

CREATE OR REPLACE VIEW bl_cl.v_last_etl_runs AS
SELECT
    procedure_name,
    table_name,
    rows_affected,
    status,
    log_ts,
    log_message
FROM bl_cl.etl_log;


SQL

CREATE SEQUENCE
CREATE TABLE
CREATE FUNCTION
CREATE VIEW


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

DROP TABLE IF EXISTS bl_cl.t_map_customers;

CREATE TABLE IF NOT EXISTS bl_cl.t_map_customers (
    customer_id        VARCHAR(20),                       -- source business key untrustable
    gender             VARCHAR(20),                       -- formatted text
    marital_status     VARCHAR(20),                       -- formatted text
    birth_of_dt        DATE,                              -- formatted from raw date_of_birth
    membership_dt      DATE,                              -- formatted from raw membership_date
    customer_zip_code  VARCHAR(30),                       -- retained for 3NF/DM
    customer_city      VARCHAR(100),                      -- retained for city resolution in 3NF
    customer_state     VARCHAR(100),                      -- retained for state-aware city resolution in 3NF
    last_purchase_dt   TIMESTAMP,                         -- formatted from raw last_purchase_date
    customer_src_id    VARCHAR(255),                      -- derived business key
    source_system      VARCHAR(100),                      -- sa_online_retail / sa_offline_retail
    source_table       VARCHAR(100),                      -- src_online_retail / src_offline_retail
    insert_dt          TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);


CREATE INDEX IF NOT EXISTS idx_t_map_customers_bk
ON bl_cl.t_map_customers (customer_id, source_system, source_table);

CREATE OR REPLACE PROCEDURE bl_cl.load_map_customers()
LANGUAGE plpgsql
AS $$
DECLARE
    v_proc        TEXT := 'bl_cl.load_map_customers';
    v_ins         INT  := 0;
    v_err_msg     TEXT;
    v_err_detail  TEXT;
    v_err_hint    TEXT;
BEGIN

    /* Clean staging'den direkt alıyoruz */
    WITH unioned_sources AS (

    /* ONLINE */
    SELECT
        COALESCE(NULLIF(src.customer_id,''),'n.a.') AS customer_id,

        COALESCE(src.gender,'n.a.') AS gender,
        COALESCE(src.marital_status,'n.a.') AS marital_status,
        src.birth_of_dt,
        src.membership_dt,
        COALESCE(src.customer_zip_code,'n.a.') AS customer_zip_code,
        COALESCE(src.customer_city,'n.a.') AS customer_city,
        COALESCE(src.customer_state,'n.a.') AS customer_state,
        src.last_purchase_dt,

        /* 🔥 CRITICAL FIX: NULL-safe src_id */
        COALESCE(src.gender,'n.a.') || '-' ||
        COALESCE(src.marital_status,'n.a.') || '-' ||
        COALESCE(src.birth_of_dt::text,'n.a.') || '-' ||
        COALESCE(src.membership_dt::text,'n.a.') || '-' ||
        COALESCE(src.customer_zip_code,'n.a.') || '-' ||
        COALESCE(src.customer_city,'n.a.') || '-' ||
        COALESCE(src.customer_state,'n.a.') AS customer_src_id,

        'sa_online_retail' AS source_system,
        'src_online_retail' AS source_table

    FROM sa_online_retail.src_online_retail src


    UNION ALL


    /* OFFLINE */
    SELECT
        COALESCE(NULLIF(src.customer_id,''),'n.a.'),

        COALESCE(src.gender,'n.a.'),
        COALESCE(src.marital_status,'n.a.'),
        src.birth_of_dt,
        src.membership_dt,
        COALESCE(src.customer_zip_code,'n.a.'),
        COALESCE(src.customer_city,'n.a.'),
        COALESCE(src.customer_state,'n.a.'),
        src.last_purchase_dt,

        COALESCE(src.gender,'n.a.') || '-' ||
        COALESCE(src.marital_status,'n.a.') || '-' ||
        COALESCE(src.birth_of_dt::text,'n.a.') || '-' ||
        COALESCE(src.membership_dt::text,'n.a.') || '-' ||
        COALESCE(src.customer_zip_code,'n.a.') || '-' ||
        COALESCE(src.customer_city,'n.a.') || '-' ||
        COALESCE(src.customer_state,'n.a.'),

        'sa_offline_retail',
        'src_offline_retail'

    FROM sa_offline_retail.src_offline_retail src
),
    /* BL_CL rule: remove repeated identical standardized rows
   DISTINCT */
    distinct_source AS (
        SELECT DISTINCT
            customer_id,
            gender,
            marital_status,
            birth_of_dt,
            membership_dt,
            customer_zip_code,
            customer_city,
            customer_state,
            last_purchase_dt,
            customer_src_id,
            source_system,
            source_table
        FROM unioned_sources WHERE customer_id <> 'n.a.'
    )
    /* Insert only rows not already present in the BL_CL map table */
    INSERT INTO bl_cl.t_map_customers (
        customer_id,
        gender,
        marital_status,
        birth_of_dt,
        membership_dt,
        customer_zip_code,
        customer_city,
        customer_state,
        last_purchase_dt,
        customer_src_id,
        source_system,
        source_table
    )
    SELECT
        cc.customer_id,
        cc.gender,
        cc.marital_status,
        cc.birth_of_dt,
        cc.membership_dt,
        cc.customer_zip_code,
        cc.customer_city,
        cc.customer_state,
        cc.last_purchase_dt,
        cc.customer_src_id,
        cc.source_system,
        cc.source_table
    FROM distinct_source cc
    WHERE NOT EXISTS (
      SELECT 1
      FROM bl_cl.t_map_customers tgt
      WHERE tgt.customer_src_id = cc.customer_src_id
        AND tgt.source_system   = cc.source_system
        AND tgt.source_table    = cc.source_table
  );

    GET DIAGNOSTICS v_ins = ROW_COUNT;

    /* Log successful or no-change load result */
    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_cl.t_map_customers',
        v_ins,
        CASE WHEN v_ins > 0 THEN 'SUCCESS' ELSE 'NO_CHANGE' END,
        'Inserted=' || v_ins ||
        '. Loaded formatted customer rows from sa_online_retail.src_online_retail and sa_offline_retail.src_offline_retail.'
    );

    RAISE NOTICE 't_map_customers completed. inserted=%', v_ins;

EXCEPTION
    WHEN OTHERS THEN
        /* Capture detailed PostgreSQL error diagnostics */
        GET STACKED DIAGNOSTICS
            v_err_detail = PG_EXCEPTION_DETAIL,
            v_err_hint   = PG_EXCEPTION_HINT;

        v_err_msg := SQLERRM;

        /* Log failure details for troubleshooting */
        PERFORM bl_cl.log_etl_event(
            v_proc,
            'bl_cl.t_map_customers',
            0,
            'FAILED',
            'Customer map load failed',
            v_err_msg || ' | DETAIL=' || COALESCE(v_err_detail, 'n.a.')
                      || ' | HINT='   || COALESCE(v_err_hint, 'n.a.'),
            'ERROR'
        );

        RAISE;
END;
$$;

SQL

DROP TABLE
CREATE TABLE
CREATE INDEX
CREATE PROCEDURE


NOTICE:  table "t_map_customers" does not exist, skipping


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CALL bl_cl.load_map_customers();


SQL

CALL


NOTICE:  t_map_customers completed. inserted=950000


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

DROP TABLE IF EXISTS bl_cl.t_map_stores;

CREATE TABLE IF NOT EXISTS bl_cl.t_map_stores (

    store_src_id        VARCHAR(255),      -- store natural key
    store_name          VARCHAR(100),

    store_zip_code      VARCHAR(30),
    store_city          VARCHAR(100),
    store_state         VARCHAR(100),
    store_location      VARCHAR(100),

    source_system       VARCHAR(100),
    source_table        VARCHAR(100),

    insert_dt           TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE INDEX IF NOT EXISTS idx_t_map_stores_src
ON bl_cl.t_map_stores (store_src_id, source_system, source_table);

CREATE OR REPLACE PROCEDURE bl_cl.load_map_stores()
LANGUAGE plpgsql
AS $$
DECLARE
    v_proc TEXT := 'bl_cl.load_map_stores';
    v_ins  INT  := 0;
    v_err_msg TEXT;
    v_err_detail TEXT;
    v_err_hint TEXT;
BEGIN

    WITH prepared_source AS (
        SELECT
            COALESCE(src.store_location, 'n.a.') || '-' ||
            COALESCE(src.store_city, 'n.a.') || '-' ||
            COALESCE(src.store_state, 'n.a.') AS store_src_id,

            src.store_location AS store_name,
            src.store_zip_code,
            src.store_city,
            src.store_state,
            src.store_location,

            'sa_offline_retail'::VARCHAR(100) AS source_system,
            'src_offline_retail'::VARCHAR(100) AS source_table
        FROM sa_offline_retail.src_offline_retail src
    ),

    distinct_source AS (
        SELECT DISTINCT
            store_src_id,
            store_name,
            store_zip_code,
            store_city,
            store_state,
            store_location,
            source_system,
            source_table
        FROM prepared_source
        WHERE store_src_id <> 'n.a., -n.a.-n.a.'
    )

    INSERT INTO bl_cl.t_map_stores (
        store_src_id,
        store_name,
        store_zip_code,
        store_city,
        store_state,
        store_location,
        source_system,
        source_table
    )
    SELECT
        s.store_src_id,
        s.store_name,
        s.store_zip_code,
        s.store_city,
        s.store_state,
        s.store_location,
        s.source_system,
        s.source_table
    FROM distinct_source s
    WHERE NOT EXISTS (
    SELECT 1
    FROM bl_cl.t_map_stores tgt
    WHERE tgt.store_src_id   = s.store_src_id
      AND tgt.source_system  = s.source_system
      AND tgt.source_table   = s.source_table
);

    GET DIAGNOSTICS v_ins = ROW_COUNT;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_cl.t_map_stores',
        v_ins,
        CASE WHEN v_ins > 0 THEN 'SUCCESS' ELSE 'NO_CHANGE' END,
        'Inserted=' || v_ins || ' store rows (full variation preserved)'
    );

    RAISE NOTICE 't_map_stores completed. inserted=%', v_ins;

EXCEPTION
WHEN OTHERS THEN

    GET STACKED DIAGNOSTICS
        v_err_detail = PG_EXCEPTION_DETAIL,
        v_err_hint   = PG_EXCEPTION_HINT;

    v_err_msg := SQLERRM;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_cl.t_map_stores',
        0,
        'FAILED',
        'Store map load failed',
        v_err_msg || ' | DETAIL=' || COALESCE(v_err_detail,'n.a.')
                  || ' | HINT='   || COALESCE(v_err_hint,'n.a.'),
        'ERROR'
    );

    RAISE;

END;
$$;

SQL

DROP TABLE
CREATE TABLE
CREATE INDEX
CREATE PROCEDURE


NOTICE:  table "t_map_stores" does not exist, skipping


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CALL bl_cl.load_map_stores();

SQL

CALL


NOTICE:  t_map_stores completed. inserted=449929


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CREATE SEQUENCE IF NOT EXISTS bl_3nf.seq_ce_state_id
START 1;

CREATE TABLE IF NOT EXISTS bl_3nf.ce_states (
    state_id        BIGINT PRIMARY KEY,
    state_src_id    VARCHAR(100) NOT NULL,
    state_name      VARCHAR(100) NOT NULL,
    source_system   VARCHAR(100) NOT NULL,
    source_table    VARCHAR(100) NOT NULL,
    insert_dt       TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    CONSTRAINT uq_ce_states_src UNIQUE (state_src_id)
);

CREATE INDEX IF NOT EXISTS idx_ce_states_src
ON bl_3nf.ce_states (state_src_id);

INSERT INTO bl_3nf.ce_states (
    state_id, state_src_id, state_name,
    source_system, source_table, insert_dt
)
SELECT
    -1, 'n.a.', 'n.a.',
    'MANUAL', 'MANUAL', NOW()
WHERE NOT EXISTS (
    SELECT 1
    FROM bl_3nf.ce_states
    WHERE state_id = -1
);

CREATE OR REPLACE PROCEDURE bl_cl.load_ce_states()
LANGUAGE plpgsql
AS $$
DECLARE
    v_proc TEXT := 'bl_cl.load_ce_states';
    v_ins INT := 0;
    v_err_msg TEXT;
    v_err_detail TEXT;
    v_err_hint TEXT;
BEGIN

/* Collect raw state candidates from customer and store mapping tables */
WITH src_states AS (

    SELECT customer_state AS state_name
    FROM bl_cl.t_map_customers

    UNION ALL

    SELECT store_state
    FROM bl_cl.t_map_stores

),

/* 3NF deduplication rule:
     keep one row per state business key */

distinct_states AS (

    SELECT DISTINCT state_name
    FROM src_states
    WHERE state_name <> 'n.a.'

)

INSERT INTO bl_3nf.ce_states (
    state_id,
    state_src_id,
    state_name,
    source_system,
    source_table
)
SELECT
    nextval('bl_3nf.seq_ce_state_id'),
    s.state_name,
    s.state_name,
    'bl_cl',
    't_map_customers + t_map_stores'
FROM distinct_states s
WHERE NOT EXISTS (
    SELECT 1
    FROM bl_3nf.ce_states tgt
    WHERE tgt.state_src_id = s.state_name
);

    GET DIAGNOSTICS v_ins = ROW_COUNT;

    /* Log successful or no-change load result */
    PERFORM bl_cl.log_etl_event(
        v_proc,
    'bl_3nf.ce_states',
        v_ins,
        CASE WHEN v_ins > 0 THEN 'SUCCESS' ELSE 'NO_CHANGE' END,
        'Inserted=' || v_ins ||
        '. Loaded formatted ce_states rows from t_map_customers + t_map_stores '
    );

    RAISE NOTICE 'ce_states completed. inserted=%', v_ins;
EXCEPTION
WHEN OTHERS THEN

    /* Capture PostgreSQL error diagnostics */
    GET STACKED DIAGNOSTICS
        v_err_detail = PG_EXCEPTION_DETAIL,
        v_err_hint   = PG_EXCEPTION_HINT;

    v_err_msg := SQLERRM;

    /* Log failure */
    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_3nf.ce_states',
        0,
        'FAILED',
        'State load failed',
        v_err_msg || ' | DETAIL=' || COALESCE(v_err_detail,'n.a.')
                  || ' | HINT='   || COALESCE(v_err_hint,'n.a.'),
        'ERROR'
    );

    RAISE;
END;
$$;

SQL

CREATE SEQUENCE
CREATE TABLE
CREATE INDEX
INSERT 0 1
CREATE PROCEDURE


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CREATE SEQUENCE IF NOT EXISTS bl_3nf.seq_ce_city_id START 1;

CREATE TABLE IF NOT EXISTS bl_3nf.ce_cities (
    city_id        BIGINT PRIMARY KEY,
    city_src_id    VARCHAR(150) NOT NULL,
    city_name      VARCHAR(100) NOT NULL,
    state_id       BIGINT NOT NULL,
    source_system  VARCHAR(100),
    source_table   VARCHAR(100),
    insert_dt      TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    CONSTRAINT fk_city_state
    FOREIGN KEY(state_id)
    REFERENCES bl_3nf.ce_states(state_id),
    CONSTRAINT uq_ce_cities_src UNIQUE (city_src_id)
);

CREATE INDEX IF NOT EXISTS idx_ce_cities_src
ON bl_3nf.ce_cities(city_src_id);

INSERT INTO bl_3nf.ce_cities (
    city_id, city_src_id, city_name, state_id,
    source_system, source_table, insert_dt
)
SELECT
    -1, 'n.a.', 'n.a.', -1,
    'MANUAL', 'MANUAL', NOW()
WHERE NOT EXISTS (
    SELECT 1
    FROM bl_3nf.ce_cities
    WHERE city_id = -1
);

CREATE OR REPLACE PROCEDURE bl_cl.load_ce_cities()
LANGUAGE plpgsql
AS $$
DECLARE
    v_proc TEXT := 'bl_cl.load_ce_cities';
    v_ins INT := 0;
    v_err_msg TEXT;
    v_err_detail TEXT;
    v_err_hint TEXT;

BEGIN

    WITH src AS (

        SELECT customer_city AS city, customer_state AS state
        FROM bl_cl.t_map_customers

        UNION ALL

        SELECT store_city, store_state
        FROM bl_cl.t_map_stores

    ),

    distinct_cities AS (

        SELECT DISTINCT
            city,
            state,
            city || '-' || state AS city_src_id
        FROM src
        WHERE city  <> 'n.a.'
          AND state <> 'n.a.'

    ),

    resolved_state AS (

        SELECT
            d.city_src_id,
            d.city,
            COALESCE(s.state_id, -1) AS state_id
        FROM distinct_cities d
        LEFT JOIN bl_3nf.ce_states s
            ON s.state_src_id = d.state

    )

INSERT INTO bl_3nf.ce_cities (
    city_id,
    city_src_id,
    city_name,
    state_id,
    source_system,
    source_table
)
SELECT
    nextval('bl_3nf.seq_ce_city_id'),
    r.city_src_id,
    r.city,
    r.state_id,
    'bl_cl',
    't_map_customers + t_map_stores'
FROM resolved_state r
WHERE NOT EXISTS (
    SELECT 1
    FROM bl_3nf.ce_cities tgt
    WHERE tgt.city_src_id = r.city_src_id
);
GET DIAGNOSTICS v_ins = ROW_COUNT;

/* Log procedure result */
PERFORM bl_cl.log_etl_event(
    v_proc,
    'bl_3nf.ce_cities',
    v_ins,
    CASE WHEN v_ins>0 THEN 'SUCCESS' ELSE 'NO_CHANGE' END,
    'Cities loaded'
);

RAISE NOTICE 'ce_cities completed. inserted=%', v_ins;

EXCEPTION
WHEN OTHERS THEN
    GET STACKED DIAGNOSTICS
        v_err_detail = PG_EXCEPTION_DETAIL,
        v_err_hint   = PG_EXCEPTION_HINT;

    v_err_msg := SQLERRM;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_3nf.ce_cities',
        0,
        'FAILED',
        'City load failed',
        v_err_msg || ' | DETAIL=' || COALESCE(v_err_detail,'n.a.')
                  || ' | HINT='   || COALESCE(v_err_hint,'n.a.'),
        'ERROR'
    );

    RAISE;
END;
$$;

SQL

CREATE SEQUENCE
CREATE TABLE
CREATE INDEX
INSERT 0 1
CREATE PROCEDURE


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CREATE SEQUENCE IF NOT EXISTS bl_3nf.seq_ce_address_id START 1;

CREATE TABLE IF NOT EXISTS bl_3nf.ce_addresses(

    address_id     BIGINT PRIMARY KEY,
    address_src_id VARCHAR(200) NOT NULL,
    zip_code       VARCHAR(30),
    city_id        BIGINT NOT NULL,
    source_system  VARCHAR(100),
    source_table   VARCHAR(100),
    insert_dt      TIMESTAMP DEFAULT CURRENT_TIMESTAMP,

CONSTRAINT fk_address_city
FOREIGN KEY(city_id)
REFERENCES bl_3nf.ce_cities(city_id),

CONSTRAINT uq_ce_addresses_src UNIQUE (address_src_id)
);


CREATE INDEX IF NOT EXISTS idx_ce_addresses_src
ON bl_3nf.ce_addresses(address_src_id);

INSERT INTO bl_3nf.ce_addresses (
    address_id, address_src_id, zip_code, city_id,
    source_system, source_table, insert_dt
)
SELECT
    -1, 'n.a.', 'n.a.', -1,
    'MANUAL', 'MANUAL', NOW()
WHERE NOT EXISTS (
    SELECT 1
    FROM bl_3nf.ce_addresses
    WHERE address_id = -1
);


CREATE OR REPLACE PROCEDURE bl_cl.load_ce_addresses()
LANGUAGE plpgsql
AS $$
DECLARE
    v_proc       TEXT := 'bl_cl.load_ce_addresses';
    v_ins        INT  := 0;
    v_err_msg    TEXT;
    v_err_detail TEXT;
    v_err_hint   TEXT;
BEGIN
    WITH src AS (

    SELECT
        customer_city AS city,
        customer_state AS state,
        customer_zip_code AS zip
    FROM bl_cl.t_map_customers

    UNION ALL

    SELECT
        store_city,
        store_state,
        store_zip_code
    FROM bl_cl.t_map_stores

),

distinct_addresses AS (
    SELECT DISTINCT
        city,
        state,
        COALESCE(zip, 'n.a.') AS zip,
        city || '-' || state AS city_src_id,
        city || '-' || state || '-' || COALESCE(zip, 'n.a.') AS address_src_id
    FROM src
    WHERE city  <> 'n.a.'
      AND state <> 'n.a.'
),

resolved_city AS (

    SELECT
        d.address_src_id,
        d.zip,
        COALESCE(c.city_id, -1) AS city_id
    FROM distinct_addresses d
    LEFT JOIN bl_3nf.ce_cities c
        ON c.city_src_id = d.city_src_id

)

INSERT INTO bl_3nf.ce_addresses (
    address_id,
    address_src_id,
    zip_code,
    city_id,
    source_system,
    source_table
)
SELECT
    nextval('bl_3nf.seq_ce_address_id'),
    r.address_src_id,
    r.zip,
    r.city_id,
    'bl_cl',
    't_map_customers + t_map_stores'
FROM resolved_city r
WHERE NOT EXISTS (
    SELECT 1
    FROM bl_3nf.ce_addresses tgt
    WHERE tgt.address_src_id = r.address_src_id
);

    GET DIAGNOSTICS v_ins = ROW_COUNT;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_3nf.ce_addresses',
        v_ins,
        CASE WHEN v_ins > 0 THEN 'SUCCESS' ELSE 'NO_CHANGE' END,
        'Inserted=' || v_ins || '. Loaded addresses from customer/store geography.'
    );

    RAISE NOTICE 'ce_addresses completed. inserted=%', v_ins;

EXCEPTION
WHEN OTHERS THEN
    GET STACKED DIAGNOSTICS
        v_err_detail = PG_EXCEPTION_DETAIL,
        v_err_hint   = PG_EXCEPTION_HINT;

    v_err_msg := SQLERRM;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_3nf.ce_addresses',
        0,
        'FAILED',
        'Address load failed',
        v_err_msg || ' | DETAIL=' || COALESCE(v_err_detail, 'n.a.')
                  || ' | HINT='   || COALESCE(v_err_hint, 'n.a.'),
        'ERROR'
    );

    RAISE;
END;
$$;

SQL

CREATE SEQUENCE
CREATE TABLE
CREATE INDEX
INSERT 0 1
CREATE PROCEDURE


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CREATE SEQUENCE IF NOT EXISTS bl_3nf.seq_ce_customer_id START 1;

CREATE TABLE IF NOT EXISTS bl_3nf.ce_customers(
    customer_id        BIGINT PRIMARY KEY,          -- surrogate key
    customer_src_id    VARCHAR(255) NOT NULL,       -- engineered stable key
    customer_id_nk     VARCHAR(100) NOT NULL,       -- raw source key (traceability)
    gender            VARCHAR(20)  NOT NULL,
    marital_status    VARCHAR(20)  NOT NULL,
    birth_of_dt       DATE         NOT NULL,
    membership_dt     DATE         NOT NULL,
    last_purchase_dt  TIMESTAMP    NOT NULL,
    address_id        BIGINT NOT NULL,
    source_system     VARCHAR(100) NOT NULL,              -- expected: bl_cl
    source_table      VARCHAR(100) NOT NULL,              -- expected: t_map_customers
    insert_dt         TIMESTAMP    NOT NULL DEFAULT CURRENT_TIMESTAMP,
    update_dt         TIMESTAMP    NOT NULL DEFAULT CURRENT_TIMESTAMP,


    CONSTRAINT fk_ce_customers_address
        FOREIGN KEY (address_id)
        REFERENCES bl_3nf.ce_addresses(address_id),
    CONSTRAINT uq_ce_customers_src UNIQUE (customer_src_id)
);

/* Performance indexes only */
CREATE INDEX IF NOT EXISTS ix_ce_customers_src_id
    ON bl_3nf.ce_customers (customer_src_id);

CREATE INDEX IF NOT EXISTS ix_ce_customers_nk
    ON bl_3nf.ce_customers (customer_id_nk);

CREATE INDEX IF NOT EXISTS ix_ce_customers_address_id
    ON bl_3nf.ce_customers (address_id);


INSERT INTO bl_3nf.ce_customers (
    customer_id, customer_src_id, customer_id_nk,
    gender, marital_status,
    birth_of_dt, membership_dt, last_purchase_dt,
    address_id,
    source_system, source_table, insert_dt, update_dt
)
SELECT
    -1, 'n.a.', 'n.a.',
    'n.a.', 'n.a.',
    DATE '1900-01-01',
    DATE '1900-01-01',
    TIMESTAMP '1900-01-01 00:00:00',
    -1,
    'MANUAL', 'MANUAL', NOW(), NOW()
WHERE NOT EXISTS (
    SELECT 1 FROM bl_3nf.ce_customers WHERE customer_id = -1
);


CREATE OR REPLACE PROCEDURE bl_cl.load_ce_customers()
LANGUAGE plpgsql
AS $$
DECLARE
    v_proc        TEXT := 'bl_cl.load_ce_customers';
    v_ins         INT  := 0;
    v_upd         INT  := 0;
    v_err_msg     TEXT;
    v_err_detail  TEXT;
    v_err_hint    TEXT;
BEGIN

    DROP TABLE IF EXISTS tmp_final_customers;

    CREATE TEMP TABLE tmp_final_customers
    ON COMMIT DROP
    AS
    WITH src_customers AS (
        SELECT
            c.customer_src_id,           -- engineered key
            c.customer_id AS customer_id_nk,  -- raw key

            c.gender,
            c.marital_status,
            c.birth_of_dt,
            c.membership_dt,
            c.last_purchase_dt,

            c.customer_city,
            c.customer_state,
            c.customer_zip_code,

            /* Address business key */
            COALESCE(c.customer_city, 'n.a.') || '-' ||
            COALESCE(c.customer_state, 'n.a.') || '-' ||
            COALESCE(c.customer_zip_code, 'n.a.') AS address_src_id

        FROM bl_cl.t_map_customers c
        WHERE c.customer_src_id <> 'n.a.'
    ),

    ranked_customers AS (
    SELECT
        s.customer_src_id,
        s.customer_id_nk,
        COALESCE(s.gender, 'n.a.') AS gender,
        COALESCE(s.marital_status, 'n.a.') AS marital_status,
        COALESCE(s.birth_of_dt, DATE '1900-01-01') AS birth_of_dt,
        COALESCE(s.membership_dt, DATE '1900-01-01') AS membership_dt,
        COALESCE(s.last_purchase_dt, TIMESTAMP '1900-01-01 00:00:00') AS last_purchase_dt,
        COALESCE(ad.address_id, -1) AS address_id,

        ROW_NUMBER() OVER (
            PARTITION BY s.customer_src_id
            ORDER BY
                s.membership_dt DESC NULLS LAST,
                s.last_purchase_dt DESC NULLS LAST,
                s.customer_id_nk DESC,
                s.address_src_id DESC
        ) AS rn

    FROM src_customers s
    LEFT JOIN bl_3nf.ce_addresses ad
        ON ad.address_src_id = s.address_src_id
)

    SELECT
        customer_src_id,
        customer_id_nk,
        gender,
        marital_status,
        birth_of_dt,
        membership_dt,
        last_purchase_dt,
        address_id,
        'bl_cl' AS source_system,
        't_map_customers' AS source_table
    FROM ranked_customers
    WHERE rn = 1;

    CREATE INDEX idx_tmp_final_customers_src
        ON tmp_final_customers (customer_src_id);

    /* ================= INSERT ================= */
    INSERT INTO bl_3nf.ce_customers (
        customer_id,
        customer_src_id,
        customer_id_nk,
        gender,
        marital_status,
        birth_of_dt,
        membership_dt,
        last_purchase_dt,
        address_id,
        source_system,
        source_table,
        insert_dt,
        update_dt
    )
    SELECT
        nextval('bl_3nf.seq_ce_customer_id'),
        f.customer_src_id,
        f.customer_id_nk,
        f.gender,
        f.marital_status,
        f.birth_of_dt,
        f.membership_dt,
        f.last_purchase_dt,
        f.address_id,
        f.source_system,
        f.source_table,
        NOW(),
        NOW()
    FROM tmp_final_customers f
    WHERE NOT EXISTS (
        SELECT 1
        FROM bl_3nf.ce_customers ce
        WHERE ce.customer_src_id = f.customer_src_id
    );

    GET DIAGNOSTICS v_ins = ROW_COUNT;

    /* ================= UPDATE (SCD TYPE 1) ================= */
    UPDATE bl_3nf.ce_customers ce
    SET
        customer_id_nk   = f.customer_id_nk,
        gender           = f.gender,
        marital_status   = f.marital_status,
        birth_of_dt      = f.birth_of_dt,
        membership_dt    = f.membership_dt,
        last_purchase_dt = f.last_purchase_dt,
        address_id       = f.address_id,
        source_system    = f.source_system,
        source_table     = f.source_table,
        update_dt        = NOW()
    FROM tmp_final_customers f
    WHERE ce.customer_src_id = f.customer_src_id
      AND (
            ce.customer_id_nk   IS DISTINCT FROM f.customer_id_nk
         OR ce.gender           IS DISTINCT FROM f.gender
         OR ce.marital_status   IS DISTINCT FROM f.marital_status
         OR ce.birth_of_dt      IS DISTINCT FROM f.birth_of_dt
         OR ce.membership_dt    IS DISTINCT FROM f.membership_dt
         OR ce.last_purchase_dt IS DISTINCT FROM f.last_purchase_dt
         OR ce.address_id       IS DISTINCT FROM f.address_id
      );

    GET DIAGNOSTICS v_upd = ROW_COUNT;

    PERFORM bl_cl.log_etl_event(
    v_proc,
    'bl_3nf.ce_customers',
    (v_ins + v_upd),
    CASE WHEN (v_ins + v_upd) > 0 THEN 'SUCCESS' ELSE 'NO_CHANGE' END,
    'Inserted=' || v_ins || ', Updated=' || v_upd ||
    '. Type 1 customer load using engineered customer_src_id.',
    NULL,
    'INFO',
    v_ins,
    v_upd,
    0,
    0,
    NULL
);

    RAISE NOTICE 'ce_customers completed. inserted=%, updated=%', v_ins, v_upd;

EXCEPTION
WHEN OTHERS THEN
    GET STACKED DIAGNOSTICS
        v_err_detail = PG_EXCEPTION_DETAIL,
        v_err_hint   = PG_EXCEPTION_HINT;

    v_err_msg := SQLERRM;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_3nf.ce_customers',
        0,
        'FAILED',
        'ce_customers load failed',
        v_err_msg || ' | DETAIL=' || COALESCE(v_err_detail, 'n.a.')
                  || ' | HINT='   || COALESCE(v_err_hint, 'n.a.'),
        'ERROR'
    );

    RAISE;
END;
$$;

SQL

CREATE SEQUENCE
CREATE TABLE
CREATE INDEX
CREATE INDEX
CREATE INDEX
INSERT 0 0
CREATE PROCEDURE


NOTICE:  relation "seq_ce_customer_id" already exists, skipping
NOTICE:  relation "ce_customers" already exists, skipping
NOTICE:  relation "ix_ce_customers_src_id" already exists, skipping
NOTICE:  relation "ix_ce_customers_nk" already exists, skipping
NOTICE:  relation "ix_ce_customers_address_id" already exists, skipping


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CALL bl_cl.load_ce_states();

SQL

CALL


NOTICE:  ce_states completed. inserted=3


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CALL bl_cl.load_ce_cities();

SQL

CALL


NOTICE:  ce_cities completed. inserted=12


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CALL bl_cl.load_ce_addresses();

SQL

CALL


NOTICE:  ce_addresses completed. inserted=791330


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CALL bl_cl.load_ce_customers();

SQL

CALL


NOTICE:  table "tmp_final_customers" does not exist, skipping
NOTICE:  ce_customers completed. inserted=949746, updated=0


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CREATE SEQUENCE IF NOT EXISTS bl_dm.seq_dim_customer_id START 1;

CREATE TABLE IF NOT EXISTS bl_dm.dim_customers (
    customer_surr_id     BIGINT PRIMARY KEY,                 -- DM surrogate key
    customer_src_id      BIGINT NOT NULL,                    -- 3NF surrogate: ce_customers.customer_id

    source_system        VARCHAR(100) NOT NULL,
    source_table         VARCHAR(100) NOT NULL,

    customer_id_nk       VARCHAR(100) NOT NULL,              -- raw trace key from 3NF
    gender               VARCHAR(20)  NOT NULL,
    marital_status       VARCHAR(20)  NOT NULL,
    birth_of_dt          DATE         NOT NULL,
    membership_dt        DATE         NOT NULL,
    last_purchase_dt     TIMESTAMP    NOT NULL,

    customer_zip_code    VARCHAR(30)  NOT NULL,
    customer_city        VARCHAR(100) NOT NULL,
    customer_state       VARCHAR(100) NOT NULL,

    insert_dt            TIMESTAMP    NOT NULL DEFAULT CURRENT_TIMESTAMP,
    update_dt            TIMESTAMP    NOT NULL DEFAULT CURRENT_TIMESTAMP
);

CREATE UNIQUE INDEX IF NOT EXISTS ux_dim_customers_src_id
    ON bl_dm.dim_customers (customer_src_id);

CREATE INDEX IF NOT EXISTS ix_dim_customers_nk
    ON bl_dm.dim_customers (customer_id_nk);

CREATE INDEX IF NOT EXISTS ix_dim_customers_city_state
    ON bl_dm.dim_customers (customer_city, customer_state);

CREATE INDEX IF NOT EXISTS ix_dim_customers_membership_dt
    ON bl_dm.dim_customers (membership_dt);

INSERT INTO bl_dm.dim_customers (
    customer_surr_id,
    customer_src_id,
    source_system,
    source_table,
    customer_id_nk,
    gender,
    marital_status,
    birth_of_dt,
    membership_dt,
    last_purchase_dt,
    customer_zip_code,
    customer_city,
    customer_state,
    insert_dt,
    update_dt
)
SELECT
    -1,
    -1,
    'MANUAL',
    'MANUAL',
    'n.a.',
    'n.a.',
    'n.a.',
    DATE '1900-01-01',
    DATE '1900-01-01',
    TIMESTAMP '1900-01-01 00:00:00',
    'n.a.',
    'n.a.',
    'n.a.',
    NOW(),
    NOW()
WHERE NOT EXISTS (
    SELECT 1
    FROM bl_dm.dim_customers
    WHERE customer_surr_id = -1
);

CREATE OR REPLACE PROCEDURE bl_cl.load_dim_customers()
LANGUAGE plpgsql
AS $$
DECLARE
    v_proc        TEXT := 'bl_cl.load_dim_customers';
    v_ins         INT  := 0;
    v_upd         INT  := 0;
    v_err_msg     TEXT;
    v_err_detail  TEXT;
    v_err_hint    TEXT;
BEGIN
    DROP TABLE IF EXISTS tmp_dim_customers;

    CREATE TEMP TABLE tmp_dim_customers
    ON COMMIT DROP
    AS
    SELECT
        c.customer_id AS customer_src_id,
        c.customer_id_nk,
        c.gender,
        c.marital_status,
        c.birth_of_dt,
        c.membership_dt,
        c.last_purchase_dt,
        COALESCE(a.zip_code, 'n.a.') AS customer_zip_code,
        COALESCE(ci.city_name, 'n.a.') AS customer_city,
        COALESCE(st.state_name, 'n.a.') AS customer_state,
        'bl_3nf'::VARCHAR(100) AS source_system,
        'ce_customers'::VARCHAR(100) AS source_table
    FROM bl_3nf.ce_customers c
    LEFT JOIN bl_3nf.ce_addresses a
        ON a.address_id = c.address_id
    LEFT JOIN bl_3nf.ce_cities ci
        ON ci.city_id = a.city_id
    LEFT JOIN bl_3nf.ce_states st
        ON st.state_id = ci.state_id
    WHERE c.customer_id <> -1;

    CREATE INDEX idx_tmp_dim_customers_src
        ON tmp_dim_customers (customer_src_id);

    /* INSERT */
    INSERT INTO bl_dm.dim_customers (
        customer_surr_id,
        customer_src_id,
        source_system,
        source_table,
        customer_id_nk,
        gender,
        marital_status,
        birth_of_dt,
        membership_dt,
        last_purchase_dt,
        customer_zip_code,
        customer_city,
        customer_state,
        insert_dt,
        update_dt
    )
    SELECT
        nextval('bl_dm.seq_dim_customer_id'),
        t.customer_src_id,
        t.source_system,
        t.source_table,
        t.customer_id_nk,
        t.gender,
        t.marital_status,
        t.birth_of_dt,
        t.membership_dt,
        t.last_purchase_dt,
        t.customer_zip_code,
        t.customer_city,
        t.customer_state,
        NOW(),
        NOW()
    FROM tmp_dim_customers t
    WHERE NOT EXISTS (
        SELECT 1
        FROM bl_dm.dim_customers d
        WHERE d.customer_src_id = t.customer_src_id
    );

    GET DIAGNOSTICS v_ins = ROW_COUNT;

    /* UPDATE TYPE 1 */
    UPDATE bl_dm.dim_customers d
    SET
        source_system     = t.source_system,
        source_table      = t.source_table,
        customer_id_nk    = t.customer_id_nk,
        gender            = t.gender,
        marital_status    = t.marital_status,
        birth_of_dt       = t.birth_of_dt,
        membership_dt     = t.membership_dt,
        last_purchase_dt  = t.last_purchase_dt,
        customer_zip_code = t.customer_zip_code,
        customer_city     = t.customer_city,
        customer_state    = t.customer_state,
        update_dt         = NOW()
    FROM tmp_dim_customers t
    WHERE d.customer_src_id = t.customer_src_id
      AND (
            d.source_system     IS DISTINCT FROM t.source_system
         OR d.source_table      IS DISTINCT FROM t.source_table
         OR d.customer_id_nk    IS DISTINCT FROM t.customer_id_nk
         OR d.gender            IS DISTINCT FROM t.gender
         OR d.marital_status    IS DISTINCT FROM t.marital_status
         OR d.birth_of_dt       IS DISTINCT FROM t.birth_of_dt
         OR d.membership_dt     IS DISTINCT FROM t.membership_dt
         OR d.last_purchase_dt  IS DISTINCT FROM t.last_purchase_dt
         OR d.customer_zip_code IS DISTINCT FROM t.customer_zip_code
         OR d.customer_city     IS DISTINCT FROM t.customer_city
         OR d.customer_state    IS DISTINCT FROM t.customer_state
      );

    GET DIAGNOSTICS v_upd = ROW_COUNT;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_dm.dim_customers',
        v_ins + v_upd,
        CASE WHEN (v_ins + v_upd) > 0 THEN 'SUCCESS' ELSE 'NO_CHANGE' END,
        'Inserted=' || v_ins || ', Updated=' || v_upd ||
        '. Source=bl_3nf.ce_customers + ce_addresses + ce_cities + ce_states.',
        NULL,
        'INFO',
        v_ins,
        v_upd,
        0,
        0,
        NULL
    );

    RAISE NOTICE 'dim_customers completed. inserted=%, updated=%', v_ins, v_upd;

EXCEPTION
WHEN OTHERS THEN
    GET STACKED DIAGNOSTICS
        v_err_detail = PG_EXCEPTION_DETAIL,
        v_err_hint   = PG_EXCEPTION_HINT;

    v_err_msg := SQLERRM;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_dm.dim_customers',
        0,
        'FAILED',
        'dim_customers load failed',
        v_err_msg || ' | DETAIL=' || COALESCE(v_err_detail, 'n.a.')
                  || ' | HINT='   || COALESCE(v_err_hint, 'n.a.'),
        'ERROR',
        0,
        0,
        0,
        0,
        NULL
    );

    RAISE;
END;
$$;


SQL

CREATE SEQUENCE
CREATE TABLE
CREATE INDEX
CREATE INDEX
CREATE INDEX
CREATE INDEX
INSERT 0 1
CREATE PROCEDURE


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'
CALL bl_cl.load_dim_customers();

SQL

CALL


NOTICE:  table "tmp_dim_customers" does not exist, skipping
NOTICE:  dim_customers completed. inserted=949746, updated=0


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CREATE SEQUENCE IF NOT EXISTS bl_3nf.seq_ce_store_id START 1;

CREATE TABLE IF NOT EXISTS bl_3nf.ce_stores (

    store_id        BIGINT PRIMARY KEY,

    store_src_id    VARCHAR(255) NOT NULL,
    store_name      VARCHAR(100),

    address_id      BIGINT NOT NULL,

    source_system   VARCHAR(100),
    source_table    VARCHAR(100),

    insert_dt       TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    update_dt       TIMESTAMP DEFAULT CURRENT_TIMESTAMP,

CONSTRAINT fk_store_address
FOREIGN KEY (address_id)
REFERENCES bl_3nf.ce_addresses(address_id),
CONSTRAINT uq_ce_stores_src
    UNIQUE (store_src_id)

);

CREATE INDEX IF NOT EXISTS idx_ce_stores_src
ON bl_3nf.ce_stores(store_src_id);

INSERT INTO bl_3nf.ce_stores (
    store_id, store_src_id, store_name, address_id,
    source_system, source_table, insert_dt, update_dt
)
SELECT
    -1, 'n.a.', 'n.a.', -1,
    'MANUAL', 'MANUAL', NOW(), NOW()
WHERE NOT EXISTS (
    SELECT 1
    FROM bl_3nf.ce_stores
    WHERE store_id = -1
);


CREATE OR REPLACE PROCEDURE bl_cl.load_ce_stores()
LANGUAGE plpgsql
AS $$
DECLARE
    v_proc        TEXT := 'bl_cl.load_ce_stores';
    v_ins         INT  := 0;
    v_upd         INT  := 0;
    v_err_msg     TEXT;
    v_err_detail  TEXT;
    v_err_hint    TEXT;
BEGIN
    -- Safe rerun in same session
    DROP TABLE IF EXISTS tmp_final_stores;

    -- Build one integrated store row per store business key
    CREATE TEMP TABLE tmp_final_stores
    ON COMMIT DROP
    AS
    WITH src_stores AS (

        SELECT
            s.store_src_id,                 -- already engineered in t_map_stores
            s.store_name,

            -- Address business key (already clean in t_map)
            s.store_city || '-' ||
            s.store_state || '-' ||
            s.store_zip_code AS address_src_id --just for matching

        FROM bl_cl.t_map_stores s
        WHERE s.store_src_id <> 'n.a.'
    ),

    resolved_address AS (

        SELECT
            ss.store_src_id,
            ss.store_name,
            COALESCE(a.address_id, -1) AS address_id

        FROM src_stores ss
        LEFT JOIN bl_3nf.ce_addresses a
            ON a.address_src_id = ss.address_src_id
    ),

    ranked_stores AS (

        SELECT
            r.*,
            ROW_NUMBER() OVER (
                PARTITION BY r.store_src_id
                ORDER BY r.address_id DESC,
                         r.store_name
            ) AS rn
        FROM resolved_address r
    )

    SELECT
        store_src_id,
        store_name,
        address_id,
        'bl_cl' AS source_system,
        't_map_stores' AS source_table
    FROM ranked_stores
    WHERE rn = 1;

    CREATE INDEX idx_tmp_final_stores_src
        ON tmp_final_stores (store_src_id);

    -- INSERT
    INSERT INTO bl_3nf.ce_stores (
        store_id,
        store_src_id,
        store_name,
        address_id,
        source_system,
        source_table,
        insert_dt,
        update_dt
    )
    SELECT
        nextval('bl_3nf.seq_ce_store_id'),
        f.store_src_id,
        f.store_name,
        f.address_id,
        f.source_system,
        f.source_table,
        NOW(),
        NOW()
    FROM tmp_final_stores f
    WHERE NOT EXISTS (
        SELECT 1
        FROM bl_3nf.ce_stores ce
        WHERE ce.store_src_id = f.store_src_id
    );

    GET DIAGNOSTICS v_ins = ROW_COUNT;

    -- UPDATE (SCD TYPE 1)
    UPDATE bl_3nf.ce_stores ce
    SET
        store_name    = f.store_name,
        address_id    = f.address_id,
        source_system = f.source_system,
        source_table  = f.source_table,
        update_dt     = NOW()
    FROM tmp_final_stores f
    WHERE ce.store_src_id = f.store_src_id
      AND (
            ce.store_name    IS DISTINCT FROM f.store_name
         OR ce.address_id    IS DISTINCT FROM f.address_id
         OR ce.source_system IS DISTINCT FROM f.source_system
         OR ce.source_table  IS DISTINCT FROM f.source_table
      );

    GET DIAGNOSTICS v_upd = ROW_COUNT;

    -- Log result
    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_3nf.ce_stores',
        (v_ins + v_upd),
        CASE WHEN (v_ins + v_upd) > 0 THEN 'SUCCESS' ELSE 'NO_CHANGE' END,
        'Inserted=' || v_ins || ', Updated=' || v_upd ||
        '. Source=bl_cl.t_map_stores; Type 1 load; address FK resolved from ce_addresses.'
    );

    RAISE NOTICE 'ce_stores completed. inserted=%, updated=%', v_ins, v_upd;

EXCEPTION
WHEN OTHERS THEN
    GET STACKED DIAGNOSTICS
        v_err_detail = PG_EXCEPTION_DETAIL,
        v_err_hint   = PG_EXCEPTION_HINT;

    v_err_msg := SQLERRM;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_3nf.ce_stores',
        0,
        'FAILED',
        'ce_stores load failed',
        v_err_msg || ' | DETAIL=' || COALESCE(v_err_detail,'n.a.')
                  || ' | HINT='   || COALESCE(v_err_hint,'n.a.'),
        'ERROR'
    );

    RAISE;
END;
$$;


SQL

CREATE SEQUENCE
CREATE TABLE
CREATE INDEX
INSERT 0 1
CREATE PROCEDURE


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CALL bl_cl.load_ce_stores();

SQL

CALL


NOTICE:  table "tmp_final_stores" does not exist, skipping
NOTICE:  ce_stores completed. inserted=48, updated=0


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CREATE SEQUENCE IF NOT EXISTS bl_dm.seq_dim_store_id START 1;

CREATE TABLE IF NOT EXISTS bl_dm.dim_stores (
    store_surr_id      BIGINT PRIMARY KEY,          -- DM surrogate key
    store_src_id       BIGINT NOT NULL,             -- 3NF surrogate: ce_stores.store_id

    source_system      VARCHAR(100) NOT NULL,
    source_table       VARCHAR(100) NOT NULL,

    store_id_nk        VARCHAR(255) NOT NULL,       -- 3NF business key: ce_stores.store_src_id
    store_name         VARCHAR(100) NOT NULL,
    store_zip_code     VARCHAR(30)  NOT NULL,
    store_city         VARCHAR(100) NOT NULL,
    store_state        VARCHAR(100) NOT NULL,
    store_location     VARCHAR(100) NOT NULL,

    insert_dt          TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
    update_dt          TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP
);

CREATE UNIQUE INDEX IF NOT EXISTS ux_dim_stores_src_id
    ON bl_dm.dim_stores (store_src_id);

CREATE INDEX IF NOT EXISTS ix_dim_stores_nk
    ON bl_dm.dim_stores (store_id_nk);

CREATE INDEX IF NOT EXISTS ix_dim_stores_city_state
    ON bl_dm.dim_stores (store_city, store_state);

CREATE INDEX IF NOT EXISTS ix_dim_stores_location
    ON bl_dm.dim_stores (store_location);

INSERT INTO bl_dm.dim_stores (
    store_surr_id,
    store_src_id,
    source_system,
    source_table,
    store_id_nk,
    store_name,
    store_zip_code,
    store_city,
    store_state,
    store_location,
    insert_dt,
    update_dt
)
SELECT
    -1,
    -1,
    'MANUAL',
    'MANUAL',
    'n.a.',
    'n.a.',
    'n.a.',
    'n.a.',
    'n.a.',
    'n.a.',
    NOW(),
    NOW()
WHERE NOT EXISTS (
    SELECT 1
    FROM bl_dm.dim_stores
    WHERE store_surr_id = -1
);

CREATE OR REPLACE PROCEDURE bl_cl.load_dim_stores()
LANGUAGE plpgsql
AS $$
DECLARE
    v_proc        TEXT := 'bl_cl.load_dim_stores';
    v_ins         INT  := 0;
    v_upd         INT  := 0;
    v_err_msg     TEXT;
    v_err_detail  TEXT;
    v_err_hint    TEXT;
BEGIN
    DROP TABLE IF EXISTS tmp_dim_stores;

    CREATE TEMP TABLE tmp_dim_stores
    ON COMMIT DROP
    AS
    SELECT
        s.store_id AS store_src_id,
        s.store_src_id AS store_id_nk,
        COALESCE(s.store_name, 'n.a.') AS store_name,
        COALESCE(a.zip_code, 'n.a.') AS store_zip_code,
        COALESCE(c.city_name, 'n.a.') AS store_city,
        COALESCE(st.state_name, 'n.a.') AS store_state,
        COALESCE(s.store_name, 'n.a.') AS store_location,
        'bl_3nf'::VARCHAR(100) AS source_system,
        'ce_stores'::VARCHAR(100) AS source_table
    FROM bl_3nf.ce_stores s
    LEFT JOIN bl_3nf.ce_addresses a
        ON a.address_id = s.address_id
    LEFT JOIN bl_3nf.ce_cities c
        ON c.city_id = a.city_id
    LEFT JOIN bl_3nf.ce_states st
        ON st.state_id = c.state_id
    WHERE s.store_id <> -1;

    CREATE INDEX idx_tmp_dim_stores_src
        ON tmp_dim_stores (store_src_id);

    /* INSERT */
    INSERT INTO bl_dm.dim_stores (
        store_surr_id,
        store_src_id,
        source_system,
        source_table,
        store_id_nk,
        store_name,
        store_zip_code,
        store_city,
        store_state,
        store_location,
        insert_dt,
        update_dt
    )
    SELECT
        nextval('bl_dm.seq_dim_store_id'),
        t.store_src_id,
        t.source_system,
        t.source_table,
        t.store_id_nk,
        t.store_name,
        t.store_zip_code,
        t.store_city,
        t.store_state,
        t.store_location,
        NOW(),
        NOW()
    FROM tmp_dim_stores t
    WHERE NOT EXISTS (
        SELECT 1
        FROM bl_dm.dim_stores d
        WHERE d.store_src_id = t.store_src_id
    );

    GET DIAGNOSTICS v_ins = ROW_COUNT;

    /* UPDATE TYPE 1 */
    UPDATE bl_dm.dim_stores d
    SET
        source_system  = t.source_system,
        source_table   = t.source_table,
        store_id_nk    = t.store_id_nk,
        store_name     = t.store_name,
        store_zip_code = t.store_zip_code,
        store_city     = t.store_city,
        store_state    = t.store_state,
        store_location = t.store_location,
        update_dt      = NOW()
    FROM tmp_dim_stores t
    WHERE d.store_src_id = t.store_src_id
      AND (
            d.source_system  IS DISTINCT FROM t.source_system
         OR d.source_table   IS DISTINCT FROM t.source_table
         OR d.store_id_nk    IS DISTINCT FROM t.store_id_nk
         OR d.store_name     IS DISTINCT FROM t.store_name
         OR d.store_zip_code IS DISTINCT FROM t.store_zip_code
         OR d.store_city     IS DISTINCT FROM t.store_city
         OR d.store_state    IS DISTINCT FROM t.store_state
         OR d.store_location IS DISTINCT FROM t.store_location
      );

    GET DIAGNOSTICS v_upd = ROW_COUNT;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_dm.dim_stores',
        v_ins + v_upd,
        CASE WHEN (v_ins + v_upd) > 0 THEN 'SUCCESS' ELSE 'NO_CHANGE' END,
        'Inserted=' || v_ins || ', Updated=' || v_upd ||
        '. Source=bl_3nf.ce_stores + ce_addresses + ce_cities + ce_states.',
        NULL,
        'INFO',
        v_ins,
        v_upd,
        0,
        0,
        NULL
    );

    RAISE NOTICE 'dim_stores completed. inserted=%, updated=%', v_ins, v_upd;

EXCEPTION
WHEN OTHERS THEN
    GET STACKED DIAGNOSTICS
        v_err_detail = PG_EXCEPTION_DETAIL,
        v_err_hint   = PG_EXCEPTION_HINT;

    v_err_msg := SQLERRM;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_dm.dim_stores',
        0,
        'FAILED',
        'dim_stores load failed',
        v_err_msg || ' | DETAIL=' || COALESCE(v_err_detail, 'n.a.')
                  || ' | HINT='   || COALESCE(v_err_hint, 'n.a.'),
        'ERROR',
        0,
        0,
        0,
        0,
        NULL
    );

    RAISE;
END;
$$;


SQL

CREATE SEQUENCE
CREATE TABLE
CREATE INDEX
CREATE INDEX
CREATE INDEX
CREATE INDEX
INSERT 0 1
CREATE PROCEDURE


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'
CALL bl_cl.load_dim_stores();


SQL

CALL


NOTICE:  table "tmp_dim_stores" does not exist, skipping
NOTICE:  dim_stores completed. inserted=48, updated=0


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

DROP TABLE IF EXISTS bl_cl.t_map_products;
CREATE TABLE IF NOT EXISTS bl_cl.t_map_products (
    product_id               VARCHAR(100),                     -- raw source product key
    product_category         VARCHAR(100),                     -- standardized category
    product_name             VARCHAR(100),                     -- standardized product name
    product_brand            VARCHAR(100),                     -- standardized brand
    product_stock            INTEGER,                          -- parsed numeric value
    product_material         VARCHAR(100),                     -- standardized material
    product_manufacture_dt   TIMESTAMP,                        -- parsed manufacture datetime
    product_expiry_dt        TIMESTAMP,                        -- parsed expiry datetime
    source_system            VARCHAR(100),                     -- sa_online_retail / sa_offline_retail
    source_table             VARCHAR(100),                     -- src_online_retail / src_offline_retail
    insert_dt                TIMESTAMP DEFAULT CURRENT_TIMESTAMP

);

CREATE INDEX IF NOT EXISTS idx_t_map_products_src
ON bl_cl.t_map_products (product_id, source_system, source_table);


CREATE OR REPLACE PROCEDURE bl_cl.load_map_products()
LANGUAGE plpgsql
AS $$
DECLARE
    v_proc        TEXT := 'bl_cl.load_map_products';
    v_ins         INT  := 0;
    v_err_msg     TEXT;
    v_err_detail  TEXT;
    v_err_hint    TEXT;
BEGIN
    WITH unioned_sources AS (

    /* ONLINE */
    SELECT
        COALESCE(NULLIF(src.product_id, ''), 'n.a.') AS product_id,
        COALESCE(src.product_category,'n.a.') AS product_category,
        COALESCE(src.product_name,'n.a.') AS product_name,
        COALESCE(src.product_brand,'n.a.') AS product_brand,
        COALESCE(src.product_material,'n.a.') AS product_material,

        src.product_stock,
        src.product_manufacture_dt,
        src.product_expiry_dt,

        'sa_online_retail' AS source_system,
        'src_online_retail' AS source_table

    FROM sa_online_retail.src_online_retail src

    UNION ALL

    /* OFFLINE */
    SELECT
        COALESCE(NULLIF(src.product_id, ''), 'n.a.'),
        COALESCE(src.product_category,'n.a.'),
        COALESCE(src.product_name,'n.a.'),
        COALESCE(src.product_brand,'n.a.'),
        COALESCE(src.product_material,'n.a.'),

        src.product_stock,
        src.product_manufacture_dt,
        src.product_expiry_dt,

        'sa_offline_retail',
        'src_offline_retail'

    FROM sa_offline_retail.src_offline_retail src
),

distinct_source AS (
    SELECT DISTINCT
        product_id,
        product_category,
        product_name,
        product_brand,
        product_material,
        product_stock,
        product_manufacture_dt,
        product_expiry_dt,
        source_system,
        source_table
    FROM unioned_sources
    WHERE product_id <> 'n.a.'
)

      INSERT INTO bl_cl.t_map_products (
          product_id,
          product_category,
          product_name,
          product_brand,
          product_stock,
          product_material,
          product_manufacture_dt,
          product_expiry_dt,
          source_system,
          source_table
      )
      SELECT
          s.product_id,
          s.product_category,
          s.product_name,
          s.product_brand,
          s.product_stock,
          s.product_material,
          s.product_manufacture_dt,
          s.product_expiry_dt,
          s.source_system,
          s.source_table
      FROM distinct_source s
      WHERE NOT EXISTS (
          SELECT 1
          FROM bl_cl.t_map_products tgt
          WHERE tgt.product_id = s.product_id
            AND tgt.source_system  = s.source_system
            AND tgt.source_table   = s.source_table
      );
    GET DIAGNOSTICS v_ins = ROW_COUNT;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_cl.t_map_products',
        v_ins,
        CASE WHEN v_ins > 0 THEN 'SUCCESS' ELSE 'NO_CHANGE' END,
        'Inserted=' || v_ins ||
        '. Loaded formatted product rows using raw product_id as business key candidate in BL_CL.'
    );

    RAISE NOTICE 't_map_products completed. inserted=%', v_ins;

EXCEPTION
WHEN OTHERS THEN
    GET STACKED DIAGNOSTICS
        v_err_detail = PG_EXCEPTION_DETAIL,
        v_err_hint   = PG_EXCEPTION_HINT;

    v_err_msg := SQLERRM;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_cl.t_map_products',
        0,
        'FAILED',
        'Product map load failed',
        v_err_msg || ' | DETAIL=' || COALESCE(v_err_detail,'n.a.')
                  || ' | HINT='   || COALESCE(v_err_hint,'n.a.'),
        'ERROR'
    );

    RAISE;
END;
$$;

SQL

DROP TABLE
CREATE TABLE
CREATE INDEX
CREATE PROCEDURE


NOTICE:  table "t_map_products" does not exist, skipping


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CALL bl_cl.load_map_products();

SQL

CALL


NOTICE:  t_map_products completed. inserted=950000


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CREATE SEQUENCE IF NOT EXISTS bl_3nf.seq_ce_product_category_id START 1;

CREATE TABLE IF NOT EXISTS bl_3nf.ce_product_categories (
    product_category_id      BIGINT PRIMARY KEY,
    product_category_src_id  VARCHAR(100) NOT NULL,
    product_category_name    VARCHAR(100) NOT NULL,
    source_system            VARCHAR(100) NOT NULL,
    source_table             VARCHAR(100) NOT NULL,
    insert_dt                TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
    CONSTRAINT uq_ce_product_categories_src
        UNIQUE (product_category_src_id)
    );


CREATE INDEX IF NOT EXISTS ix_ce_product_categories_src_id
    ON bl_3nf.ce_product_categories (product_category_src_id);

INSERT INTO bl_3nf.ce_product_categories (
    product_category_id, product_category_src_id, product_category_name,
    source_system, source_table, insert_dt
)
SELECT
    -1,
    'n.a.',
    'n.a.',
    'MANUAL',
    'MANUAL',
    NOW()
WHERE NOT EXISTS (
    SELECT 1
    FROM bl_3nf.ce_product_categories
    WHERE product_category_id = -1
);


CREATE OR REPLACE PROCEDURE bl_cl.load_ce_product_categories()
LANGUAGE plpgsql
AS $$
DECLARE
    v_proc        TEXT := 'bl_cl.load_ce_product_categories';
    v_ins         INT  := 0;
    v_err_msg     TEXT;
    v_err_detail  TEXT;
    v_err_hint    TEXT;
BEGIN
    -- Drop temp table safely for reruns in the same session
    DROP TABLE IF EXISTS tmp_final_product_categories;

    -- Build one unique normalized category set from BL_CL products
    -- DISTINCT is enough because this entity is lookup-like and Type 0
    CREATE TEMP TABLE tmp_final_product_categories
    ON COMMIT DROP
    AS
    SELECT DISTINCT
        p.product_category        AS product_category_src_id,
        p.product_category        AS product_category_name,
        'bl_cl'::VARCHAR(100)     AS source_system,
        't_map_products'::VARCHAR(100) AS source_table
    FROM bl_cl.t_map_products p
    WHERE p.product_category IS NOT NULL
      AND p.product_category <> 'n.a.';

    -- Optional temp index for faster anti-join check
    CREATE INDEX idx_tmp_final_product_categories_src
        ON tmp_final_product_categories (product_category_src_id);

    -- Insert only brand-new category business keys
    INSERT INTO bl_3nf.ce_product_categories (
        product_category_id,
        product_category_src_id,
        product_category_name,
        source_system,
        source_table,
        insert_dt
            )
    SELECT
        nextval('bl_3nf.seq_ce_product_category_id'),
        f.product_category_src_id,
        f.product_category_name,
        f.source_system,
        f.source_table,
        NOW()
    FROM tmp_final_product_categories f
    WHERE NOT EXISTS (
        SELECT 1
        FROM bl_3nf.ce_product_categories ce
        WHERE ce.product_category_src_id = f.product_category_src_id
    );

    GET DIAGNOSTICS v_ins = ROW_COUNT;

    -- Log ETL result
   PERFORM bl_cl.log_etl_event(
    v_proc,
    'bl_3nf.ce_product_categories',
    v_ins,
    CASE WHEN v_ins > 0 THEN 'SUCCESS' ELSE 'NO_CHANGE' END,
    'Inserted=' || v_ins ||
    '. Loaded distinct product_categories from bl_cl.t_map_products.',
    NULL,
    'INFO',
    v_ins,
    0,
    0,
    NULL
);

    RAISE NOTICE 'ce_product_categories completed. inserted=%', v_ins;

EXCEPTION
WHEN OTHERS THEN
    -- Capture detailed PostgreSQL diagnostics
    GET STACKED DIAGNOSTICS
        v_err_detail = PG_EXCEPTION_DETAIL,
        v_err_hint   = PG_EXCEPTION_HINT;

    v_err_msg := SQLERRM;

    -- Log failure
    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_3nf.ce_product_categories',
        0,
        'FAILED',
        'Product categories 3nf load failed',
        v_err_msg || ' | DETAIL=' || COALESCE(v_err_detail,'n.a.')
                  || ' | HINT='   || COALESCE(v_err_hint,'n.a.'),
        'ERROR'
    );

    RAISE;
END;
$$;

SQL

CREATE SEQUENCE
CREATE TABLE
CREATE INDEX
INSERT 0 1
CREATE PROCEDURE


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CALL bl_cl.load_ce_product_categories();

SQL

CALL


NOTICE:  table "tmp_final_product_categories" does not exist, skipping
NOTICE:  ce_product_categories completed. inserted=5


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CREATE SEQUENCE IF NOT EXISTS bl_3nf.seq_ce_product_id START 1;

CREATE TABLE IF NOT EXISTS bl_3nf.ce_products(
    product_id               BIGINT PRIMARY KEY,
    product_src_id           VARCHAR(255) NOT NULL,

    product_category_id      BIGINT NOT NULL,
    product_name             VARCHAR(100) NOT NULL,
    product_brand            VARCHAR(100) NOT NULL,
    product_stock            INTEGER,
    product_material         VARCHAR(100) NOT NULL,
    product_manufacture_dt   TIMESTAMP,
    product_expiry_dt        TIMESTAMP,

    source_system            VARCHAR(100) NOT NULL,
    source_table             VARCHAR(100) NOT NULL,
    insert_dt                TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    update_dt                TIMESTAMP DEFAULT CURRENT_TIMESTAMP,

    CONSTRAINT fk_ce_products_category
        FOREIGN KEY (product_category_id)
        REFERENCES bl_3nf.ce_product_categories(product_category_id),
    CONSTRAINT uq_ce_products_src
        UNIQUE (product_src_id)
);

CREATE INDEX IF NOT EXISTS ix_ce_products_src_id
    ON bl_3nf.ce_products (product_src_id);

CREATE INDEX IF NOT EXISTS ix_ce_products_category_id
    ON bl_3nf.ce_products (product_category_id);

CREATE INDEX IF NOT EXISTS ix_ce_products_name_brand
    ON bl_3nf.ce_products (product_name, product_brand);

INSERT INTO bl_3nf.ce_products (
    product_id,
    product_src_id,
    product_category_id,
    product_name,
    product_brand,
    product_stock,
    product_material,
    product_manufacture_dt,
    product_expiry_dt,
    source_system,
    source_table,
    insert_dt,
    update_dt
)
SELECT
    -1,
    'n.a.',
    -1,
    'n.a.',
    'n.a.',
    0,
    'n.a.',
    TIMESTAMP '1900-01-01 00:00:00',
    TIMESTAMP '1900-01-01 00:00:00',
    'MANUAL',
    'MANUAL',
    NOW(),
    NOW()
WHERE NOT EXISTS (
    SELECT 1
    FROM bl_3nf.ce_products
    WHERE product_id = -1
);

CREATE OR REPLACE PROCEDURE bl_cl.load_ce_products()
LANGUAGE plpgsql
AS $$
DECLARE
    v_proc        TEXT := 'bl_cl.load_ce_products';
    v_ins         INT  := 0;
    v_upd         INT  := 0;
    v_err_msg     TEXT;
    v_err_detail  TEXT;
    v_err_hint    TEXT;
BEGIN
    DROP TABLE IF EXISTS tmp_final_products;

    CREATE TEMP TABLE tmp_final_products
    ON COMMIT DROP
    AS
    WITH src_products AS (
        SELECT
            p.product_id AS product_src_id,
            p.product_category,
            p.product_name,
            p.product_brand,
            p.product_stock,
            p.product_material,
            p.product_manufacture_dt,
            p.product_expiry_dt
        FROM bl_cl.t_map_products p
        WHERE p.product_id <> 'n.a.'
    ),
    category_resolved AS (
        SELECT
            s.product_src_id,
            COALESCE(c.product_category_id, -1) AS product_category_id,
            s.product_name,
            s.product_brand,
            s.product_stock,
            s.product_material,
            s.product_manufacture_dt,
            s.product_expiry_dt
        FROM src_products s
        LEFT JOIN bl_3nf.ce_product_categories c
            ON c.product_category_src_id = s.product_category
    ),
    ranked_products AS (
        SELECT
            cr.*,
            ROW_NUMBER() OVER (
                PARTITION BY cr.product_src_id
                ORDER BY
                    cr.product_manufacture_dt DESC NULLS LAST,
                    cr.product_expiry_dt DESC NULLS LAST,
                    cr.product_stock DESC NULLS LAST,
                    cr.product_category_id DESC,
                    cr.product_name DESC,
                    cr.product_brand DESC,
                    cr.product_material DESC
            ) AS rn
        FROM category_resolved cr
    )
    SELECT
        product_src_id,
        product_category_id,
        product_name,
        product_brand,
        product_stock,
        product_material,
        product_manufacture_dt,
        product_expiry_dt,
        'bl_cl' AS source_system,
        't_map_products' AS source_table
    FROM ranked_products
    WHERE rn = 1;

    CREATE INDEX idx_tmp_final_products_src
        ON tmp_final_products (product_src_id);

    INSERT INTO bl_3nf.ce_products (
        product_id,
        product_src_id,
        product_category_id,
        product_name,
        product_brand,
        product_stock,
        product_material,
        product_manufacture_dt,
        product_expiry_dt,
        source_system,
        source_table,
        insert_dt,
        update_dt
    )
    SELECT
        nextval('bl_3nf.seq_ce_product_id'),
        f.product_src_id,
        f.product_category_id,
        f.product_name,
        f.product_brand,
        f.product_stock,
        f.product_material,
        f.product_manufacture_dt,
        f.product_expiry_dt,
        f.source_system,
        f.source_table,
        NOW(),
        NOW()
    FROM tmp_final_products f
    WHERE NOT EXISTS (
        SELECT 1
        FROM bl_3nf.ce_products ce
        WHERE ce.product_src_id = f.product_src_id
    );

    GET DIAGNOSTICS v_ins = ROW_COUNT;

    UPDATE bl_3nf.ce_products ce
    SET
        product_category_id    = f.product_category_id,
        product_name           = f.product_name,
        product_brand          = f.product_brand,
        product_stock          = f.product_stock,
        product_material       = f.product_material,
        product_manufacture_dt = f.product_manufacture_dt,
        product_expiry_dt      = f.product_expiry_dt,
        update_dt              = NOW()
    FROM tmp_final_products f
    WHERE ce.product_src_id = f.product_src_id
      AND (
            ce.product_category_id    IS DISTINCT FROM f.product_category_id
         OR ce.product_name           IS DISTINCT FROM f.product_name
         OR ce.product_brand          IS DISTINCT FROM f.product_brand
         OR ce.product_stock          IS DISTINCT FROM f.product_stock
         OR ce.product_material       IS DISTINCT FROM f.product_material
         OR ce.product_manufacture_dt IS DISTINCT FROM f.product_manufacture_dt
         OR ce.product_expiry_dt      IS DISTINCT FROM f.product_expiry_dt
      );

    GET DIAGNOSTICS v_upd = ROW_COUNT;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_3nf.ce_products',
        v_upd + v_ins,
        CASE WHEN (v_upd + v_ins) > 0 THEN 'SUCCESS' ELSE 'NO_CHANGE' END,
        'SCD Type 1 upsert on products completed.',
        NULL,
        'INFO',
        v_ins,
        v_upd,
        0,
        0,
        NULL
    );

    RAISE NOTICE 'ce_products completed. inserted=%, updated=%', v_ins, v_upd;

EXCEPTION
WHEN OTHERS THEN
    GET STACKED DIAGNOSTICS
        v_err_detail = PG_EXCEPTION_DETAIL,
        v_err_hint   = PG_EXCEPTION_HINT;

    v_err_msg := SQLERRM;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_3nf.ce_products',
        0,
        'FAILED',
        'Products 3nf load failed',
        v_err_msg || ' | DETAIL=' || COALESCE(v_err_detail,'n.a.')
                  || ' | HINT='   || COALESCE(v_err_hint,'n.a.'),
        'ERROR'
    );

    RAISE;
END;
$$;

SQL

CREATE SEQUENCE
CREATE TABLE
CREATE INDEX
CREATE INDEX
CREATE INDEX
INSERT 0 0
CREATE PROCEDURE


NOTICE:  relation "seq_ce_product_id" already exists, skipping
NOTICE:  relation "ce_products" already exists, skipping
NOTICE:  relation "ix_ce_products_src_id" already exists, skipping
NOTICE:  relation "ix_ce_products_category_id" already exists, skipping
NOTICE:  relation "ix_ce_products_name_brand" already exists, skipping


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CALL bl_cl.load_ce_products();

SQL

CALL


NOTICE:  table "tmp_final_products" does not exist, skipping
NOTICE:  ce_products completed. inserted=19998, updated=0


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CREATE SEQUENCE IF NOT EXISTS bl_dm.seq_dim_product_id START 1;

CREATE TABLE IF NOT EXISTS bl_dm.dim_products (
    product_surr_id          BIGINT PRIMARY KEY,      -- DM surrogate
    product_src_id           BIGINT NOT NULL,         -- 3NF surrogate: ce_products.product_id

    source_system            VARCHAR(100) NOT NULL,
    source_table             VARCHAR(100) NOT NULL,

    product_category_name    VARCHAR(100) NOT NULL,
    product_name             VARCHAR(100) NOT NULL,
    product_brand            VARCHAR(100) NOT NULL,
    product_stock            INTEGER,
    product_material         VARCHAR(100) NOT NULL,
    product_manufacture_dt   TIMESTAMP,
    product_expiry_dt        TIMESTAMP,

    insert_dt                TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
    update_dt                TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP
);

CREATE UNIQUE INDEX IF NOT EXISTS ux_dim_products_src_id
    ON bl_dm.dim_products (product_src_id);

CREATE INDEX IF NOT EXISTS ix_dim_products_category
    ON bl_dm.dim_products (product_category_name);

CREATE INDEX IF NOT EXISTS ix_dim_products_name_brand
    ON bl_dm.dim_products (product_name, product_brand);

INSERT INTO bl_dm.dim_products (
    product_surr_id,
    product_src_id,
    source_system,
    source_table,
    product_category_name,
    product_name,
    product_brand,
    product_stock,
    product_material,
    product_manufacture_dt,
    product_expiry_dt,
    insert_dt,
    update_dt
)
SELECT
    -1,
    -1,
    'MANUAL',
    'MANUAL',
    'n.a.',
    'n.a.',
    'n.a.',
    0,
    'n.a.',
    TIMESTAMP '1900-01-01 00:00:00',
    TIMESTAMP '1900-01-01 00:00:00',
    NOW(),
    NOW()
WHERE NOT EXISTS (
    SELECT 1
    FROM bl_dm.dim_products
    WHERE product_surr_id = -1
);

CREATE OR REPLACE PROCEDURE bl_cl.load_dim_products()
LANGUAGE plpgsql
AS $$
DECLARE
    v_proc        TEXT := 'bl_cl.load_dim_products';
    v_ins         INT  := 0;
    v_upd         INT  := 0;
    v_err_msg     TEXT;
    v_err_detail  TEXT;
    v_err_hint    TEXT;
BEGIN
    DROP TABLE IF EXISTS tmp_dim_products;

    CREATE TEMP TABLE tmp_dim_products
    ON COMMIT DROP
    AS
    SELECT
        p.product_id AS product_src_id,
        'bl_3nf'::VARCHAR(100) AS source_system,
        'ce_products'::VARCHAR(100) AS source_table,
        COALESCE(pc.product_category_name, 'n.a.') AS product_category_name,
        COALESCE(p.product_name, 'n.a.') AS product_name,
        COALESCE(p.product_brand, 'n.a.') AS product_brand,
        p.product_stock,
        COALESCE(p.product_material, 'n.a.') AS product_material,
        p.product_manufacture_dt,
        p.product_expiry_dt
    FROM bl_3nf.ce_products p
    LEFT JOIN bl_3nf.ce_product_categories pc
        ON pc.product_category_id = p.product_category_id
    WHERE p.product_id <> -1;

    CREATE INDEX idx_tmp_dim_products_src
        ON tmp_dim_products (product_src_id);

    /* INSERT */
    INSERT INTO bl_dm.dim_products (
        product_surr_id,
        product_src_id,
        source_system,
        source_table,
        product_category_name,
        product_name,
        product_brand,
        product_stock,
        product_material,
        product_manufacture_dt,
        product_expiry_dt,
        insert_dt,
        update_dt
    )
    SELECT
        nextval('bl_dm.seq_dim_product_id'),
        t.product_src_id,
        t.source_system,
        t.source_table,
        t.product_category_name,
        t.product_name,
        t.product_brand,
        t.product_stock,
        t.product_material,
        t.product_manufacture_dt,
        t.product_expiry_dt,
        NOW(),
        NOW()
    FROM tmp_dim_products t
    WHERE NOT EXISTS (
        SELECT 1
        FROM bl_dm.dim_products d
        WHERE d.product_src_id = t.product_src_id
    );

    GET DIAGNOSTICS v_ins = ROW_COUNT;

    /* UPDATE TYPE 1 */
    UPDATE bl_dm.dim_products d
    SET
        source_system          = t.source_system,
        source_table           = t.source_table,
        product_category_name  = t.product_category_name,
        product_name           = t.product_name,
        product_brand          = t.product_brand,
        product_stock          = t.product_stock,
        product_material       = t.product_material,
        product_manufacture_dt = t.product_manufacture_dt,
        product_expiry_dt      = t.product_expiry_dt,
        update_dt              = NOW()
    FROM tmp_dim_products t
    WHERE d.product_src_id = t.product_src_id
      AND (
            d.source_system          IS DISTINCT FROM t.source_system
         OR d.source_table           IS DISTINCT FROM t.source_table
         OR d.product_category_name  IS DISTINCT FROM t.product_category_name
         OR d.product_name           IS DISTINCT FROM t.product_name
         OR d.product_brand          IS DISTINCT FROM t.product_brand
         OR d.product_stock          IS DISTINCT FROM t.product_stock
         OR d.product_material       IS DISTINCT FROM t.product_material
         OR d.product_manufacture_dt IS DISTINCT FROM t.product_manufacture_dt
         OR d.product_expiry_dt      IS DISTINCT FROM t.product_expiry_dt
      );

    GET DIAGNOSTICS v_upd = ROW_COUNT;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_dm.dim_products',
        (v_ins + v_upd),
        CASE WHEN (v_ins + v_upd) > 0 THEN 'SUCCESS' ELSE 'NO_CHANGE' END,
        'Inserted=' || v_ins || ', Updated=' || v_upd ||
        '. Source=bl_3nf.ce_products + ce_product_categories.',
        NULL,
        'INFO',
        v_ins,
        v_upd,
        0,
        0,
        NULL
    );

    RAISE NOTICE 'dim_products completed. inserted=%, updated=%', v_ins, v_upd;

EXCEPTION
WHEN OTHERS THEN
    GET STACKED DIAGNOSTICS
        v_err_detail = PG_EXCEPTION_DETAIL,
        v_err_hint   = PG_EXCEPTION_HINT;

    v_err_msg := SQLERRM;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_dm.dim_products',
        0,
        'FAILED',
        'dim_products load failed',
        v_err_msg || ' | DETAIL=' || COALESCE(v_err_detail, 'n.a.')
                  || ' | HINT='   || COALESCE(v_err_hint, 'n.a.'),
        'ERROR',
        0,
        0,
        0,
        0,
        NULL
    );

    RAISE;
END;
$$;


SQL

CREATE SEQUENCE
CREATE TABLE
CREATE INDEX
CREATE INDEX
CREATE INDEX
INSERT 0 1
CREATE PROCEDURE


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CALL bl_cl.load_dim_products();

SQL

CALL


NOTICE:  table "tmp_dim_products" does not exist, skipping
NOTICE:  dim_products completed. inserted=19998, updated=0


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'
DROP TABLE IF EXISTS bl_cl.t_map_promotions;
CREATE TABLE IF NOT EXISTS bl_cl.t_map_promotions (
    promotion_id        VARCHAR(100),                     -- raw source promotion key
    promotion_type      VARCHAR(100),                     -- standardized promotion type
    promotion_channel   VARCHAR(100),                     -- standardized promotion channel
    promotion_start_dt  TIMESTAMP,                        -- parsed promotion start timestamp
    promotion_end_dt    TIMESTAMP,                        -- parsed promotion end timestamp
    source_system       VARCHAR(100),                     -- sa_online_retail / sa_offline_retail
    source_table        VARCHAR(100),                     -- src_online_retail / src_offline_retail
    insert_dt           TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);


CREATE INDEX IF NOT EXISTS idx_t_map_promotions_src
ON bl_cl.t_map_promotions (promotion_id, source_system, source_table);

CREATE INDEX IF NOT EXISTS idx_t_map_promotions_type
ON bl_cl.t_map_promotions (promotion_type);

CREATE INDEX IF NOT EXISTS idx_t_map_promotions_channel
ON bl_cl.t_map_promotions (promotion_channel);

CREATE OR REPLACE PROCEDURE bl_cl.load_map_promotions()
LANGUAGE plpgsql
AS $$
DECLARE
    v_proc        TEXT := 'bl_cl.load_map_promotions';
    v_ins         INT  := 0;
    v_err_msg     TEXT;
    v_err_detail  TEXT;
    v_err_hint    TEXT;
BEGIN

    WITH unioned_sources AS (
        SELECT
            src.promotion_id,
            src.promotion_type,
            src.promotion_channel,
            src.promotion_start_dt,
            src.promotion_end_dt,

            'sa_online_retail' AS source_system,
            'src_online_retail' AS source_table

        FROM sa_online_retail.src_online_retail src

        UNION ALL

        SELECT
            src.promotion_id,
            src.promotion_type,
            src.promotion_channel,
            src.promotion_start_dt,
            src.promotion_end_dt,

            'sa_offline_retail',
            'src_offline_retail'

        FROM sa_offline_retail.src_offline_retail src
    ),

    distinct_source AS (
        SELECT DISTINCT
            promotion_id,
            promotion_type,
            promotion_channel,
            promotion_start_dt,
            promotion_end_dt,
            source_system,
            source_table
        FROM unioned_sources
        WHERE promotion_id <> 'n.a.'
    )

    INSERT INTO bl_cl.t_map_promotions (
        promotion_id,
        promotion_type,
        promotion_channel,
        promotion_start_dt,
        promotion_end_dt,
        source_system,
        source_table
    )
    SELECT
        s.promotion_id,
        s.promotion_type,
        s.promotion_channel,
        s.promotion_start_dt,
        s.promotion_end_dt,
        s.source_system,
        s.source_table
    FROM distinct_source s
    WHERE NOT EXISTS (
    SELECT 1
    FROM bl_cl.t_map_promotions tgt
    WHERE tgt.promotion_id = s.promotion_id
      AND tgt.source_system    = s.source_system
      AND tgt.source_table     = s.source_table
);

    GET DIAGNOSTICS v_ins = ROW_COUNT;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_cl.t_map_promotions',
        v_ins,
        CASE WHEN v_ins > 0 THEN 'SUCCESS' ELSE 'NO_CHANGE' END,
        'Inserted=' || v_ins || ' . Loaded formatted promotion rows using promotion_id as business key candidate'
    );

    RAISE NOTICE 't_map_promotions completed. inserted=%', v_ins;

EXCEPTION
WHEN OTHERS THEN

    GET STACKED DIAGNOSTICS
        v_err_detail = PG_EXCEPTION_DETAIL,
        v_err_hint   = PG_EXCEPTION_HINT;

    v_err_msg := SQLERRM;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_cl.t_map_promotions',
        0,
        'FAILED',
        'Promotion map load failed',
        v_err_msg || ' | DETAIL=' || COALESCE(v_err_detail, 'n.a.')
                  || ' | HINT='   || COALESCE(v_err_hint, 'n.a.'),
        'ERROR'
    );

    RAISE;

END;
$$;

SQL

DROP TABLE
CREATE TABLE
CREATE INDEX
CREATE INDEX
CREATE INDEX
CREATE PROCEDURE


NOTICE:  table "t_map_promotions" does not exist, skipping


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CALL bl_cl.load_map_promotions();

SQL

CALL


NOTICE:  t_map_promotions completed. inserted=950000


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CREATE SEQUENCE IF NOT EXISTS bl_3nf.seq_ce_promotion_type_id START 1;

CREATE TABLE IF NOT EXISTS bl_3nf.ce_promotion_types (
    promotion_type_id      BIGINT PRIMARY KEY,
    promotion_type_src_id  VARCHAR(255) NOT NULL,
    promotion_type_name    VARCHAR(100) NOT NULL,
    source_system          VARCHAR(100) NOT NULL,
    source_table           VARCHAR(100) NOT NULL,
    insert_dt              TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
    CONSTRAINT ux_ce_promotion_types_src_id
        UNIQUE (promotion_type_src_id)
);

CREATE INDEX IF NOT EXISTS ix_ce_promotion_types_src
    ON bl_3nf.ce_promotion_types (promotion_type_src_id);

INSERT INTO bl_3nf.ce_promotion_types (
    promotion_type_id,
    promotion_type_src_id,
    promotion_type_name,
    source_system,
    source_table,
    insert_dt
)
SELECT
    -1,
    'n.a.',
    'n.a.',
    'MANUAL',
    'MANUAL',
    NOW()
    WHERE NOT EXISTS (
    SELECT 1
    FROM bl_3nf.ce_promotion_types
    WHERE promotion_type_id = -1
);

CREATE OR REPLACE PROCEDURE bl_cl.load_ce_promotion_types()
LANGUAGE plpgsql
AS $$
DECLARE
    v_proc        TEXT := 'bl_cl.load_ce_promotion_types';
    v_ins         INT  := 0;
    v_err_msg     TEXT;
    v_err_detail  TEXT;
    v_err_hint    TEXT;
BEGIN
    INSERT INTO bl_3nf.ce_promotion_types (
        promotion_type_id,
        promotion_type_src_id,
        promotion_type_name,
        source_system,
        source_table,
        insert_dt
    )
    SELECT
        nextval('bl_3nf.seq_ce_promotion_type_id'),
        s.promotion_type_src_id,
        s.promotion_type_name,
        'bl_cl',
        't_map_promotions',
        NOW()
    FROM (
        SELECT DISTINCT
            p.promotion_type AS promotion_type_src_id,
            p.promotion_type AS promotion_type_name
        FROM bl_cl.t_map_promotions p
        WHERE COALESCE(p.promotion_type, 'n.a.') <> 'n.a.'
    ) s
    WHERE NOT EXISTS (
        SELECT 1
        FROM bl_3nf.ce_promotion_types t
        WHERE t.promotion_type_src_id = s.promotion_type_src_id
    );

    GET DIAGNOSTICS v_ins = ROW_COUNT;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_3nf.ce_promotion_types',
        v_ins,
        CASE WHEN v_ins > 0 THEN 'SUCCESS' ELSE 'NO_CHANGE' END,
        'Inserted=' || v_ins || '. Loaded promotion types from bl_cl.t_map_promotions.'
    );
            RAISE NOTICE 'ce_promotion_types completed. inserted=%', v_ins;


EXCEPTION
WHEN OTHERS THEN
    GET STACKED DIAGNOSTICS
        v_err_detail = PG_EXCEPTION_DETAIL,
        v_err_hint   = PG_EXCEPTION_HINT;

    v_err_msg := SQLERRM;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_3nf.ce_promotion_types',
        0,
        'FAILED',
        'Promotion types load failed',
        v_err_msg || ' | DETAIL=' || COALESCE(v_err_detail, 'n.a.')
                  || ' | HINT='   || COALESCE(v_err_hint, 'n.a.'),
        'ERROR'
    );

    RAISE;
END;
$$;

SQL

CREATE SEQUENCE
CREATE TABLE
CREATE INDEX
INSERT 0 1
CREATE PROCEDURE


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'
CALL bl_cl.load_ce_promotion_types();

SQL

CALL


NOTICE:  ce_promotion_types completed. inserted=3


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CREATE SEQUENCE IF NOT EXISTS bl_3nf.seq_ce_promotion_id START 1;

CREATE TABLE IF NOT EXISTS bl_3nf.ce_promotions (
    promotion_id        BIGINT PRIMARY KEY,
    promotion_src_id    VARCHAR(255) NOT NULL,   -- business key
    promotion_type_id   BIGINT NOT NULL,
    promotion_channel   VARCHAR(100) NOT NULL,
    promotion_start_dt  TIMESTAMP,
    promotion_end_dt    TIMESTAMP,
    source_system       VARCHAR(100) NOT NULL,
    source_table        VARCHAR(100) NOT NULL,
    insert_dt           TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,

    CONSTRAINT ux_ce_promotions_src_id
        UNIQUE (promotion_src_id),

    CONSTRAINT fk_ce_promotions_type
        FOREIGN KEY (promotion_type_id)
        REFERENCES bl_3nf.ce_promotion_types (promotion_type_id)
);

CREATE INDEX IF NOT EXISTS ix_ce_promotions_src_id
    ON bl_3nf.ce_promotions (promotion_src_id);

-- Default row
INSERT INTO bl_3nf.ce_promotions (
    promotion_id,
    promotion_src_id,
    promotion_type_id,
    promotion_channel,
    promotion_start_dt,
    promotion_end_dt,
    source_system,
    source_table,
    insert_dt
)
SELECT
    -1,
    'n.a.',
    -1,
    'n.a.',
    TIMESTAMP '1900-01-01',
    TIMESTAMP '1900-01-01',
    'MANUAL',
    'MANUAL',
    NOW()
WHERE NOT EXISTS (
    SELECT 1
    FROM bl_3nf.ce_promotions
    WHERE promotion_id = -1
);

CREATE OR REPLACE PROCEDURE bl_cl.load_ce_promotions()
LANGUAGE plpgsql
AS $$
DECLARE
    v_proc        TEXT := 'bl_cl.load_ce_promotions';
    v_ins         INT  := 0;
    v_err_msg     TEXT;
    v_err_detail  TEXT;
    v_err_hint    TEXT;
BEGIN
    DROP TABLE IF EXISTS tmp_final_promotions;

    CREATE TEMP TABLE tmp_final_promotions
    ON COMMIT DROP
    AS
    WITH src_promotions AS (

        SELECT
            p.promotion_id AS promotion_src_id,
            p.promotion_type,
            p.promotion_channel,
            p.promotion_start_dt,
            p.promotion_end_dt

        FROM bl_cl.t_map_promotions p
        WHERE p.promotion_id <> 'n.a.'
    ),

    ranked AS (
        SELECT
            s.*,
            ROW_NUMBER() OVER (
                PARTITION BY s.promotion_src_id
                ORDER BY
                    s.promotion_start_dt DESC NULLS LAST,
                    s.promotion_end_dt   DESC NULLS LAST,
                    s.promotion_type DESC,
                    s.promotion_channel DESC
            ) AS rn
        FROM src_promotions s
    ),

    resolved_type AS (
        SELECT
            r.promotion_src_id,
            COALESCE(pt.promotion_type_id, -1) AS promotion_type_id,
            r.promotion_channel,
            r.promotion_start_dt,
            r.promotion_end_dt,
            'bl_cl'::VARCHAR(100) AS source_system,
            't_map_promotions'::VARCHAR(100) AS source_table
        FROM ranked r
        LEFT JOIN bl_3nf.ce_promotion_types pt
            ON pt.promotion_type_src_id = r.promotion_type
        WHERE r.rn = 1
    )

    SELECT
        promotion_src_id,
        promotion_type_id,
        promotion_channel,
        promotion_start_dt,
        promotion_end_dt,
        source_system,
        source_table
    FROM resolved_type;

    CREATE INDEX idx_tmp_final_promotions_src
        ON tmp_final_promotions (promotion_src_id);

    -- ================= INSERT ONLY (TYPE 0) =================
    INSERT INTO bl_3nf.ce_promotions (
        promotion_id,
        promotion_src_id,
        promotion_type_id,
        promotion_channel,
        promotion_start_dt,
        promotion_end_dt,
        source_system,
        source_table,
        insert_dt
            )
    SELECT
        nextval('bl_3nf.seq_ce_promotion_id'),
        f.promotion_src_id,
        f.promotion_type_id,
        f.promotion_channel,
        f.promotion_start_dt,
        f.promotion_end_dt,
        f.source_system,
        f.source_table,
        NOW()
    FROM tmp_final_promotions f
    WHERE NOT EXISTS (
        SELECT 1
        FROM bl_3nf.ce_promotions ce
        WHERE ce.promotion_src_id = f.promotion_src_id
    );

    GET DIAGNOSTICS v_ins = ROW_COUNT;

    PERFORM bl_cl.log_etl_event(
    v_proc,
    'bl_3nf.ce_promotions',
    v_ins,
    CASE WHEN v_ins > 0 THEN 'SUCCESS' ELSE 'NO_CHANGE' END,
    'Inserted=' || v_ins || '. Type 0 load: one immutable row per promotion_src_id.'
);

            RAISE NOTICE 'ce_promotion completed. inserted=%', v_ins;


EXCEPTION
WHEN OTHERS THEN
    GET STACKED DIAGNOSTICS
        v_err_detail = PG_EXCEPTION_DETAIL,
        v_err_hint   = PG_EXCEPTION_HINT;

    v_err_msg := SQLERRM;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_3nf.ce_promotions',
        0,
        'FAILED',
        'Promotions load failed',
        v_err_msg || ' | DETAIL=' || COALESCE(v_err_detail, 'n.a.')
                  || ' | HINT='   || COALESCE(v_err_hint, 'n.a.'),
        'ERROR'
    );

    RAISE;
END;
$$;

SQL

CREATE SEQUENCE
CREATE TABLE
CREATE INDEX
INSERT 0 1
CREATE PROCEDURE


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CALL bl_cl.load_ce_promotions();

SQL

CALL


NOTICE:  table "tmp_final_promotions" does not exist, skipping
NOTICE:  ce_promotion completed. inserted=1998


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CREATE SEQUENCE IF NOT EXISTS bl_dm.seq_dim_promotion_id START 1;

CREATE TABLE IF NOT EXISTS bl_dm.dim_promotions (
    promotion_surr_id     BIGINT PRIMARY KEY,
    promotion_src_id      BIGINT NOT NULL,     -- 3NF surrogate: ce_promotions.promotion_id

    source_system         VARCHAR(100) NOT NULL,
    source_table          VARCHAR(100) NOT NULL,

    promotion_id_nk       VARCHAR(255) NOT NULL,   -- 3NF business key: ce_promotions.promotion_src_id
    promotion_channel     VARCHAR(100) NOT NULL,
    promotion_type_name   VARCHAR(100) NOT NULL,
    promotion_start_dt    TIMESTAMP,
    promotion_end_dt      TIMESTAMP,

    insert_dt             TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
    update_dt             TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP
);

CREATE UNIQUE INDEX IF NOT EXISTS ux_dim_promotions_src_id
    ON bl_dm.dim_promotions (promotion_src_id);

CREATE INDEX IF NOT EXISTS ix_dim_promotions_nk
    ON bl_dm.dim_promotions (promotion_id_nk);

CREATE INDEX IF NOT EXISTS ix_dim_promotions_type
    ON bl_dm.dim_promotions (promotion_type_name);

CREATE INDEX IF NOT EXISTS ix_dim_promotions_channel
    ON bl_dm.dim_promotions (promotion_channel);

INSERT INTO bl_dm.dim_promotions (
    promotion_surr_id,
    promotion_src_id,
    source_system,
    source_table,
    promotion_id_nk,
    promotion_channel,
    promotion_type_name,
    promotion_start_dt,
    promotion_end_dt,
    insert_dt,
    update_dt
)
SELECT
    -1,
    -1,
    'MANUAL',
    'MANUAL',
    'n.a.',
    'n.a.',
    'n.a.',
    TIMESTAMP '1900-01-01 00:00:00',
    TIMESTAMP '1900-01-01 00:00:00',
    NOW(),
    NOW()
WHERE NOT EXISTS (
    SELECT 1
    FROM bl_dm.dim_promotions
    WHERE promotion_surr_id = -1
);

CREATE OR REPLACE PROCEDURE bl_cl.load_dim_promotions()
LANGUAGE plpgsql
AS $$
DECLARE
    v_proc        TEXT := 'bl_cl.load_dim_promotions';
    v_ins         INT  := 0;
    v_upd         INT  := 0;
    v_err_msg     TEXT;
    v_err_detail  TEXT;
    v_err_hint    TEXT;
BEGIN
    DROP TABLE IF EXISTS tmp_dim_promotions;

    CREATE TEMP TABLE tmp_dim_promotions
    ON COMMIT DROP
    AS
    SELECT
        p.promotion_id AS promotion_src_id,
        'bl_3nf'::VARCHAR(100) AS source_system,
        'ce_promotions'::VARCHAR(100) AS source_table,
        COALESCE(p.promotion_src_id, 'n.a.') AS promotion_id_nk,
        COALESCE(p.promotion_channel, 'n.a.') AS promotion_channel,
        COALESCE(pt.promotion_type_name, 'n.a.') AS promotion_type_name,
        p.promotion_start_dt,
        p.promotion_end_dt
    FROM bl_3nf.ce_promotions p
    LEFT JOIN bl_3nf.ce_promotion_types pt
        ON pt.promotion_type_id = p.promotion_type_id
    WHERE p.promotion_id <> -1;

    CREATE INDEX idx_tmp_dim_promotions_src
        ON tmp_dim_promotions (promotion_src_id);

    /* INSERT */
    INSERT INTO bl_dm.dim_promotions (
        promotion_surr_id,
        promotion_src_id,
        source_system,
        source_table,
        promotion_id_nk,
        promotion_channel,
        promotion_type_name,
        promotion_start_dt,
        promotion_end_dt,
        insert_dt,
        update_dt
    )
    SELECT
        nextval('bl_dm.seq_dim_promotion_id'),
        t.promotion_src_id,
        t.source_system,
        t.source_table,
        t.promotion_id_nk,
        t.promotion_channel,
        t.promotion_type_name,
        t.promotion_start_dt,
        t.promotion_end_dt,
        NOW(),
        NOW()
    FROM tmp_dim_promotions t
    WHERE NOT EXISTS (
        SELECT 1
        FROM bl_dm.dim_promotions d
        WHERE d.promotion_src_id = t.promotion_src_id
    );

    GET DIAGNOSTICS v_ins = ROW_COUNT;

    /* UPDATE TYPE 1 */
    UPDATE bl_dm.dim_promotions d
    SET
        source_system       = t.source_system,
        source_table        = t.source_table,
        promotion_id_nk     = t.promotion_id_nk,
        promotion_channel   = t.promotion_channel,
        promotion_type_name = t.promotion_type_name,
        promotion_start_dt  = t.promotion_start_dt,
        promotion_end_dt    = t.promotion_end_dt,
        update_dt           = NOW()
    FROM tmp_dim_promotions t
    WHERE d.promotion_src_id = t.promotion_src_id
      AND (
            d.source_system       IS DISTINCT FROM t.source_system
         OR d.source_table        IS DISTINCT FROM t.source_table
         OR d.promotion_id_nk     IS DISTINCT FROM t.promotion_id_nk
         OR d.promotion_channel   IS DISTINCT FROM t.promotion_channel
         OR d.promotion_type_name IS DISTINCT FROM t.promotion_type_name
         OR d.promotion_start_dt  IS DISTINCT FROM t.promotion_start_dt
         OR d.promotion_end_dt    IS DISTINCT FROM t.promotion_end_dt
      );

    GET DIAGNOSTICS v_upd = ROW_COUNT;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_dm.dim_promotions',
        (v_ins + v_upd),
        CASE WHEN (v_ins + v_upd) > 0 THEN 'SUCCESS' ELSE 'NO_CHANGE' END,
        'Inserted=' || v_ins || ', Updated=' || v_upd ||
        '. Source=bl_3nf.ce_promotions + ce_promotion_types.',
        NULL,
        'INFO',
        v_ins,
        v_upd,
        0,
        0,
        NULL
    );

    RAISE NOTICE 'dim_promotions completed. inserted=%, updated=%', v_ins, v_upd;

EXCEPTION
WHEN OTHERS THEN
    GET STACKED DIAGNOSTICS
        v_err_detail = PG_EXCEPTION_DETAIL,
        v_err_hint   = PG_EXCEPTION_HINT;

    v_err_msg := SQLERRM;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_dm.dim_promotions',
        0,
        'FAILED',
        'dim_promotions load failed',
        v_err_msg || ' | DETAIL=' || COALESCE(v_err_detail, 'n.a.')
                  || ' | HINT='   || COALESCE(v_err_hint, 'n.a.'),
        'ERROR',
        0,
        0,
        0,
        0,
        NULL
    );

    RAISE;
END;
$$;


SQL

CREATE SEQUENCE
CREATE TABLE
CREATE INDEX
CREATE INDEX
CREATE INDEX
CREATE INDEX
INSERT 0 1
CREATE PROCEDURE


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CALL bl_cl.load_dim_promotions();

SQL

CALL


NOTICE:  table "tmp_dim_promotions" does not exist, skipping
NOTICE:  dim_promotions completed. inserted=1998, updated=0


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

-- Profiling sonucu gösterdi ki aynı transaction id içinde birden fazla eng id var. Bu da engagement'ın sentetik veride bir line-level/row-level identifier gibi davraqndığını gösterdi.

DROP TABLE IF EXISTS bl_cl.t_map_deliveries;

CREATE TABLE IF NOT EXISTS bl_cl.t_map_deliveries (
    delivery_id        VARCHAR(100),
    delivery_type      VARCHAR(100),
    delivery_status    VARCHAR(100),
    shipping_partner   VARCHAR(100),
    source_system      VARCHAR(100),
    source_table       VARCHAR(100),
    insert_dt          TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE INDEX IF NOT EXISTS idx_t_map_deliveries_src
ON bl_cl.t_map_deliveries (delivery_id, source_system, source_table);

CREATE INDEX IF NOT EXISTS idx_t_map_deliveries_partner
ON bl_cl.t_map_deliveries (shipping_partner);

CREATE INDEX IF NOT EXISTS idx_t_map_deliveries_type
ON bl_cl.t_map_deliveries (delivery_type);

CREATE OR REPLACE PROCEDURE bl_cl.load_map_deliveries()
LANGUAGE plpgsql
AS $$
DECLARE
    v_proc        TEXT := 'bl_cl.load_map_deliveries';
    v_ins         INT  := 0;
    v_err_msg     TEXT;
    v_err_detail  TEXT;
    v_err_hint    TEXT;
BEGIN
   WITH unioned_sources AS (

    /* ONLINE */
    SELECT
        COALESCE(NULLIF(src.delivery_id,''),'n.a.') AS delivery_id,

        COALESCE(src.delivery_type,'n.a.') AS delivery_type,
        COALESCE(src.delivery_status,'n.a.') AS delivery_status,
        COALESCE(src.shipping_partner,'n.a.') AS shipping_partner,

        'sa_online_retail' AS source_system,
        'src_online_retail' AS source_table

    FROM sa_online_retail.src_online_retail src

    UNION ALL

    /* OFFLINE */
    SELECT
        COALESCE(NULLIF(src.delivery_id,''),'n.a.'),

        COALESCE(src.delivery_type,'n.a.'),
        COALESCE(src.delivery_status,'n.a.'),
        COALESCE(src.shipping_partner,'n.a.'),

        'sa_offline_retail',
        'src_offline_retail'

    FROM sa_offline_retail.src_offline_retail src
),
    -- distinct_source MUST include src_id
distinct_source AS (
    SELECT DISTINCT
        delivery_id,
        delivery_type,
        delivery_status,
        shipping_partner,
        source_system,
        source_table
    FROM unioned_sources
    WHERE delivery_id <> 'n.a.'
)

-- INSERT MUST include it
INSERT INTO bl_cl.t_map_deliveries (
    delivery_id,
    delivery_type,
    delivery_status,
    shipping_partner,
    source_system,
    source_table
)
SELECT
    s.delivery_id,
    s.delivery_type,
    s.delivery_status,
    s.shipping_partner,
    s.source_system,
    s.source_table
    FROM distinct_source s
    WHERE NOT EXISTS (
        SELECT 1
        FROM bl_cl.t_map_deliveries t
        WHERE t.delivery_id      = s.delivery_id
          AND t.source_system    = s.source_system
          AND t.source_table     = s.source_table
    );

    GET DIAGNOSTICS v_ins = ROW_COUNT;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_cl.t_map_deliveries',
        v_ins,
        CASE WHEN v_ins > 0 THEN 'SUCCESS' ELSE 'NO_CHANGE' END,
        'Inserted=' || v_ins || '. Loaded cleaned delivery rows using SELECT DISTINCT only.'
    );

    RAISE NOTICE 't_map_deliveries completed. inserted=%', v_ins;

EXCEPTION
WHEN OTHERS THEN
    GET STACKED DIAGNOSTICS
        v_err_detail = PG_EXCEPTION_DETAIL,
        v_err_hint   = PG_EXCEPTION_HINT;

    v_err_msg := SQLERRM;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_cl.t_map_deliveries',
        0,
        'FAILED',
        'Delivery map load failed',
        v_err_msg || ' | DETAIL=' || COALESCE(v_err_detail, 'n.a.')
                  || ' | HINT='   || COALESCE(v_err_hint, 'n.a.'),
        'ERROR'
    );

    RAISE;
END;
$$;


SQL

DROP TABLE
CREATE TABLE
CREATE INDEX
CREATE INDEX
CREATE INDEX
CREATE PROCEDURE


NOTICE:  table "t_map_deliveries" does not exist, skipping


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CALL bl_cl.load_map_deliveries();
SQL

CALL


NOTICE:  t_map_deliveries completed. inserted=950000


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CREATE SEQUENCE IF NOT EXISTS bl_3nf.seq_ce_shipping_partner_id START 1;

CREATE TABLE IF NOT EXISTS bl_3nf.ce_shipping_partners (
    shipping_partner_id      BIGINT PRIMARY KEY,
    shipping_partner_src_id  VARCHAR(100) NOT NULL,
    shipping_partner_name    VARCHAR(100) NOT NULL,
    source_system            VARCHAR(100) NOT NULL,
    source_table             VARCHAR(100) NOT NULL,
    insert_dt                TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
    CONSTRAINT ux_ce_shipping_partners_src
        UNIQUE (shipping_partner_src_id)
);

CREATE INDEX IF NOT EXISTS ix_ce_shipping_partners_src
ON bl_3nf.ce_shipping_partners (shipping_partner_src_id);


INSERT INTO bl_3nf.ce_shipping_partners (
    shipping_partner_id,
    shipping_partner_src_id,
    shipping_partner_name,
    source_system,
    source_table,
    insert_dt
)
SELECT
    -1,
    'n.a.',
    'n.a.',
    'MANUAL',
    'MANUAL',
    NOW()
WHERE NOT EXISTS (
    SELECT 1
    FROM bl_3nf.ce_shipping_partners
    WHERE shipping_partner_id = -1
);

CREATE OR REPLACE PROCEDURE bl_cl.load_ce_shipping_partners()
LANGUAGE plpgsql
AS $$
DECLARE
    v_proc        TEXT := 'bl_cl.load_ce_shipping_partners';
    v_ins         INT  := 0;
    v_err_msg     TEXT;
    v_err_detail  TEXT;
    v_err_hint    TEXT;

BEGIN
    INSERT INTO bl_3nf.ce_shipping_partners (
        shipping_partner_id,
        shipping_partner_src_id,
        shipping_partner_name,
        source_system,
        source_table,
        insert_dt
    )
    SELECT
        nextval('bl_3nf.seq_ce_shipping_partner_id'),
        s.shipping_partner_src_id,
        s.shipping_partner_name,
        'bl_cl',
        't_map_deliveries',
        NOW()
    FROM (
        SELECT DISTINCT
            COALESCE(d.shipping_partner, 'n.a.') AS shipping_partner_src_id,
            COALESCE(d.shipping_partner, 'n.a.') AS shipping_partner_name
        FROM bl_cl.t_map_deliveries d
        WHERE COALESCE(d.shipping_partner, 'n.a.') <> 'n.a.'
    ) s
    WHERE NOT EXISTS (
        SELECT 1
        FROM bl_3nf.ce_shipping_partners t
        WHERE t.shipping_partner_src_id = s.shipping_partner_src_id
    );


    GET DIAGNOSTICS v_ins = ROW_COUNT;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_3nf.ce_shipping_partners',
        v_ins,
        CASE WHEN v_ins > 0 THEN 'SUCCESS' ELSE 'NO_CHANGE' END,
        'Inserted=' || v_ins || '. Loaded unique shipping partners from bl_cl.t_map_deliveries.'
    );

    RAISE NOTICE 'ce_shipping_partners completed. inserted=%', v_ins;

EXCEPTION
WHEN OTHERS THEN
    GET STACKED DIAGNOSTICS
        v_err_detail = PG_EXCEPTION_DETAIL,
        v_err_hint   = PG_EXCEPTION_HINT;

    v_err_msg := SQLERRM;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_3nf.ce_shipping_partners',
        0,
        'FAILED',
        'Shipping partners load failed',
        v_err_msg || ' | DETAIL=' || COALESCE(v_err_detail, 'n.a.')
                  || ' | HINT='   || COALESCE(v_err_hint, 'n.a.'),
        'ERROR'
    );

    RAISE;
END;
$$;

SQL

CREATE SEQUENCE
CREATE TABLE
CREATE INDEX
INSERT 0 1
CREATE PROCEDURE


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CALL bl_cl.load_ce_shipping_partners();


SQL

CALL


NOTICE:  ce_shipping_partners completed. inserted=4


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'


CREATE SEQUENCE IF NOT EXISTS bl_3nf.seq_ce_delivery_id START 1;
CREATE TABLE IF NOT EXISTS bl_3nf.ce_deliveries (
    delivery_id           BIGINT PRIMARY KEY,
    delivery_src_id        VARCHAR(100) NOT NULL,
    shipping_partner_id   BIGINT NOT NULL,
    delivery_type         VARCHAR(100) NOT NULL,
    delivery_status       VARCHAR(100) NOT NULL,
    source_system         VARCHAR(100) NOT NULL,
    source_table          VARCHAR(100) NOT NULL,
    insert_dt             TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,

    CONSTRAINT fk_ce_deliveries_shipping_partner
        FOREIGN KEY (shipping_partner_id)
        REFERENCES bl_3nf.ce_shipping_partners (shipping_partner_id),
   CONSTRAINT ux_ce_deliveries_src
        UNIQUE (delivery_src_id)
);

CREATE INDEX IF NOT EXISTS ix_ce_deliveries_partner_id
ON bl_3nf.ce_deliveries (shipping_partner_id);

CREATE INDEX IF NOT EXISTS ix_ce_deliveries_type
ON bl_3nf.ce_deliveries (delivery_type);


INSERT INTO bl_3nf.ce_deliveries (
    delivery_id,
    delivery_src_id,
    shipping_partner_id,
    delivery_type,
    delivery_status,
    source_system,
    source_table,
    insert_dt
)
SELECT
    -1,
    'n.a.',
    -1,
    'n.a.',
    'n.a.',
    'MANUAL',
    'MANUAL',
    NOW()
WHERE NOT EXISTS (
    SELECT 1
    FROM bl_3nf.ce_deliveries
    WHERE delivery_id = -1
);

CREATE OR REPLACE PROCEDURE bl_cl.load_ce_deliveries()
LANGUAGE plpgsql
AS $$
DECLARE
    v_proc        TEXT := 'bl_cl.load_ce_deliveries';
    v_ins         INT  := 0;
    v_err_msg     TEXT;
    v_err_detail  TEXT;
    v_err_hint    TEXT;
BEGIN
       DROP TABLE IF EXISTS tmp_final_deliveries;

    CREATE TEMP TABLE tmp_final_deliveries
    ON COMMIT DROP
    AS
    WITH src AS (
        SELECT
            /* 🔹 RAW KEY (traceability only) */
            COALESCE(d.delivery_id, 'n.a.') AS delivery_src_id,

            COALESCE(d.shipping_partner, 'n.a.') AS shipping_partner_src_id,
            COALESCE(d.delivery_type, 'n.a.') AS delivery_type,
            COALESCE(d.delivery_status, 'n.a.') AS delivery_status,

            'bl_cl' AS source_system,
            't_map_deliveries' AS source_table
        FROM bl_cl.t_map_deliveries d
        WHERE COALESCE(d.delivery_id, 'n.a.') <> 'n.a.'
    ),

    ranked AS (
        SELECT
            s.*,
            ROW_NUMBER() OVER (
                PARTITION BY s.delivery_src_id
                ORDER BY
                    s.delivery_status,
                    s.delivery_type,
                    s.shipping_partner_src_id
            ) AS rn
        FROM src s
    ),

    resolved_partner AS (
        SELECT
            r.delivery_src_id,
            COALESCE(sp.shipping_partner_id, -1) AS shipping_partner_id,
            r.delivery_type,
            r.delivery_status,
            r.source_system,
            r.source_table
        FROM ranked r
        LEFT JOIN bl_3nf.ce_shipping_partners sp
            ON sp.shipping_partner_src_id = r.shipping_partner_src_id
        WHERE r.rn = 1
    )

    SELECT
        delivery_src_id,
        shipping_partner_id,
        delivery_type,
        delivery_status,
        source_system,
        source_table
    FROM resolved_partner;

    INSERT INTO bl_3nf.ce_deliveries (
        delivery_id,
        delivery_src_id,
        shipping_partner_id,
        delivery_type,
        delivery_status,
        source_system,
        source_table,
        insert_dt
    )
    SELECT
        nextval('bl_3nf.seq_ce_delivery_id'),
        f.delivery_src_id,
        f.shipping_partner_id,
        f.delivery_type,
        f.delivery_status,
        f.source_system,
        f.source_table,
        NOW()
    FROM tmp_final_deliveries f
    WHERE NOT EXISTS (
        SELECT 1
        FROM bl_3nf.ce_deliveries ce
        WHERE ce.delivery_src_id = f.delivery_src_id
    );

    GET DIAGNOSTICS v_ins = ROW_COUNT;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_3nf.ce_deliveries',
        v_ins,
        CASE WHEN v_ins > 0 THEN 'SUCCESS' ELSE 'NO_CHANGE' END,
        'Inserted=' || v_ins ||
        '. Source=bl_cl.t_map_deliveries; Type 0 load; one row per delivery_src_id; shipping_partner FK resolved from ce_shipping_partners.'
    );

    RAISE NOTICE 'ce_deliveries completed. inserted=%', v_ins;

EXCEPTION
WHEN OTHERS THEN
    GET STACKED DIAGNOSTICS
        v_err_detail = PG_EXCEPTION_DETAIL,
        v_err_hint   = PG_EXCEPTION_HINT;

    v_err_msg := SQLERRM;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_3nf.ce_deliveries',
        0,
        'FAILED',
        'Deliveries 3nf load failed',
        v_err_msg || ' | DETAIL=' || COALESCE(v_err_detail, 'n.a.')
                  || ' | HINT='   || COALESCE(v_err_hint, 'n.a.'),
        'ERROR'
    );

    RAISE;
END;
$$;

SQL

CREATE SEQUENCE
CREATE TABLE
CREATE INDEX
CREATE INDEX
INSERT 0 1
CREATE PROCEDURE


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CALL bl_cl.load_ce_deliveries();

SQL

CALL


NOTICE:  table "tmp_final_deliveries" does not exist, skipping
NOTICE:  ce_deliveries completed. inserted=950000


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CREATE SEQUENCE IF NOT EXISTS bl_dm.seq_dim_delivery_id START 1;

CREATE TABLE IF NOT EXISTS bl_dm.dim_deliveries (
    delivery_surr_id      BIGINT PRIMARY KEY,
    delivery_src_id       BIGINT NOT NULL,   -- 3NF surrogate: ce_deliveries.delivery_id

    source_system         VARCHAR(100) NOT NULL,
    source_table          VARCHAR(100) NOT NULL,

    delivery_id_nk        VARCHAR(100) NOT NULL,   -- 3NF business key: ce_deliveries.delivery_src_id
    delivery_type         VARCHAR(100) NOT NULL,
    delivery_status       VARCHAR(100) NOT NULL,
    shipping_partner_name VARCHAR(100) NOT NULL,

    insert_dt             TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
    update_dt             TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP
);

CREATE UNIQUE INDEX IF NOT EXISTS ux_dim_deliveries_src_id
    ON bl_dm.dim_deliveries (delivery_src_id);

CREATE INDEX IF NOT EXISTS ix_dim_deliveries_nk
    ON bl_dm.dim_deliveries (delivery_id_nk);

CREATE INDEX IF NOT EXISTS ix_dim_deliveries_type
    ON bl_dm.dim_deliveries (delivery_type);

CREATE INDEX IF NOT EXISTS ix_dim_deliveries_status
    ON bl_dm.dim_deliveries (delivery_status);

CREATE INDEX IF NOT EXISTS ix_dim_deliveries_partner
    ON bl_dm.dim_deliveries (shipping_partner_name);

INSERT INTO bl_dm.dim_deliveries (
    delivery_surr_id,
    delivery_src_id,
    source_system,
    source_table,
    delivery_id_nk,
    delivery_type,
    delivery_status,
    shipping_partner_name,
    insert_dt,
    update_dt
)
SELECT
    -1,
    -1,
    'MANUAL',
    'MANUAL',
    'n.a.',
    'n.a.',
    'n.a.',
    'n.a.',
    NOW(),
    NOW()
WHERE NOT EXISTS (
    SELECT 1
    FROM bl_dm.dim_deliveries
    WHERE delivery_surr_id = -1
);

CREATE OR REPLACE PROCEDURE bl_cl.load_dim_deliveries()
LANGUAGE plpgsql
AS $$
DECLARE
    v_proc        TEXT := 'bl_cl.load_dim_deliveries';
    v_ins         INT  := 0;
    v_upd         INT  := 0;
    v_err_msg     TEXT;
    v_err_detail  TEXT;
    v_err_hint    TEXT;
BEGIN
    DROP TABLE IF EXISTS tmp_dim_deliveries;

    CREATE TEMP TABLE tmp_dim_deliveries
    ON COMMIT DROP
    AS
    SELECT
        d.delivery_id AS delivery_src_id,
        'bl_3nf'::VARCHAR(100) AS source_system,
        'ce_deliveries'::VARCHAR(100) AS source_table,
        COALESCE(d.delivery_src_id, 'n.a.') AS delivery_id_nk,
        COALESCE(d.delivery_type, 'n.a.') AS delivery_type,
        COALESCE(d.delivery_status, 'n.a.') AS delivery_status,
        COALESCE(sp.shipping_partner_name, 'n.a.') AS shipping_partner_name
    FROM bl_3nf.ce_deliveries d
    LEFT JOIN bl_3nf.ce_shipping_partners sp
        ON sp.shipping_partner_id = d.shipping_partner_id
    WHERE d.delivery_id <> -1;

    CREATE INDEX idx_tmp_dim_deliveries_src
        ON tmp_dim_deliveries (delivery_src_id);

    /* INSERT */
    INSERT INTO bl_dm.dim_deliveries (
        delivery_surr_id,
        delivery_src_id,
        source_system,
        source_table,
        delivery_id_nk,
        delivery_type,
        delivery_status,
        shipping_partner_name,
        insert_dt,
        update_dt
    )
    SELECT
        nextval('bl_dm.seq_dim_delivery_id'),
        t.delivery_src_id,
        t.source_system,
        t.source_table,
        t.delivery_id_nk,
        t.delivery_type,
        t.delivery_status,
        t.shipping_partner_name,
        NOW(),
        NOW()
    FROM tmp_dim_deliveries t
    WHERE NOT EXISTS (
        SELECT 1
        FROM bl_dm.dim_deliveries d
        WHERE d.delivery_src_id = t.delivery_src_id
    );

    GET DIAGNOSTICS v_ins = ROW_COUNT;

    /* UPDATE TYPE 1 */
    UPDATE bl_dm.dim_deliveries d
    SET
        source_system         = t.source_system,
        source_table          = t.source_table,
        delivery_id_nk        = t.delivery_id_nk,
        delivery_type         = t.delivery_type,
        delivery_status       = t.delivery_status,
        shipping_partner_name = t.shipping_partner_name,
        update_dt             = NOW()
    FROM tmp_dim_deliveries t
    WHERE d.delivery_src_id = t.delivery_src_id
      AND (
            d.source_system         IS DISTINCT FROM t.source_system
         OR d.source_table          IS DISTINCT FROM t.source_table
         OR d.delivery_id_nk        IS DISTINCT FROM t.delivery_id_nk
         OR d.delivery_type         IS DISTINCT FROM t.delivery_type
         OR d.delivery_status       IS DISTINCT FROM t.delivery_status
         OR d.shipping_partner_name IS DISTINCT FROM t.shipping_partner_name
      );

    GET DIAGNOSTICS v_upd = ROW_COUNT;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_dm.dim_deliveries',
        (v_ins + v_upd),
        CASE WHEN (v_ins + v_upd) > 0 THEN 'SUCCESS' ELSE 'NO_CHANGE' END,
        'Inserted=' || v_ins || ', Updated=' || v_upd ||
        '. Source=bl_3nf.ce_deliveries + ce_shipping_partners.',
        NULL,
        'INFO',
        v_ins,
        v_upd,
        0,
        0,
        NULL
    );

    RAISE NOTICE 'dim_deliveries completed. inserted=%, updated=%', v_ins, v_upd;

EXCEPTION
WHEN OTHERS THEN
    GET STACKED DIAGNOSTICS
        v_err_detail = PG_EXCEPTION_DETAIL,
        v_err_hint   = PG_EXCEPTION_HINT;

    v_err_msg := SQLERRM;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_dm.dim_deliveries',
        0,
        'FAILED',
        'dim_deliveries load failed',
        v_err_msg || ' | DETAIL=' || COALESCE(v_err_detail, 'n.a.')
                  || ' | HINT='   || COALESCE(v_err_hint, 'n.a.'),
        'ERROR',
        0, 0, 0, 0, NULL
    );

    RAISE;
END;
$$;


SQL

CREATE SEQUENCE
CREATE TABLE
CREATE INDEX
CREATE INDEX
CREATE INDEX
CREATE INDEX
CREATE INDEX
INSERT 0 1
CREATE PROCEDURE


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CALL bl_cl.load_dim_deliveries();

SQL

CALL


NOTICE:  table "tmp_dim_deliveries" does not exist, skipping
NOTICE:  dim_deliveries completed. inserted=950000, updated=0


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

--For the engagement entity, I preserved the raw engagement_id as the business key because source profiling showed
--that engagement behaves as a row-level synthetic identifier within the transaction-line dataset,
--making additional composite key engineering unnecessary and analytically unjustified.


DROP TABLE IF EXISTS bl_cl.t_map_engagements;

CREATE TABLE IF NOT EXISTS bl_cl.t_map_engagements (
    engagement_id              VARCHAR(100),
    customer_support_calls     INTEGER,
    website_address            VARCHAR(250),
    order_channel              VARCHAR(100),
    customer_support_method    VARCHAR(100),
    issue_status               VARCHAR(100),
    app_usage                  VARCHAR(100),
    website_visits             INTEGER,
    social_media_engagement    VARCHAR(100),
    source_system              VARCHAR(100),
    source_table               VARCHAR(100),
    insert_dt                  TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE INDEX IF NOT EXISTS idx_t_map_engagements_src
ON bl_cl.t_map_engagements (engagement_id, source_system, source_table);

CREATE OR REPLACE PROCEDURE bl_cl.load_map_engagements()
LANGUAGE plpgsql
AS $$
DECLARE
    v_proc        TEXT := 'bl_cl.load_map_engagements';
    v_ins         INT  := 0;
    v_err_msg     TEXT;
    v_err_detail  TEXT;
    v_err_hint    TEXT;
BEGIN
    WITH prepared_source AS (
        SELECT
            COALESCE(NULLIF(src.engagement_id,''),'n.a.') AS engagement_id,
            src.customer_support_calls,
            src.website_visits,
            COALESCE(src.website_address,'n.a.') AS website_address,
            COALESCE(src.order_channel,'n.a.') AS order_channel,
            COALESCE(src.customer_support_method,'n.a.') AS customer_support_method,
            COALESCE(src.issue_status,'n.a.') AS issue_status,
            COALESCE(src.app_usage,'n.a.') AS app_usage,
            COALESCE(src.social_media_engagement,'n.a.') AS social_media_engagement,
            'sa_online_retail' AS source_system,
            'src_online_retail' AS source_table
        FROM sa_online_retail.src_online_retail src
    ),
    distinct_source AS (
        SELECT DISTINCT
            engagement_id,
            customer_support_calls,
            website_address,
            order_channel,
            customer_support_method,
            issue_status,
            app_usage,
            website_visits,
            social_media_engagement,
            source_system,
            source_table
        FROM prepared_source
        WHERE engagement_id <> 'n.a.'
    )
    INSERT INTO bl_cl.t_map_engagements (
        engagement_id,
        customer_support_calls,
        website_address,
        order_channel,
        customer_support_method,
        issue_status,
        app_usage,
        website_visits,
        social_media_engagement,
        source_system,
        source_table
    )
    SELECT
        s.engagement_id,
        s.customer_support_calls,
        s.website_address,
        s.order_channel,
        s.customer_support_method,
        s.issue_status,
        s.app_usage,
        s.website_visits,
        s.social_media_engagement,
        s.source_system,
        s.source_table
    FROM distinct_source s
    WHERE NOT EXISTS (
        SELECT 1
        FROM bl_cl.t_map_engagements t
        WHERE t.engagement_id = s.engagement_id
          AND t.source_system = s.source_system
          AND t.source_table  = s.source_table
    );

    GET DIAGNOSTICS v_ins = ROW_COUNT;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_cl.t_map_engagements',
        v_ins,
        CASE WHEN v_ins > 0 THEN 'SUCCESS' ELSE 'NO_CHANGE' END,
        'Inserted=' || v_ins || '. Loaded online engagement map rows.'
    );

    RAISE NOTICE 't_map_engagements completed. inserted=%', v_ins;

EXCEPTION
WHEN OTHERS THEN
    GET STACKED DIAGNOSTICS
        v_err_detail = PG_EXCEPTION_DETAIL,
        v_err_hint   = PG_EXCEPTION_HINT;

    v_err_msg := SQLERRM;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_cl.t_map_engagements',
        0,
        'FAILED',
        'Engagement map load failed',
        v_err_msg || ' | DETAIL=' || COALESCE(v_err_detail,'n.a.')
                  || ' | HINT='   || COALESCE(v_err_hint,'n.a.'),
        'ERROR'
    );

    RAISE;
END;
$$;


SQL

DROP TABLE
CREATE TABLE
CREATE INDEX
CREATE PROCEDURE


NOTICE:  table "t_map_engagements" does not exist, skipping


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CALL bl_cl.load_map_engagements();

SQL

CALL


NOTICE:  t_map_engagements completed. inserted=475000


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CREATE SEQUENCE IF NOT EXISTS bl_3nf.seq_ce_engagement_id START 1;
CREATE TABLE IF NOT EXISTS bl_3nf.ce_engagements (
    engagement_id              BIGINT PRIMARY KEY,
    engagement_src_id          VARCHAR(100) NOT NULL,
    customer_support_calls     INTEGER,
    website_address            VARCHAR(250) NOT NULL,
    order_channel              VARCHAR(100) NOT NULL,
    customer_support_method    VARCHAR(100) NOT NULL,
    issue_status               VARCHAR(100) NOT NULL,
    app_usage                  VARCHAR(100) NOT NULL,
    website_visits             INTEGER,
    social_media_engagement    VARCHAR(100) NOT NULL,
    source_system              VARCHAR(100) NOT NULL,
    source_table               VARCHAR(100) NOT NULL,
    insert_dt                  TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
    CONSTRAINT ux_ce_engagements_src
    UNIQUE (engagement_src_id)
);

CREATE INDEX IF NOT EXISTS ix_ce_engagements_src
ON bl_3nf.ce_engagements (engagement_src_id);

INSERT INTO bl_3nf.ce_engagements (
    engagement_id,
    engagement_src_id,
    customer_support_calls,
    website_address,
    order_channel,
    customer_support_method,
    issue_status,
    app_usage,
    website_visits,
    social_media_engagement,
    source_system,
    source_table,
    insert_dt
)
SELECT
    -1,
    'n.a.',
    0,
    'n.a.',
    'n.a.',
    'n.a.',
    'n.a.',
    'n.a.',
    0,
    'n.a.',
    'MANUAL',
    'MANUAL',
    NOW()
WHERE NOT EXISTS (
    SELECT 1
    FROM bl_3nf.ce_engagements
    WHERE engagement_id = -1
);

CREATE OR REPLACE PROCEDURE bl_cl.load_ce_engagements()
LANGUAGE plpgsql
AS $$
DECLARE
    v_proc        TEXT := 'bl_cl.load_ce_engagements';
    v_ins         INT  := 0;
    v_err_msg     TEXT;
    v_err_detail  TEXT;
    v_err_hint    TEXT;
BEGIN
    DROP TABLE IF EXISTS tmp_final_engagements;

    CREATE TEMP TABLE tmp_final_engagements
    ON COMMIT DROP
    AS
    WITH ranked AS (
        SELECT
            /*  TRUE KEY */
            e.engagement_id AS engagement_src_id,

            e.customer_support_calls,
            e.website_address,
            e.order_channel,
            e.customer_support_method,
            e.issue_status,
            e.app_usage,
            e.website_visits,
            e.social_media_engagement,

            ROW_NUMBER() OVER (
                PARTITION BY e.engagement_id
                ORDER BY e.website_visits DESC NULLS LAST
            ) AS rn

        FROM bl_cl.t_map_engagements e
        WHERE e.engagement_id <> 'n.a.'
    )
    SELECT
        engagement_src_id,
        customer_support_calls,
        website_address,
        order_channel,
        customer_support_method,
        issue_status,
        app_usage,
        website_visits,
        social_media_engagement,
        'bl_cl'::VARCHAR(100) AS source_system,
        't_map_engagements'::VARCHAR(100) AS source_table
    FROM ranked
    WHERE rn = 1;

    CREATE INDEX idx_tmp_final_engagements_src
        ON tmp_final_engagements (engagement_src_id);

    /* ================= INSERT ONLY (TYPE 0) ================= */
    INSERT INTO bl_3nf.ce_engagements (
        engagement_id,
        engagement_src_id,
        customer_support_calls,
        website_address,
        order_channel,
        customer_support_method,
        issue_status,
        app_usage,
        website_visits,
        social_media_engagement,
        source_system,
        source_table,
        insert_dt
            )
    SELECT
        nextval('bl_3nf.seq_ce_engagement_id'),
        f.engagement_src_id,
        f.customer_support_calls,
        f.website_address,
        f.order_channel,
        f.customer_support_method,
        f.issue_status,
        f.app_usage,
        f.website_visits,
        f.social_media_engagement,
        f.source_system,
        f.source_table,
        NOW()
    FROM tmp_final_engagements f
    WHERE NOT EXISTS (
        SELECT 1
        FROM bl_3nf.ce_engagements ce
        WHERE ce.engagement_src_id = f.engagement_src_id
    );

    GET DIAGNOSTICS v_ins = ROW_COUNT;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_3nf.ce_engagements',
        v_ins,
        CASE WHEN v_ins > 0 THEN 'SUCCESS' ELSE 'NO_CHANGE' END,
        'Inserted=' || v_ins ||'. Source = bl_cl.t_map_engagements; Type 0 load. one row per engagement.'
    );

    RAISE NOTICE 'ce_engagements completed. inserted=%', v_ins;

EXCEPTION
WHEN OTHERS THEN
    GET STACKED DIAGNOSTICS
        v_err_detail = PG_EXCEPTION_DETAIL,
        v_err_hint   = PG_EXCEPTION_HINT;

    v_err_msg := SQLERRM;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_3nf.ce_engagements',
        0,
        'FAILED',
        'Engagements 3nf load failed',
        v_err_msg || ' | DETAIL=' || COALESCE(v_err_detail, 'n.a.')
                  || ' | HINT='   || COALESCE(v_err_hint, 'n.a.'),
        'ERROR'
    );

    RAISE;
END;
$$;

SQL

CREATE SEQUENCE
CREATE TABLE
CREATE INDEX
INSERT 0 1
CREATE PROCEDURE


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CALL bl_cl.load_ce_engagements();

SQL

CALL


NOTICE:  table "tmp_final_engagements" does not exist, skipping
NOTICE:  ce_engagements completed. inserted=475000


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CREATE SEQUENCE IF NOT EXISTS bl_dm.seq_dim_engagement_id START 1;

CREATE TABLE IF NOT EXISTS bl_dm.dim_engagements (
    engagement_surr_id         BIGINT PRIMARY KEY,
    engagement_src_id          BIGINT NOT NULL,   -- 3NF surrogate: ce_engagements.engagement_id

    source_system              VARCHAR(100) NOT NULL,
    source_table               VARCHAR(100) NOT NULL,

    engagement_id_nk           VARCHAR(100) NOT NULL,
    customer_support_calls     INTEGER,
    website_address            VARCHAR(250) NOT NULL,
    order_channel              VARCHAR(100) NOT NULL,
    customer_support_method    VARCHAR(100) NOT NULL,
    issue_status               VARCHAR(100) NOT NULL,
    app_usage                  VARCHAR(100) NOT NULL,
    website_visits             INTEGER,
    social_media_engagement    VARCHAR(100) NOT NULL,

    insert_dt                  TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
    update_dt                  TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP
);

CREATE UNIQUE INDEX IF NOT EXISTS ux_dim_engagements_src_id
    ON bl_dm.dim_engagements (engagement_src_id);

CREATE INDEX IF NOT EXISTS ix_dim_engagements_order_channel
    ON bl_dm.dim_engagements (order_channel);

CREATE INDEX IF NOT EXISTS ix_dim_engagements_issue_status
    ON bl_dm.dim_engagements (issue_status);

CREATE INDEX IF NOT EXISTS ix_dim_engagements_support_method
    ON bl_dm.dim_engagements (customer_support_method);

INSERT INTO bl_dm.dim_engagements (
    engagement_surr_id,
    engagement_src_id,
    source_system,
    source_table,
    engagement_id_nk,
    customer_support_calls,
    website_address,
    order_channel,
    customer_support_method,
    issue_status,
    app_usage,
    website_visits,
    social_media_engagement,
    insert_dt,
    update_dt
)
SELECT
    -1,
    -1,
    'MANUAL',
    'MANUAL',
    'n.a.',
    0,
    'n.a.',
    'n.a.',
    'n.a.',
    'n.a.',
    'n.a.',
    0,
    'n.a.',
    NOW(),
    NOW()
WHERE NOT EXISTS (
    SELECT 1
    FROM bl_dm.dim_engagements
    WHERE engagement_surr_id = -1
);

CREATE OR REPLACE PROCEDURE bl_cl.load_dim_engagements()
LANGUAGE plpgsql
AS $$
DECLARE
    v_proc        TEXT := 'bl_cl.load_dim_engagements';
    v_ins         INT  := 0;
    v_upd         INT  := 0;
    v_err_msg     TEXT;
    v_err_detail  TEXT;
    v_err_hint    TEXT;
BEGIN
    /* INSERT new engagement rows from 3NF into DM */
    INSERT INTO bl_dm.dim_engagements (
        engagement_surr_id,
        engagement_src_id,
        source_system,
        source_table,
        engagement_id_nk,
        customer_support_calls,
        website_address,
        order_channel,
        customer_support_method,
        issue_status,
        app_usage,
        website_visits,
        social_media_engagement,
        insert_dt,
        update_dt
    )
    SELECT
        nextval('bl_dm.seq_dim_engagement_id'),
        e.engagement_id,            -- 3NF surrogate
        'bl_3nf',
        'ce_engagements',
        e.engagement_src_id,
        COALESCE(e.customer_support_calls, 0),
        e.website_address,
        e.order_channel,
        e.customer_support_method,
        e.issue_status,
        e.app_usage,
        COALESCE(e.website_visits, 0),
        e.social_media_engagement,
        NOW(),
        NOW()
    FROM bl_3nf.ce_engagements e
    WHERE e.engagement_id <> -1
      AND NOT EXISTS (
          SELECT 1
          FROM bl_dm.dim_engagements d
          WHERE d.engagement_src_id = e.engagement_id
      );

    GET DIAGNOSTICS v_ins = ROW_COUNT;

    /* TYPE 1 UPDATE */
    UPDATE bl_dm.dim_engagements d
    SET
        engagement_id_nk        = s.engagement_id_nk,
        customer_support_calls  = s.customer_support_calls,
        website_address         = s.website_address,
        order_channel           = s.order_channel,
        customer_support_method = s.customer_support_method,
        issue_status            = s.issue_status,
        app_usage               = s.app_usage,
        website_visits          = s.website_visits,
        social_media_engagement = s.social_media_engagement,
        update_dt               = NOW()
    FROM (
        SELECT
            e.engagement_id AS engagement_src_id,
            e.engagement_src_id AS engagement_id_nk,
            COALESCE(e.customer_support_calls, 0) AS customer_support_calls,
            e.website_address,
            e.order_channel,
            e.customer_support_method,
            e.issue_status,
            e.app_usage,
            COALESCE(e.website_visits, 0) AS website_visits,
            e.social_media_engagement
        FROM bl_3nf.ce_engagements e
        WHERE e.engagement_id <> -1
    ) s
    WHERE d.engagement_src_id = s.engagement_src_id
      AND (
            d.engagement_id_nk        IS DISTINCT FROM s.engagement_id_nk
         OR d.customer_support_calls  IS DISTINCT FROM s.customer_support_calls
         OR d.website_address         IS DISTINCT FROM s.website_address
         OR d.order_channel           IS DISTINCT FROM s.order_channel
         OR d.customer_support_method IS DISTINCT FROM s.customer_support_method
         OR d.issue_status            IS DISTINCT FROM s.issue_status
         OR d.app_usage               IS DISTINCT FROM s.app_usage
         OR d.website_visits          IS DISTINCT FROM s.website_visits
         OR d.social_media_engagement IS DISTINCT FROM s.social_media_engagement
      );

    GET DIAGNOSTICS v_upd = ROW_COUNT;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_dm.dim_engagements',
        (v_ins + v_upd),
        CASE WHEN (v_ins + v_upd) > 0 THEN 'SUCCESS' ELSE 'NO_CHANGE' END,
        'Inserted=' || v_ins || ', Updated=' || v_upd ||
        '. Source=bl_3nf.ce_engagements.',
        NULL,
        'INFO',
        v_ins,
        v_upd,
        0,
        0,
        NULL
    );

    RAISE NOTICE 'dim_engagements completed. inserted=%, updated=%', v_ins, v_upd;

EXCEPTION
WHEN OTHERS THEN
    GET STACKED DIAGNOSTICS
        v_err_detail = PG_EXCEPTION_DETAIL,
        v_err_hint   = PG_EXCEPTION_HINT;

    v_err_msg := SQLERRM;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_dm.dim_engagements',
        0,
        'FAILED',
        'dim_engagements load failed',
        v_err_msg || ' | DETAIL=' || COALESCE(v_err_detail, 'n.a.')
                  || ' | HINT='   || COALESCE(v_err_hint, 'n.a.'),
        'ERROR',
        0, 0, 0, 0, NULL
    );

    RAISE;
END;
$$;


SQL

CREATE SEQUENCE
CREATE TABLE
CREATE INDEX
CREATE INDEX
CREATE INDEX
CREATE INDEX
INSERT 0 1
CREATE PROCEDURE


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'


CALL bl_cl.load_dim_engagements();
SQL

CALL


NOTICE:  dim_engagements completed. inserted=475000, updated=0


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

-- In the employee SCD Type 2 design, the sequence is used only when a completely new employee business key appears for the first time.
-- If an existing employee changes one or more tracked attributes, the process does not generate a new business identifier.
-- Instead, it closes the previous version and inserts a new historical version under the same employee_id.
-- This preserves the employee entity identity across time while allowing multiple temporal versions of that employee record.


DROP TABLE IF EXISTS bl_cl.t_map_employees;
CREATE TABLE IF NOT EXISTS bl_cl.t_map_employees (
    employee_src_id     VARCHAR(100) NOT NULL,
    employee_name       VARCHAR(100),
    employee_position   VARCHAR(100),
    employee_salary     NUMERIC(10,2),
    employee_hire_date  DATE,
    observed_ts         TIMESTAMP,
    source_system       VARCHAR(100),
    source_table        VARCHAR(100),
    insert_dt           TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE INDEX IF NOT EXISTS ix_map_employees_src
ON bl_cl.t_map_employees (employee_src_id, source_system, source_table);

CREATE INDEX IF NOT EXISTS ix_map_employees_version
ON bl_cl.t_map_employees (
    employee_src_id,
    employee_position,
    employee_hire_date,
    observed_ts
);

CREATE OR REPLACE PROCEDURE bl_cl.load_map_employees()
LANGUAGE plpgsql
AS $$
DECLARE
    v_proc        TEXT := 'bl_cl.load_map_employees';
    v_ins         INT  := 0;
    v_err_msg     TEXT;
    v_err_detail  TEXT;
    v_err_hint    TEXT;
BEGIN

    DROP TABLE IF EXISTS tmp_employee_map_source;

    CREATE TEMP TABLE tmp_employee_map_source (
        employee_src_id     VARCHAR(100),
        employee_name       VARCHAR(100),
        employee_position   VARCHAR(100),
        employee_salary     NUMERIC(10,2),
        employee_hire_date  DATE,
        observed_ts         TIMESTAMP,
        source_system       VARCHAR(100),
        source_table        VARCHAR(100)
    ) ON COMMIT DROP;

    /* =========================
       ONLINE NOT USED → SAME (kept offline only)
       burayı aynı bıraktım
       ========================= */

    INSERT INTO tmp_employee_map_source
    SELECT DISTINCT
        src.employee_name || '-' || src.employee_hire_date AS employee_src_id,   --  (clean staging → no re-clean)
        src.employee_name,
        src.employee_position,
        src.employee_salary,
        src.employee_hire_date,
        COALESCE(src.transaction_dt, TIMESTAMP '1900-01-01') AS observed_ts,  -- burayı değiştirdim (clean column + null safe)
        'sa_offline_retail' AS source_system,
        'src_offline_retail' AS source_table
    FROM sa_offline_retail.src_offline_retail src
    WHERE COALESCE(src.employee_name, 'n.a.') IS NOT NULL;

    /* =========================
       INCREMENTAL SOURCE (optional)
       burayı aynı bıraktım (logic correct)
       ========================= */

    IF to_regclass('sa_offline_retail.src_offline_retail_employee_inc') IS NOT NULL THEN

        INSERT INTO tmp_employee_map_source
        SELECT DISTINCT
            inc.employee_name || '-' || inc.employee_hire_date AS employee_src_id,
            inc.employee_name,
            inc.employee_position,
            inc.employee_salary,
            inc.employee_hire_date,

            COALESCE(inc.transaction_dt, CURRENT_TIMESTAMP) AS observed_ts,  -- burayı değiştirdim

            'sa_offline_retail',
            'src_offline_retail_employee_inc'
        FROM sa_offline_retail.src_offline_retail_employee_inc inc
        WHERE inc.employee_name IS NOT NULL;

    END IF;

    /* =========================
       FINAL INSERT (SCD2 READY)
       burayı büyük ölçüde aynı bıraktım
       ========================= */

    INSERT INTO bl_cl.t_map_employees (
        employee_src_id,
        employee_name,
        employee_position,
        employee_salary,
        employee_hire_date,
        observed_ts,
        source_system,
        source_table
    )
    SELECT
        s.employee_src_id,
        s.employee_name,
        s.employee_position,
        s.employee_salary,
        s.employee_hire_date,
        s.observed_ts,
        s.source_system,
        s.source_table
    FROM tmp_employee_map_source s
    WHERE s.employee_src_id IS NOT NULL

      AND NOT EXISTS (
    SELECT 1
    FROM bl_cl.t_map_employees t
    WHERE t.employee_src_id = s.employee_src_id
      AND COALESCE(t.employee_position, 'n.a.')
          = COALESCE(s.employee_position, 'n.a.')
      AND COALESCE(t.employee_salary, -1)
          = COALESCE(s.employee_salary, -1)
      AND COALESCE(t.employee_hire_date, DATE '1900-01-01')
          = COALESCE(s.employee_hire_date, DATE '1900-01-01')
      AND COALESCE(t.observed_ts, TIMESTAMP '1900-01-01 00:00:00')
          = COALESCE(s.observed_ts, TIMESTAMP '1900-01-01 00:00:00')
      AND t.source_system = s.source_system
      AND t.source_table  = s.source_table
);

    GET DIAGNOSTICS v_ins = ROW_COUNT;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_cl.t_map_employees',
        v_ins,
        CASE WHEN v_ins > 0 THEN 'SUCCESS' ELSE 'NO_CHANGE' END,
        'Inserted=' || v_ins || '. Loaded employee version rows (SCD2-ready)'
    );

    RAISE NOTICE 't_map_employees completed. inserted=%', v_ins;

EXCEPTION
WHEN OTHERS THEN

    GET STACKED DIAGNOSTICS
        v_err_detail = PG_EXCEPTION_DETAIL,
        v_err_hint   = PG_EXCEPTION_HINT;

    v_err_msg := SQLERRM;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_cl.t_map_employees',
        0,
        'FAILED',
        'Employee map load failed',
        v_err_msg || ' | DETAIL=' || COALESCE(v_err_detail, 'n.a.')
                  || ' | HINT='   || COALESCE(v_err_hint, 'n.a.'),
        'ERROR'
    );

    RAISE;

END;
$$;

SQL

DROP TABLE
CREATE TABLE
CREATE INDEX
CREATE INDEX
CREATE PROCEDURE


NOTICE:  table "t_map_employees" does not exist, skipping


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CALL bl_cl.load_map_employees();

SQL

CALL


NOTICE:  table "tmp_employee_map_source" does not exist, skipping
NOTICE:  t_map_employees completed. inserted=475000


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CREATE SEQUENCE IF NOT EXISTS bl_3nf.seq_ce_employee_id START 1;

CREATE TABLE IF NOT EXISTS bl_3nf.ce_employees_scd (
    employee_id        BIGINT,
    employee_src_id    VARCHAR(255) NOT NULL,
    employee_name      VARCHAR(100),
    employee_position  VARCHAR(100),
    employee_salary    NUMERIC(10,2),
    employee_hire_date DATE,
    start_dt           TIMESTAMP NOT NULL,
    end_dt             TIMESTAMP NOT NULL,
    is_active          BOOLEAN NOT NULL,
    source_system      VARCHAR(100),
    source_table       VARCHAR(100),
    insert_dt          TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    update_dt          TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    CONSTRAINT pk_ce_emp PRIMARY KEY (employee_id, start_dt)
);

CREATE UNIQUE INDEX IF NOT EXISTS uq_ce_emp_active
ON bl_3nf.ce_employees_scd (employee_src_id)
WHERE is_active = TRUE;

CREATE INDEX IF NOT EXISTS ix_ce_emp_src_start
ON bl_3nf.ce_employees_scd (employee_src_id, start_dt);

CREATE INDEX IF NOT EXISTS ix_ce_emp_lookup
ON bl_3nf.ce_employees_scd (employee_src_id, is_active, start_dt, end_dt);

INSERT INTO bl_3nf.ce_employees_scd (
    employee_id,
    employee_src_id,
    employee_name,
    employee_position,
    employee_salary,
    employee_hire_date,
    start_dt,
    end_dt,
    is_active,
    source_system,
    source_table,
    insert_dt,
    update_dt
)
SELECT
    -1,
    'n.a.',
    'n.a.',
    'n.a.',
    -1,
    DATE '1900-01-01',
    TIMESTAMP '1900-01-01 00:00:00',
    TIMESTAMP '9999-12-31 23:59:59',
    FALSE,
    'MANUAL',
    'MANUAL',
    NOW(),
    NOW()
WHERE NOT EXISTS (
    SELECT 1
    FROM bl_3nf.ce_employees_scd
    WHERE employee_id = -1
      AND start_dt = TIMESTAMP '1900-01-01 00:00:00'
);


CREATE OR REPLACE PROCEDURE bl_cl.load_ce_employees_scd()
LANGUAGE plpgsql
    AS $$
    DECLARE
        v_proc        TEXT := 'bl_cl.load_ce_employees_scd';
        v_closed      INT  := 0;
        v_ins         INT  := 0;
        v_err_msg     TEXT;
        v_err_detail  TEXT;
        v_err_hint    TEXT;
    BEGIN
        DROP TABLE IF EXISTS tmp_employee_src;
        DROP TABLE IF EXISTS tmp_employee_changes;

        CREATE TEMP TABLE tmp_employee_src
        ON COMMIT DROP
        AS
        WITH ranked_src AS (
            SELECT
                s.employee_src_id,
                s.employee_name,
                s.employee_position,
                s.employee_salary,
                s.employee_hire_date,
                COALESCE(s.observed_ts, TIMESTAMP '1900-01-01 00:00:00') AS observed_ts,
                s.source_system,
                s.source_table,
                ROW_NUMBER() OVER (
                    PARTITION BY s.employee_src_id
                    ORDER BY
                        COALESCE(s.observed_ts, TIMESTAMP '1900-01-01 00:00:00') DESC,
                        s.employee_salary DESC NULLS LAST,
                        s.source_table DESC
                ) AS rn
            FROM bl_cl.t_map_employees s
            WHERE s.employee_src_id <> 'n.a.'
        )
        SELECT
            employee_src_id,
            employee_name,
            employee_position,
            employee_salary,
            employee_hire_date,
            observed_ts,
            source_system,
            source_table
        FROM ranked_src
        WHERE rn = 1;

        CREATE INDEX idx_tmp_employee_src_id
            ON tmp_employee_src (employee_src_id);

        CREATE TEMP TABLE tmp_employee_changes
        ON COMMIT DROP
        AS
        SELECT
            src.employee_src_id,
            src.employee_name,
            src.employee_position,
            src.employee_salary,
            src.employee_hire_date,
            src.observed_ts,
            src.source_system,
            src.source_table,
            cur.employee_id,
            cur.start_dt AS current_start_dt,
            cur.end_dt   AS current_end_dt,
            cur.is_active,
            CASE
                WHEN cur.employee_id IS NULL THEN 'NEW'
                WHEN cur.employee_name      IS DISTINCT FROM src.employee_name
                  OR cur.employee_position  IS DISTINCT FROM src.employee_position
                  OR cur.employee_salary    IS DISTINCT FROM src.employee_salary
                  OR cur.employee_hire_date IS DISTINCT FROM src.employee_hire_date
                THEN 'CHANGED'
                ELSE 'NO_CHANGE'
            END AS change_type
        FROM tmp_employee_src src
        LEFT JOIN bl_3nf.ce_employees_scd cur
            ON cur.employee_src_id = src.employee_src_id
          AND cur.is_active = TRUE;

        UPDATE bl_3nf.ce_employees_scd tgt
        SET
            end_dt    = ch.observed_ts,
            is_active = FALSE,
            update_dt = CURRENT_TIMESTAMP
        FROM tmp_employee_changes ch
        WHERE tgt.employee_src_id = ch.employee_src_id
          AND tgt.is_active = TRUE
          AND ch.change_type = 'CHANGED'
          AND ch.observed_ts > tgt.start_dt;

        GET DIAGNOSTICS v_closed = ROW_COUNT;

        INSERT INTO bl_3nf.ce_employees_scd (
            employee_id,
            employee_src_id,
            employee_name,
            employee_position,
            employee_salary,
            employee_hire_date,
            start_dt,
            end_dt,
            is_active,
            source_system,
            source_table,
            insert_dt,
            update_dt
        )
        SELECT
            COALESCE(ch.employee_id, nextval('bl_3nf.seq_ce_employee_id')),
            ch.employee_src_id,
            ch.employee_name,
            ch.employee_position,
            ch.employee_salary,
            ch.employee_hire_date,
            ch.observed_ts,
            TIMESTAMP '9999-12-31 23:59:59',
            TRUE,
            ch.source_system,
            ch.source_table,
            CURRENT_TIMESTAMP,
            CURRENT_TIMESTAMP
        FROM tmp_employee_changes ch
        WHERE ch.change_type = 'NEW'
          OR (
                ch.change_type = 'CHANGED'
                AND ch.observed_ts > COALESCE(ch.current_start_dt, TIMESTAMP '1900-01-01 00:00:00')
          );

        GET DIAGNOSTICS v_ins = ROW_COUNT;

        PERFORM bl_cl.log_etl_event(
            v_proc,
            'bl_3nf.ce_employees_scd',
            v_closed + v_ins,
            CASE WHEN (v_closed + v_ins) > 0 THEN 'SUCCESS' ELSE 'NO_CHANGE' END,
            'Employee SCD2 load completed. Closed=' || v_closed || ', Inserted=' || v_ins,
            NULL,
            'INFO',
            v_ins,
            0,
            0,
            v_closed,
            NULL
        );

        RAISE NOTICE 'ce_employees_scd completed. closed=%, inserted=%', v_closed, v_ins;

    EXCEPTION
    WHEN OTHERS THEN
        GET STACKED DIAGNOSTICS
            v_err_detail = PG_EXCEPTION_DETAIL,
            v_err_hint   = PG_EXCEPTION_HINT;

        v_err_msg := SQLERRM;

        PERFORM bl_cl.log_etl_event(
            v_proc,
            'bl_3nf.ce_employees_scd',
            0,
            'FAILED',
            'Employee SCD2 upsert failed',
            v_err_msg || ' | DETAIL=' || COALESCE(v_err_detail, 'n.a.')
                      || ' | HINT='   || COALESCE(v_err_hint, 'n.a.'),
            'ERROR'
        );

        RAISE;
    END;
    $$;

SQL

CREATE SEQUENCE
CREATE TABLE
CREATE INDEX
CREATE INDEX
CREATE INDEX
INSERT 0 1
CREATE PROCEDURE


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'
CALL bl_cl.load_ce_employees_scd();

SQL

CALL


NOTICE:  table "tmp_employee_src" does not exist, skipping
NOTICE:  table "tmp_employee_changes" does not exist, skipping
NOTICE:  ce_employees_scd completed. closed=0, inserted=100


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CREATE SEQUENCE IF NOT EXISTS bl_dm.seq_dim_employee_id START 1;

CREATE TABLE IF NOT EXISTS bl_dm.dim_employees_scd (
    employee_surr_id   BIGINT PRIMARY KEY,
    employee_src_id    BIGINT NOT NULL,

    employee_name      VARCHAR(100),
    employee_position  VARCHAR(100),
    employee_salary    NUMERIC(10,2),
    employee_hire_date DATE,

    start_dt           TIMESTAMP NOT NULL,
    end_dt             TIMESTAMP NOT NULL,
    is_active          BOOLEAN NOT NULL,

    source_system      VARCHAR(100) NOT NULL,
    source_table       VARCHAR(100) NOT NULL,

    insert_dt          TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    update_dt          TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE UNIQUE INDEX IF NOT EXISTS uq_dm_emp_active
ON bl_dm.dim_employees_scd (employee_src_id)
WHERE is_active = TRUE;

CREATE UNIQUE INDEX IF NOT EXISTS uq_dm_emp_version
ON bl_dm.dim_employees_scd (employee_src_id, start_dt);

CREATE INDEX IF NOT EXISTS ix_dm_emp_scd_lookup
ON bl_dm.dim_employees_scd (employee_src_id, start_dt, end_dt);

INSERT INTO bl_dm.dim_employees_scd (
    employee_surr_id,
    employee_src_id,
    employee_name,
    employee_position,
    employee_salary,
    employee_hire_date,
    start_dt,
    end_dt,
    is_active,
    source_system,
    source_table,
    insert_dt,
    update_dt
)
SELECT
    -1,
    -1,
    'n.a.',
    'n.a.',
    -1,
    DATE '1900-01-01',
    TIMESTAMP '1900-01-01 00:00:00',
    TIMESTAMP '9999-12-31 23:59:59',
    FALSE,
    'MANUAL',
    'MANUAL',
    NOW(),
    NOW()
WHERE NOT EXISTS (
    SELECT 1
    FROM bl_dm.dim_employees_scd
    WHERE employee_surr_id = -1
);

CREATE OR REPLACE PROCEDURE bl_cl.load_dim_employees_scd()
LANGUAGE plpgsql
AS $$
DECLARE
    v_proc        TEXT := 'bl_cl.load_dim_employees_scd';
    v_ins         INT  := 0;
    v_upd         INT  := 0;
    v_err_msg     TEXT;
    v_err_detail  TEXT;
    v_err_hint    TEXT;
BEGIN
    INSERT INTO bl_dm.dim_employees_scd (
        employee_surr_id,
        employee_src_id,
        employee_name,
        employee_position,
        employee_salary,
        employee_hire_date,
        start_dt,
        end_dt,
        is_active,
        source_system,
        source_table,
        insert_dt,
        update_dt
    )
    SELECT
        nextval('bl_dm.seq_dim_employee_id'),
        e.employee_id,
        e.employee_name,
        e.employee_position,
        e.employee_salary,
        e.employee_hire_date,
        e.start_dt,
        e.end_dt,
        e.is_active,
        'bl_3nf',
        'ce_employees_scd',
        NOW(),
        NOW()
    FROM bl_3nf.ce_employees_scd e
    LEFT JOIN bl_dm.dim_employees_scd d
        ON d.employee_src_id = e.employee_id
       AND d.start_dt = e.start_dt
    WHERE e.employee_id <> -1
      AND d.employee_surr_id IS NULL;

    GET DIAGNOSTICS v_ins = ROW_COUNT;

    UPDATE bl_dm.dim_employees_scd d
    SET
        employee_name      = e.employee_name,
        employee_position  = e.employee_position,
        employee_salary    = e.employee_salary,
        employee_hire_date = e.employee_hire_date,
        end_dt             = e.end_dt,
        is_active          = e.is_active,
        update_dt          = NOW()
    FROM bl_3nf.ce_employees_scd e
    WHERE d.employee_src_id = e.employee_id
      AND d.start_dt = e.start_dt
      AND e.employee_id <> -1
      AND (
            d.employee_name      IS DISTINCT FROM e.employee_name
         OR d.employee_position  IS DISTINCT FROM e.employee_position
         OR d.employee_salary    IS DISTINCT FROM e.employee_salary
         OR d.employee_hire_date IS DISTINCT FROM e.employee_hire_date
         OR d.end_dt             IS DISTINCT FROM e.end_dt
         OR d.is_active          IS DISTINCT FROM e.is_active
      );

    GET DIAGNOSTICS v_upd = ROW_COUNT;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_dm.dim_employees_scd',
        v_ins + v_upd,
        CASE WHEN v_ins + v_upd > 0 THEN 'SUCCESS' ELSE 'NO_CHANGE' END,
        'Inserted=' || v_ins || ', Updated=' || v_upd || '. Mirrored employee SCD history from 3NF to DM.'
    );

    RAISE NOTICE 'dim_employees_scd completed. inserted=%, updated=%', v_ins, v_upd;

EXCEPTION
WHEN OTHERS THEN
    GET STACKED DIAGNOSTICS
        v_err_detail = PG_EXCEPTION_DETAIL,
        v_err_hint   = PG_EXCEPTION_HINT;

    v_err_msg := SQLERRM;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_dm.dim_employees_scd',
        0,
        'FAILED',
        'dim_employees_scd load failed',
        v_err_msg || ' | DETAIL=' || COALESCE(v_err_detail, 'n.a.')
                  || ' | HINT='   || COALESCE(v_err_hint, 'n.a.'),
        'ERROR'
    );

    RAISE;
END;
$$;

SQL

CREATE SEQUENCE
CREATE TABLE
CREATE INDEX
CREATE INDEX
CREATE INDEX
INSERT 0 1
CREATE PROCEDURE


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CALL bl_cl.load_dim_employees_scd();

SQL

CALL


NOTICE:  dim_employees_scd completed. inserted=100, updated=0


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

DROP TABLE IF EXISTS bl_cl.t_map_transactions;

CREATE TABLE bl_cl.t_map_transactions (
    transaction_id          VARCHAR(100),
    transaction_dt          TIMESTAMP,

    total_sales             NUMERIC(10,2),
    payment_method          VARCHAR(100),
    quantity                INTEGER,
    unit_price              NUMERIC(10,2),
    discount_applied        NUMERIC(10,2),

    day_of_week             VARCHAR(100),
    week_of_year            INTEGER,
    month_of_year           INTEGER,

    /* raw business keys */
    customer_id             VARCHAR(100),
    product_id              VARCHAR(100),
    promotion_id            VARCHAR(100),
    delivery_id             VARCHAR(100),
    engagement_id           VARCHAR(100),

    employee_name           VARCHAR(100),
    employee_hire_date      DATE,

    customer_city           VARCHAR(100),
    customer_state          VARCHAR(100),

    store_zip_code          VARCHAR(30),
    store_city              VARCHAR(100),
    store_state             VARCHAR(100),
    store_location          VARCHAR(100),

    /* derived source keys for downstream 3NF joins */
    customer_src_id         VARCHAR(255),
    product_src_id          VARCHAR(255),
    promotion_src_id        VARCHAR(255),
    store_src_id            VARCHAR(255),
    city_src_id             VARCHAR(255),
    employee_src_id         VARCHAR(255),

    row_sig                 TEXT,
    source_system           VARCHAR(100),
    source_table            VARCHAR(100),
    insert_dt               TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE INDEX IF NOT EXISTS ix_t_map_transactions_src
    ON bl_cl.t_map_transactions (transaction_id, source_system, source_table);

CREATE INDEX IF NOT EXISTS ix_t_map_transactions_dt
    ON bl_cl.t_map_transactions (transaction_dt);

CREATE UNIQUE INDEX IF NOT EXISTS ux_t_map_transactions_rowsig
    ON bl_cl.t_map_transactions (row_sig);

CREATE INDEX IF NOT EXISTS ix_t_map_transactions_customer_src
    ON bl_cl.t_map_transactions (customer_src_id);

CREATE INDEX IF NOT EXISTS ix_t_map_transactions_product_src
    ON bl_cl.t_map_transactions (product_src_id);

CREATE INDEX IF NOT EXISTS ix_t_map_transactions_promotion_src
    ON bl_cl.t_map_transactions (promotion_src_id);

CREATE INDEX IF NOT EXISTS ix_t_map_transactions_store_src
    ON bl_cl.t_map_transactions (store_src_id);

CREATE INDEX IF NOT EXISTS ix_t_map_transactions_city_src
    ON bl_cl.t_map_transactions (city_src_id);

CREATE INDEX IF NOT EXISTS ix_t_map_transactions_employee_src
    ON bl_cl.t_map_transactions (employee_src_id);

CREATE INDEX IF NOT EXISTS ix_t_map_transactions_delivery_id
    ON bl_cl.t_map_transactions (delivery_id);

CREATE INDEX IF NOT EXISTS ix_t_map_transactions_engagement_id
    ON bl_cl.t_map_transactions (engagement_id);


CREATE OR REPLACE PROCEDURE bl_cl.load_map_transactions()
LANGUAGE plpgsql
AS $$
DECLARE
    v_proc        TEXT := 'bl_cl.load_map_transactions';
    v_ins         INT  := 0;
    v_err_msg     TEXT;
    v_err_detail  TEXT;
    v_err_hint    TEXT;
BEGIN

    WITH unioned_sources AS (

        /* =========================================================
           ONLINE
           ========================================================= */
        SELECT
            COALESCE(NULLIF(src.transaction_id,''),'n.a.')                AS transaction_id,
            src.transaction_dt,
            src.total_sales,
            COALESCE(src.payment_method,'n.a.')                           AS payment_method,
            src.quantity,
            src.unit_price,
            src.discount_applied,

            COALESCE(src.day_of_week,'n.a.')                              AS day_of_week,
            COALESCE(src.week_of_year, -1)                                AS week_of_year,
            COALESCE(src.month_of_year, -1)                               AS month_of_year,

            /* raw ids */
            COALESCE(NULLIF(src.customer_id,''),'n.a.')                   AS customer_id,
            COALESCE(NULLIF(src.product_id,''),'n.a.')                    AS product_id,
            COALESCE(NULLIF(src.promotion_id,''),'n.a.')                  AS promotion_id,
            COALESCE(NULLIF(src.delivery_id,''),'n.a.')                   AS delivery_id,
            COALESCE(NULLIF(src.engagement_id,''),'n.a.')                 AS engagement_id,

            /* employee not present in online */
            'n.a.'                                                        AS employee_name,
            DATE '1900-01-01'                                             AS employee_hire_date,

            COALESCE(src.customer_city,'n.a.')                            AS customer_city,
            COALESCE(src.customer_state,'n.a.')                           AS customer_state,

            'n.a.'                                                        AS store_zip_code,
            'n.a.'                                                        AS store_city,
            'n.a.'                                                        AS store_state,
            'n.a.'                                                        AS store_location,

            /* derived customer src id */
            COALESCE(src.gender,'n.a.') || '-' ||
            COALESCE(src.marital_status,'n.a.') || '-' ||
            COALESCE(src.birth_of_dt::TEXT,'n.a.') || '-' ||
            COALESCE(src.membership_dt::TEXT,'n.a.') || '-' ||
            COALESCE(src.customer_zip_code,'n.a.') || '-' ||
            COALESCE(src.customer_city,'n.a.') || '-' ||
            COALESCE(src.customer_state,'n.a.')                           AS customer_src_id,


            COALESCE(NULLIF(src.product_id, ''), 'n.a.')                  AS product_src_id,

            COALESCE(NULLIF(src.promotion_id, ''), 'n.a.')                AS promotion_src_id,

            /* online has no store */
            'n.a.'                                                        AS store_src_id,

            /* city: customer geography */
            COALESCE(src.customer_city,'n.a.') || '-' ||
            COALESCE(src.customer_state,'n.a.')                           AS city_src_id,

            /* online has no employee */
            'n.a.'                                                        AS employee_src_id,

            md5(
                concat_ws(
                    '|',
                    'sa_online_retail',
                    COALESCE(NULLIF(src.transaction_id,''),'n.a.'),
                    COALESCE(src.transaction_dt::TEXT,'1900-01-01 00:00:00'),
                    COALESCE(NULLIF(src.customer_id,''),'n.a.'),
                    COALESCE(NULLIF(src.product_id,''),'n.a.')
                )
            )                                                             AS row_sig,

            'sa_online_retail'                                            AS source_system,
            'src_online_retail'                                           AS source_table

        FROM sa_online_retail.src_online_retail src
        WHERE COALESCE(NULLIF(src.transaction_id,''),'n.a.') <> 'n.a.'
          AND src.transaction_dt IS NOT NULL

        UNION ALL

        /* =========================================================
           OFFLINE
           ========================================================= */
        SELECT
            COALESCE(NULLIF(src.transaction_id,''),'n.a.')                AS transaction_id,
            src.transaction_dt,
            src.total_sales,
            COALESCE(src.payment_method,'n.a.')                           AS payment_method,
            src.quantity,
            src.unit_price,
            src.discount_applied,

            COALESCE(src.day_of_week,'n.a.')                              AS day_of_week,
            COALESCE(src.week_of_year, -1)                                AS week_of_year,
            COALESCE(src.month_of_year, -1)                               AS month_of_year,

            /* raw ids */
            COALESCE(NULLIF(src.customer_id,''),'n.a.')                   AS customer_id,
            COALESCE(NULLIF(src.product_id,''),'n.a.')                    AS product_id,
            COALESCE(NULLIF(src.promotion_id,''),'n.a.')                  AS promotion_id,
            COALESCE(NULLIF(src.delivery_id,''),'n.a.')                   AS delivery_id,
            'n.a.'                                                        AS engagement_id,

            COALESCE(src.employee_name,'n.a.')                            AS employee_name,
            COALESCE(src.employee_hire_date, DATE '1900-01-01')           AS employee_hire_date,

            COALESCE(src.customer_city,'n.a.')                            AS customer_city,
            COALESCE(src.customer_state,'n.a.')                           AS customer_state,

            COALESCE(src.store_zip_code,'n.a.')                           AS store_zip_code,
            COALESCE(src.store_city,'n.a.')                               AS store_city,
            COALESCE(src.store_state,'n.a.')                              AS store_state,
            COALESCE(src.store_location,'n.a.')                           AS store_location,

            /* derived customer src id */
            COALESCE(src.gender,'n.a.') || '-' ||
            COALESCE(src.marital_status,'n.a.') || '-' ||
            COALESCE(src.birth_of_dt::TEXT,'n.a.') || '-' ||
            COALESCE(src.membership_dt::TEXT,'n.a.') || '-' ||
            COALESCE(src.customer_zip_code,'n.a.') || '-' ||
            COALESCE(src.customer_city,'n.a.') || '-' ||
            COALESCE(src.customer_state,'n.a.')                           AS customer_src_id,


            COALESCE(NULLIF(src.product_id, ''), 'n.a.')                  AS product_src_id,

            COALESCE(NULLIF(src.promotion_id, ''), 'n.a.')                AS promotion_src_id,

            /* derived store src id */
            CASE
                WHEN COALESCE(src.store_location,'n.a.') <> 'n.a.'
                THEN COALESCE(src.store_location,'n.a.') || '-' ||
                     COALESCE(src.store_city,'n.a.') || '-' ||
                     COALESCE(src.store_state,'n.a.')
                ELSE 'n.a.'
            END                                                           AS store_src_id,

            /* city prefers store geography if store exists */
            CASE
                WHEN COALESCE(src.store_city,'n.a.') <> 'n.a.'
                 AND COALESCE(src.store_state,'n.a.') <> 'n.a.'
                THEN COALESCE(src.store_city,'n.a.') || '-' ||
                     COALESCE(src.store_state,'n.a.')
                ELSE COALESCE(src.customer_city,'n.a.') || '-' ||
                     COALESCE(src.customer_state,'n.a.')
            END                                                           AS city_src_id,

            /* derived employee src id */
            CASE
                WHEN COALESCE(src.employee_name,'n.a.') <> 'n.a.'
                THEN COALESCE(src.employee_name,'n.a.') || '-' ||
                     COALESCE(src.employee_hire_date::TEXT, '1900-01-01')
                ELSE 'n.a.'
            END                                                           AS employee_src_id,

            md5(
                concat_ws(
                    '|',
                    'sa_offline_retail',
                    COALESCE(NULLIF(src.transaction_id,''),'n.a.'),
                    COALESCE(src.transaction_dt::TEXT,'1900-01-01 00:00:00'),
                    COALESCE(NULLIF(src.customer_id,''),'n.a.'),
                    COALESCE(NULLIF(src.product_id,''),'n.a.')
                )
            )                                                             AS row_sig,

            'sa_offline_retail'                                           AS source_system,
            'src_offline_retail'                                          AS source_table

        FROM sa_offline_retail.src_offline_retail src
        WHERE COALESCE(NULLIF(src.transaction_id,''),'n.a.') <> 'n.a.'
          AND src.transaction_dt IS NOT NULL
    ),

    distinct_source AS (
        SELECT DISTINCT ON (row_sig)
            transaction_id,
            transaction_dt,
            total_sales,
            payment_method,
            quantity,
            unit_price,
            discount_applied,
            day_of_week,
            week_of_year,
            month_of_year,
            customer_id,
            product_id,
            promotion_id,
            delivery_id,
            engagement_id,
            employee_name,
            employee_hire_date,
            customer_city,
            customer_state,
            store_zip_code,
            store_city,
            store_state,
            store_location,
            customer_src_id,
            product_src_id,
            promotion_src_id,
            store_src_id,
            city_src_id,
            employee_src_id,
            row_sig,
            source_system,
            source_table
        FROM unioned_sources
        ORDER BY
            row_sig,
            transaction_dt DESC,
            customer_src_id DESC,
            product_src_id DESC,
            promotion_src_id DESC,
            source_system
    )

    INSERT INTO bl_cl.t_map_transactions (
        transaction_id,
        transaction_dt,
        total_sales,
        payment_method,
        quantity,
        unit_price,
        discount_applied,
        day_of_week,
        week_of_year,
        month_of_year,
        customer_id,
        product_id,
        promotion_id,
        delivery_id,
        engagement_id,
        employee_name,
        employee_hire_date,
        customer_city,
        customer_state,
        store_zip_code,
        store_city,
        store_state,
        store_location,
        customer_src_id,
        product_src_id,
        promotion_src_id,
        store_src_id,
        city_src_id,
        employee_src_id,
        row_sig,
        source_system,
        source_table
    )
    SELECT
        s.transaction_id,
        s.transaction_dt,
        s.total_sales,
        s.payment_method,
        s.quantity,
        s.unit_price,
        s.discount_applied,
        s.day_of_week,
        s.week_of_year,
        s.month_of_year,
        s.customer_id,
        s.product_id,
        s.promotion_id,
        s.delivery_id,
        s.engagement_id,
        s.employee_name,
        s.employee_hire_date,
        s.customer_city,
        s.customer_state,
        s.store_zip_code,
        s.store_city,
        s.store_state,
        s.store_location,
        s.customer_src_id,
        s.product_src_id,
        s.promotion_src_id,
        s.store_src_id,
        s.city_src_id,
        s.employee_src_id,
        s.row_sig,
        s.source_system,
        s.source_table
    FROM distinct_source s
    WHERE NOT EXISTS (
        SELECT 1
        FROM bl_cl.t_map_transactions tgt
        WHERE tgt.row_sig = s.row_sig
    );

    GET DIAGNOSTICS v_ins = ROW_COUNT;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_cl.t_map_transactions',
        v_ins,
        CASE WHEN v_ins > 0 THEN 'SUCCESS' ELSE 'NO_CHANGE' END,
        'Inserted=' || v_ins || '. Loaded transaction-line rows directly from clean staging and derived src keys inside the same procedure.',
        NULL,
        'INFO',
        v_ins,
        0,
        0,
        0,
        NULL
    );

    RAISE NOTICE 't_map_transactions completed. inserted=%', v_ins;

EXCEPTION
WHEN OTHERS THEN
    GET STACKED DIAGNOSTICS
        v_err_detail = PG_EXCEPTION_DETAIL,
        v_err_hint   = PG_EXCEPTION_HINT;

    v_err_msg := SQLERRM;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_cl.t_map_transactions',
        0,
        'FAILED',
        'Transaction map load failed',
        v_err_msg || ' | DETAIL=' || COALESCE(v_err_detail, 'n.a.')
                  || ' | HINT='   || COALESCE(v_err_hint, 'n.a.'),
        'ERROR',
        0,
        0,
        0,
        0,
        NULL
    );

    RAISE;
END;
$$;

SQL

DROP TABLE
CREATE TABLE
CREATE INDEX
CREATE INDEX
CREATE INDEX
CREATE INDEX
CREATE INDEX
CREATE INDEX
CREATE INDEX
CREATE INDEX
CREATE INDEX
CREATE INDEX
CREATE INDEX
CREATE PROCEDURE


NOTICE:  table "t_map_transactions" does not exist, skipping


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'
CALL bl_cl.load_map_transactions();

SQL

CALL


NOTICE:  t_map_transactions completed. inserted=950000


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CREATE TABLE IF NOT EXISTS bl_3nf.ce_transactions (
    transaction_id      VARCHAR(100)  NOT NULL,
    transaction_dt      TIMESTAMP     NOT NULL,
    total_sales         NUMERIC(10,2) NOT NULL DEFAULT 0,
    payment_method      VARCHAR(100)  NOT NULL DEFAULT 'n.a.',
    quantity            INTEGER       NOT NULL DEFAULT 0,
    unit_price          NUMERIC(10,2) NOT NULL DEFAULT 0,
    discount_applied    NUMERIC(10,2) NOT NULL DEFAULT 0,
    day_of_week         VARCHAR(20)   NOT NULL DEFAULT 'n.a.',
    week_of_year        INTEGER       NOT NULL DEFAULT -1,
    month_of_year       INTEGER       NOT NULL DEFAULT -1,

    store_id            BIGINT        NOT NULL DEFAULT -1,
    customer_id         BIGINT        NOT NULL DEFAULT -1,
    promotion_id        BIGINT        NOT NULL DEFAULT -1,
    delivery_id         BIGINT        NOT NULL DEFAULT -1,
    product_id          BIGINT        NOT NULL DEFAULT -1,
    engagement_id       BIGINT        NOT NULL DEFAULT -1,
    city_id             BIGINT        NOT NULL DEFAULT -1,

    /* employee history resolved by employee_src_id + transaction_dt */
    employee_id         BIGINT        NOT NULL DEFAULT -1,

    row_sig             TEXT          NOT NULL,
    source_system       VARCHAR(100)  NOT NULL,
    source_table        VARCHAR(100)  NOT NULL,
    insert_dt           TIMESTAMP     NOT NULL DEFAULT CURRENT_TIMESTAMP,

    CONSTRAINT fk_ce_transactions_store
        FOREIGN KEY (store_id) REFERENCES bl_3nf.ce_stores(store_id),

    CONSTRAINT fk_ce_transactions_customer
        FOREIGN KEY (customer_id) REFERENCES bl_3nf.ce_customers(customer_id),

    CONSTRAINT fk_ce_transactions_promotion
        FOREIGN KEY (promotion_id) REFERENCES bl_3nf.ce_promotions(promotion_id),

    CONSTRAINT fk_ce_transactions_delivery
        FOREIGN KEY (delivery_id) REFERENCES bl_3nf.ce_deliveries(delivery_id),

    CONSTRAINT fk_ce_transactions_product
        FOREIGN KEY (product_id) REFERENCES bl_3nf.ce_products(product_id),

    CONSTRAINT fk_ce_transactions_engagement
        FOREIGN KEY (engagement_id) REFERENCES bl_3nf.ce_engagements(engagement_id),

    CONSTRAINT fk_ce_transactions_city
        FOREIGN KEY (city_id) REFERENCES bl_3nf.ce_cities(city_id)
);

CREATE INDEX IF NOT EXISTS ix_ce_transactions_trx_id
    ON bl_3nf.ce_transactions (transaction_id);

CREATE INDEX IF NOT EXISTS ix_ce_transactions_trx_dt_brin
    ON bl_3nf.ce_transactions USING brin (transaction_dt);

CREATE UNIQUE INDEX IF NOT EXISTS ux_ce_transactions_row_sig
    ON bl_3nf.ce_transactions (row_sig);

CREATE INDEX IF NOT EXISTS ix_ce_transactions_source
    ON bl_3nf.ce_transactions (source_system, source_table);

CREATE INDEX IF NOT EXISTS ix_ce_transactions_customer
    ON bl_3nf.ce_transactions (customer_id);

CREATE INDEX IF NOT EXISTS ix_ce_transactions_product
    ON bl_3nf.ce_transactions (product_id);

CREATE INDEX IF NOT EXISTS ix_ce_transactions_employee
    ON bl_3nf.ce_transactions (employee_id);

/* lookup indexes for join speed */
CREATE INDEX IF NOT EXISTS ix_ce_customers_src_id
    ON bl_3nf.ce_customers (customer_src_id);

CREATE INDEX IF NOT EXISTS ix_ce_products_src_id
    ON bl_3nf.ce_products (product_src_id);

CREATE INDEX IF NOT EXISTS ix_ce_promotions_src_id
    ON bl_3nf.ce_promotions (promotion_src_id);


CREATE INDEX IF NOT EXISTS ix_ce_engagements_src_id
    ON bl_3nf.ce_engagements (engagement_src_id);

CREATE INDEX IF NOT EXISTS ix_ce_cities_src_id
    ON bl_3nf.ce_cities (city_src_id);

CREATE INDEX IF NOT EXISTS ix_ce_stores_src_id
    ON bl_3nf.ce_stores (store_src_id);

CREATE INDEX IF NOT EXISTS ix_ce_emp_lookup
    ON bl_3nf.ce_employees_scd (employee_src_id, is_active, start_dt, end_dt);


/* default row */
INSERT INTO bl_3nf.ce_transactions (
    transaction_id,
    transaction_dt,
    total_sales,
    payment_method,
    quantity,
    unit_price,
    discount_applied,
    day_of_week,
    week_of_year,
    month_of_year,
    store_id,
    customer_id,
    promotion_id,
    delivery_id,
    product_id,
    engagement_id,
    city_id,
    employee_id,
    row_sig,
    source_system,
    source_table
)
SELECT
    'n.a.',
    TIMESTAMP '1900-01-01 00:00:00',
    0,
    'n.a.',
    0,
    0,
    0,
    'n.a.',
    -1,
    -1,
    -1,
    -1,
    -1,
    -1,
    -1,
    -1,
    -1,
    -1,
    'n.a.',
    'MANUAL',
    'MANUAL'
WHERE NOT EXISTS (
    SELECT 1
    FROM bl_3nf.ce_transactions
    WHERE row_sig = 'n.a.'
);


CREATE OR REPLACE PROCEDURE bl_cl.load_ce_transactions()
LANGUAGE plpgsql
AS $$
DECLARE
    v_proc        TEXT := 'bl_cl.load_ce_transactions';
    v_ins_total   INT  := 0;
    v_err_msg     TEXT;
    v_err_detail  TEXT;
    v_err_hint    TEXT;
BEGIN

    INSERT INTO bl_3nf.ce_transactions (
        transaction_id,
        transaction_dt,
        total_sales,
        payment_method,
        quantity,
        unit_price,
        discount_applied,
        day_of_week,
        week_of_year,
        month_of_year,
        store_id,
        customer_id,
        promotion_id,
        delivery_id,
        product_id,
        engagement_id,
        city_id,
        employee_id,
        row_sig,
        source_system,
        source_table,
        insert_dt
    )
    SELECT
        t.transaction_id,
        t.transaction_dt,
        COALESCE(t.total_sales, 0),
        COALESCE(t.payment_method, 'n.a.'),
        COALESCE(t.quantity, 0),
        COALESCE(t.unit_price, 0),
        COALESCE(t.discount_applied, 0),
        COALESCE(t.day_of_week, 'n.a.'),
        COALESCE(t.week_of_year, -1),
        COALESCE(t.month_of_year, -1),

        COALESCE(st.store_id, -1)         AS store_id,
        COALESCE(cu.customer_id, -1)      AS customer_id,
        COALESCE(pm.promotion_id, -1)     AS promotion_id,
        COALESCE(dv.delivery_id, -1)      AS delivery_id,
        COALESCE(pr.product_id, -1)       AS product_id,
        COALESCE(eg.engagement_id, -1)    AS engagement_id,
        COALESCE(ci.city_id, -1)          AS city_id,
        COALESCE(emp.employee_id, -1)     AS employee_id,

        t.row_sig,
        t.source_system,
        t.source_table,
        NOW()

    FROM bl_cl.t_map_transactions t

    /* customer: src_id to src_id */
    LEFT JOIN bl_3nf.ce_customers cu
        ON cu.customer_src_id = t.customer_src_id

    /* product: src_id to src_id */
    LEFT JOIN bl_3nf.ce_products pr
        ON pr.product_src_id = t.product_src_id

    /* promotion: src_id to src_id */
    LEFT JOIN bl_3nf.ce_promotions pm
        ON pm.promotion_src_id = t.promotion_src_id

    /* delivery: raw id to natural key */
    LEFT JOIN bl_3nf.ce_deliveries dv
        ON dv.delivery_src_id = t.delivery_id

    /* engagement: raw id to raw/source key */
    LEFT JOIN bl_3nf.ce_engagements eg
        ON eg.engagement_src_id = t.engagement_id

    /* city: pre-derived src key */
    LEFT JOIN bl_3nf.ce_cities ci
        ON ci.city_src_id = t.city_src_id

    /* store: pre-derived src key */
    LEFT JOIN bl_3nf.ce_stores st
        ON st.store_src_id = t.store_src_id

    /* employee: pre-derived src key + temporal SCD2 match */
    LEFT JOIN bl_3nf.ce_employees_scd emp
    ON emp.employee_src_id = t.employee_src_id
   AND emp.is_active = TRUE

    WHERE t.transaction_id <> 'n.a.'
      AND t.transaction_dt IS NOT NULL
      AND NOT EXISTS (
          SELECT 1
          FROM bl_3nf.ce_transactions x
          WHERE x.row_sig = t.row_sig
      );

    GET DIAGNOSTICS v_ins_total = ROW_COUNT;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_3nf.ce_transactions',
        v_ins_total,
        CASE WHEN v_ins_total > 0 THEN 'SUCCESS' ELSE 'NO_CHANGE' END,
        'Inserted=' || v_ins_total || '. Final transaction load from t_map_transactions derived src keys into 3NF surrogate keys.',
        NULL,
        'INFO',
        v_ins_total,
        0,
        0,
        0,
        NULL
    );

    RAISE NOTICE 'ce_transactions completed. inserted=%', v_ins_total;

EXCEPTION
WHEN OTHERS THEN
    GET STACKED DIAGNOSTICS
        v_err_detail = PG_EXCEPTION_DETAIL,
        v_err_hint   = PG_EXCEPTION_HINT;

    v_err_msg := SQLERRM;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_3nf.ce_transactions',
        0,
        'FAILED',
        'Transaction load failed',
        v_err_msg || ' | DETAIL=' || COALESCE(v_err_detail, 'n.a.')
                  || ' | HINT='   || COALESCE(v_err_hint, 'n.a.'),
        'ERROR',
        0,
        0,
        0,
        0,
        NULL
    );

    RAISE;
END;
$$;

SQL

CREATE TABLE
CREATE INDEX
CREATE INDEX
CREATE INDEX
CREATE INDEX
CREATE INDEX
CREATE INDEX
CREATE INDEX
CREATE INDEX
CREATE INDEX
CREATE INDEX
CREATE INDEX
CREATE INDEX
CREATE INDEX
CREATE INDEX
INSERT 0 1
CREATE PROCEDURE


NOTICE:  relation "ix_ce_customers_src_id" already exists, skipping
NOTICE:  relation "ix_ce_products_src_id" already exists, skipping
NOTICE:  relation "ix_ce_promotions_src_id" already exists, skipping
NOTICE:  relation "ix_ce_emp_lookup" already exists, skipping


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -P pager=off <<'SQL'

\d+ bl_cl.t_map_transactions
\d+ bl_3nf.ce_transactions
\d+ bl_3nf.ce_customers
\d+ bl_3nf.ce_products
\d+ bl_3nf.ce_promotions
\d+ bl_3nf.ce_deliveries
\d+ bl_3nf.ce_engagements
\d+ bl_3nf.ce_cities
\d+ bl_3nf.ce_stores
\d+ bl_3nf.ce_employees_scd

SQL

                                                         Table "bl_cl.t_map_transactions"
       Column       |            Type             | Collation | Nullable |      Default      | Storage  | Compression | Stats target | Description 
--------------------+-----------------------------+-----------+----------+-------------------+----------+-------------+--------------+-------------
 transaction_id     | character varying(100)      |           |          |                   | extended |             |              | 
 transaction_dt     | timestamp without time zone |           |          |                   | plain    |             |              | 
 total_sales        | numeric(10,2)               |           |          |                   | main     |             |              | 
 payment_method     | character varying(100)      |           |          |                   | extended |             |              | 
 quantity           | integer                     |           |       

In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'
CALL bl_cl.load_ce_transactions();
SQL

CALL


NOTICE:  ce_transactions completed. inserted=950000


In [ ]:


%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CREATE TABLE IF NOT EXISTS bl_dm.dim_dates (
    date_surr_id        BIGINT PRIMARY KEY,      -- yyyymmdd
    full_date           DATE NOT NULL UNIQUE,

    day_of_month        INTEGER NOT NULL,
    month_of_year       INTEGER NOT NULL,
    year_of_date        INTEGER NOT NULL,
    quarter_of_year     INTEGER NOT NULL,
    week_of_year        INTEGER NOT NULL,

    day_name            VARCHAR(20) NOT NULL,
    month_name          VARCHAR(20) NOT NULL,

    is_weekend          BOOLEAN NOT NULL,
    is_month_start      BOOLEAN NOT NULL,
    is_month_end        BOOLEAN NOT NULL,
    is_quarter_start    BOOLEAN NOT NULL,
    is_quarter_end      BOOLEAN NOT NULL,
    is_year_start       BOOLEAN NOT NULL,
    is_year_end         BOOLEAN NOT NULL,

    insert_dt           TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP
);

CREATE INDEX IF NOT EXISTS ix_dim_dates_full_date
    ON bl_dm.dim_dates (full_date);

CREATE INDEX IF NOT EXISTS ix_dim_dates_year_month
    ON bl_dm.dim_dates (year_of_date, month_of_year);

CREATE INDEX IF NOT EXISTS ix_dim_dates_year_week
    ON bl_dm.dim_dates (year_of_date, week_of_year);

CREATE INDEX IF NOT EXISTS ix_dim_dates_month_name
    ON bl_dm.dim_dates (month_name);

CREATE INDEX IF NOT EXISTS ix_dim_dates_day_name
    ON bl_dm.dim_dates (day_name);


CREATE OR REPLACE PROCEDURE bl_cl.load_dim_dates(
    p_start_date DATE,
    p_end_date   DATE
)
LANGUAGE plpgsql
AS $$
DECLARE
    v_proc        TEXT := 'bl_cl.load_dim_dates';
    v_ins         INT  := 0;
    v_err_msg     TEXT;
    v_err_detail  TEXT;
    v_err_hint    TEXT;
BEGIN
    INSERT INTO bl_dm.dim_dates (
        date_surr_id,
        full_date,
        day_of_month,
        month_of_year,
        year_of_date,
        quarter_of_year,
        week_of_year,
        day_name,
        month_name,
        is_weekend,
        is_month_start,
        is_month_end,
        is_quarter_start,
        is_quarter_end,
        is_year_start,
        is_year_end,
        insert_dt
    )
    SELECT
        TO_CHAR(d::date, 'YYYYMMDD')::BIGINT                    AS date_surr_id,
        d::date                                                 AS full_date,
        EXTRACT(DAY FROM d)::INT                                AS day_of_month,
        EXTRACT(MONTH FROM d)::INT                              AS month_of_year,
        EXTRACT(YEAR FROM d)::INT                               AS year_of_date,
        EXTRACT(QUARTER FROM d)::INT                            AS quarter_of_year,
        EXTRACT(WEEK FROM d)::INT                               AS week_of_year,
        TRIM(TO_CHAR(d::date, 'Day'))                           AS day_name,
        TRIM(TO_CHAR(d::date, 'Month'))                         AS month_name,
        CASE WHEN EXTRACT(ISODOW FROM d) IN (6, 7) THEN TRUE ELSE FALSE END AS is_weekend,
        CASE WHEN d::date = date_trunc('month', d)::date THEN TRUE ELSE FALSE END AS is_month_start,
        CASE WHEN d::date = (date_trunc('month', d) + interval '1 month - 1 day')::date THEN TRUE ELSE FALSE END AS is_month_end,
        CASE WHEN d::date = date_trunc('quarter', d)::date THEN TRUE ELSE FALSE END AS is_quarter_start,
        CASE WHEN d::date = (date_trunc('quarter', d) + interval '3 month - 1 day')::date THEN TRUE ELSE FALSE END AS is_quarter_end,
        CASE WHEN d::date = make_date(EXTRACT(YEAR FROM d)::INT, 1, 1) THEN TRUE ELSE FALSE END AS is_year_start,
        CASE WHEN d::date = make_date(EXTRACT(YEAR FROM d)::INT, 12, 31) THEN TRUE ELSE FALSE END AS is_year_end,
        NOW()
    FROM generate_series(p_start_date, p_end_date, interval '1 day') AS gs(d)
    WHERE NOT EXISTS (
        SELECT 1
        FROM bl_dm.dim_dates dd
        WHERE dd.full_date = gs.d::date
    );

    GET DIAGNOSTICS v_ins = ROW_COUNT;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_dm.dim_dates',
        v_ins,
        CASE WHEN v_ins > 0 THEN 'SUCCESS' ELSE 'NO_CHANGE' END,
        'Inserted=' || v_ins || '. Loaded date rows from ' || p_start_date || ' to ' || p_end_date || '.'
    );

    RAISE NOTICE 'dim_dates completed. inserted=%', v_ins;

EXCEPTION
WHEN OTHERS THEN
    GET STACKED DIAGNOSTICS
        v_err_detail = PG_EXCEPTION_DETAIL,
        v_err_hint   = PG_EXCEPTION_HINT;

    v_err_msg := SQLERRM;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_dm.dim_dates',
        0,
        'FAILED',
        'dim_dates load failed',
        v_err_msg || ' | DETAIL=' || COALESCE(v_err_detail, 'n.a.')
                  || ' | HINT='   || COALESCE(v_err_hint, 'n.a.'),
        'ERROR'
    );

    RAISE;
END;
$$;

SQL


CREATE TABLE
CREATE INDEX
CREATE INDEX
CREATE INDEX
CREATE INDEX
CREATE INDEX
CREATE PROCEDURE


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CALL bl_cl.load_dim_dates(DATE '2019-01-01', DATE '2023-12-31');

SQL

CALL


NOTICE:  dim_dates completed. inserted=1826


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CREATE TABLE IF NOT EXISTS bl_dm.fct_transactions (
    transaction_src_id   VARCHAR(100) NOT NULL,
    total_sales          NUMERIC(10,2) NOT NULL DEFAULT 0,
    quantity             INTEGER      NOT NULL DEFAULT 0,
    unit_price           NUMERIC(10,2) NOT NULL DEFAULT 0,
    discount_applied     NUMERIC(10,2) NOT NULL DEFAULT 0,
    payment_method       VARCHAR(100) NOT NULL DEFAULT 'n.a.',

    product_surr_id      BIGINT NOT NULL DEFAULT -1,
    promotion_surr_id    BIGINT NOT NULL DEFAULT -1,
    delivery_surr_id     BIGINT NOT NULL DEFAULT -1,
    engagement_surr_id   BIGINT NOT NULL DEFAULT -1,
    store_surr_id        BIGINT NOT NULL DEFAULT -1,
    customer_surr_id     BIGINT NOT NULL DEFAULT -1,
    employee_surr_id     BIGINT NOT NULL DEFAULT -1,

    transaction_date_sk  BIGINT NOT NULL,
    transaction_date     DATE   NOT NULL,

    source_system        VARCHAR(100) NOT NULL,
    source_table         VARCHAR(100) NOT NULL,

    insert_dt            TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
    update_dt            TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,

    CONSTRAINT pk_fct_transactions
        PRIMARY KEY (
            transaction_date,
            transaction_src_id,
            product_surr_id,
            promotion_surr_id,
            delivery_surr_id,
            engagement_surr_id,
            store_surr_id,
            customer_surr_id,
            employee_surr_id
        ),

    CONSTRAINT fk_fct_trx_prod
        FOREIGN KEY (product_surr_id)
        REFERENCES bl_dm.dim_products(product_surr_id),

    CONSTRAINT fk_fct_trx_prom
        FOREIGN KEY (promotion_surr_id)
        REFERENCES bl_dm.dim_promotions(promotion_surr_id),

    CONSTRAINT fk_fct_trx_deliv
        FOREIGN KEY (delivery_surr_id)
        REFERENCES bl_dm.dim_deliveries(delivery_surr_id),

    CONSTRAINT fk_fct_trx_eng
        FOREIGN KEY (engagement_surr_id)
        REFERENCES bl_dm.dim_engagements(engagement_surr_id),

    CONSTRAINT fk_fct_trx_store
        FOREIGN KEY (store_surr_id)
        REFERENCES bl_dm.dim_stores(store_surr_id),

    CONSTRAINT fk_fct_trx_cust
        FOREIGN KEY (customer_surr_id)
        REFERENCES bl_dm.dim_customers(customer_surr_id),

    CONSTRAINT fk_fct_trx_emp
        FOREIGN KEY (employee_surr_id)
        REFERENCES bl_dm.dim_employees_scd(employee_surr_id),

    CONSTRAINT fk_fct_trx_date
        FOREIGN KEY (transaction_date_sk)
        REFERENCES bl_dm.dim_dates(date_surr_id)
) PARTITION BY RANGE (transaction_date);


CREATE INDEX IF NOT EXISTS ix_fct_trx_date_sk
    ON bl_dm.fct_transactions (transaction_date_sk);

CREATE INDEX IF NOT EXISTS ix_fct_trx_product
    ON bl_dm.fct_transactions (product_surr_id);

CREATE INDEX IF NOT EXISTS ix_fct_trx_promotion
    ON bl_dm.fct_transactions (promotion_surr_id);

CREATE INDEX IF NOT EXISTS ix_fct_trx_delivery
    ON bl_dm.fct_transactions (delivery_surr_id);

CREATE INDEX IF NOT EXISTS ix_fct_trx_engagement
    ON bl_dm.fct_transactions (engagement_surr_id);

CREATE INDEX IF NOT EXISTS ix_fct_trx_store
    ON bl_dm.fct_transactions (store_surr_id);

CREATE INDEX IF NOT EXISTS ix_fct_trx_customer
    ON bl_dm.fct_transactions (customer_surr_id);

CREATE INDEX IF NOT EXISTS ix_fct_trx_employee
    ON bl_dm.fct_transactions (employee_surr_id);

CREATE INDEX IF NOT EXISTS ix_fct_trx_source
    ON bl_dm.fct_transactions (source_system, source_table);

CREATE INDEX IF NOT EXISTS ix_fct_trx_src_id
    ON bl_dm.fct_transactions (transaction_src_id);

SQL

CREATE TABLE
CREATE INDEX
CREATE INDEX
CREATE INDEX
CREATE INDEX
CREATE INDEX
CREATE INDEX
CREATE INDEX
CREATE INDEX
CREATE INDEX
CREATE INDEX


NOTICE:  relation "fct_transactions" already exists, skipping
NOTICE:  relation "ix_fct_trx_date_sk" already exists, skipping
NOTICE:  relation "ix_fct_trx_product" already exists, skipping
NOTICE:  relation "ix_fct_trx_promotion" already exists, skipping
NOTICE:  relation "ix_fct_trx_delivery" already exists, skipping
NOTICE:  relation "ix_fct_trx_engagement" already exists, skipping
NOTICE:  relation "ix_fct_trx_store" already exists, skipping
NOTICE:  relation "ix_fct_trx_customer" already exists, skipping
NOTICE:  relation "ix_fct_trx_employee" already exists, skipping
NOTICE:  relation "ix_fct_trx_source" already exists, skipping
NOTICE:  relation "ix_fct_trx_src_id" already exists, skipping


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CREATE OR REPLACE PROCEDURE bl_dm.load_fct_transactions_by_month(
    p_year  INTEGER,
    p_month INTEGER
)
LANGUAGE plpgsql
AS $$
DECLARE
    v_proc           TEXT := 'bl_dm.load_fct_transactions_by_month';
    v_partition_name TEXT;
    v_start_date     DATE;
    v_end_date       DATE;
    v_rows_inserted  INTEGER := 0;
    v_err_msg        TEXT;
    v_err_detail     TEXT;
    v_err_hint       TEXT;
BEGIN
    v_start_date := make_date(p_year, p_month, 1);
    v_end_date   := (v_start_date + INTERVAL '1 month')::DATE;
    v_partition_name := 'fct_transactions_' || to_char(v_start_date, 'YYYYMM');

    RAISE NOTICE '--------------------------------------------------';
    RAISE NOTICE 'Processing month: %', to_char(v_start_date, 'YYYY-MM');
    RAISE NOTICE 'Partition name: %', v_partition_name;

    BEGIN
        EXECUTE format('ALTER TABLE bl_dm.fct_transactions DETACH PARTITION bl_dm.%I;', v_partition_name);
        EXECUTE format('DROP TABLE bl_dm.%I;', v_partition_name);
        RAISE NOTICE 'Old partition detached and dropped: %', v_partition_name;
    EXCEPTION
        WHEN undefined_table THEN
            RAISE NOTICE 'No existing partition to drop for %', v_partition_name;
    END;

    EXECUTE format(
        'CREATE TABLE bl_dm.%I PARTITION OF bl_dm.fct_transactions
         FOR VALUES FROM (%L) TO (%L);',
        v_partition_name, v_start_date, v_end_date
    );

    RAISE NOTICE 'Created new partition: %', v_partition_name;

    EXECUTE format($SQL$
        INSERT INTO bl_dm.%I (
            transaction_src_id,
            total_sales,
            quantity,
            unit_price,
            discount_applied,
            payment_method,
            product_surr_id,
            promotion_surr_id,
            delivery_surr_id,
            engagement_surr_id,
            store_surr_id,
            customer_surr_id,
            employee_surr_id,
            transaction_date_sk,
            transaction_date,
            source_system,
            source_table,
            insert_dt,
            update_dt
        )
        SELECT
            t.transaction_id,
            COALESCE(t.total_sales, 0.00),
            COALESCE(t.quantity, 0),
            COALESCE(t.unit_price, 0.00),
            COALESCE(t.discount_applied, 0.00),
            COALESCE(NULLIF(t.payment_method, ''), 'n.a.'),

            COALESCE(dp.product_surr_id, -1),
            COALESCE(dpr.promotion_surr_id, -1),
            COALESCE(ddv.delivery_surr_id, -1),
            COALESCE(den.engagement_surr_id, -1),
            COALESCE(ds.store_surr_id, -1),
            COALESCE(dc.customer_surr_id, -1),

            COALESCE(
                (
                    SELECT des.employee_surr_id
                    FROM bl_dm.dim_employees_scd des
                    WHERE des.employee_src_id = t.employee_id
                      AND t.transaction_dt >= des.start_dt
                      AND t.transaction_dt <  des.end_dt
                    ORDER BY des.start_dt DESC
                    LIMIT 1
                ),
                (
                    SELECT des.employee_surr_id
                    FROM bl_dm.dim_employees_scd des
                    WHERE des.employee_src_id = t.employee_id
                      AND des.start_dt <= t.transaction_dt
                    ORDER BY des.start_dt DESC
                    LIMIT 1
                ),
                (
                    SELECT des.employee_surr_id
                    FROM bl_dm.dim_employees_scd des
                    WHERE des.employee_src_id = t.employee_id
                    ORDER BY des.start_dt ASC
                    LIMIT 1
                ),
                -1
            ) AS employee_surr_id,

            dd.date_surr_id,
            dd.full_date,
            COALESCE(t.source_system, 'n.a.'),
            COALESCE(t.source_table, 'n.a.'),
            CURRENT_TIMESTAMP,
            CURRENT_TIMESTAMP
        FROM bl_3nf.ce_transactions t
        JOIN bl_dm.dim_dates dd
          ON dd.full_date = DATE(t.transaction_dt)
        LEFT JOIN bl_dm.dim_products    dp  ON dp.product_src_id      = t.product_id
        LEFT JOIN bl_dm.dim_promotions  dpr ON dpr.promotion_src_id   = t.promotion_id
        LEFT JOIN bl_dm.dim_deliveries  ddv ON ddv.delivery_src_id    = t.delivery_id
        LEFT JOIN bl_dm.dim_engagements den ON den.engagement_src_id  = t.engagement_id
        LEFT JOIN bl_dm.dim_stores      ds  ON ds.store_src_id        = t.store_id
        LEFT JOIN bl_dm.dim_customers   dc  ON dc.customer_src_id     = t.customer_id
        WHERE dd.full_date >= %L
          AND dd.full_date <  %L;
    $SQL$, v_partition_name, v_start_date, v_end_date);

    GET DIAGNOSTICS v_rows_inserted = ROW_COUNT;

    RAISE NOTICE 'Loaded rows into % : %', v_partition_name, v_rows_inserted;
    RAISE NOTICE 'Finished month: %', to_char(v_start_date, 'YYYY-MM');

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_dm.fct_transactions',
        v_rows_inserted,
        CASE WHEN v_rows_inserted > 0 THEN 'SUCCESS' ELSE 'NO_CHANGE' END,
        'Loaded month=' || to_char(v_start_date, 'YYYY-MM') ||
        ', partition=' || v_partition_name ||
        ', inserted=' || v_rows_inserted || '.'
    );

EXCEPTION
WHEN OTHERS THEN
    GET STACKED DIAGNOSTICS
        v_err_detail = PG_EXCEPTION_DETAIL,
        v_err_hint   = PG_EXCEPTION_HINT;

    v_err_msg := SQLERRM;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_dm.fct_transactions',
        0,
        'FAILED',
        'Monthly fact load failed for ' || COALESCE(to_char(v_start_date, 'YYYY-MM'), 'n.a.'),
        v_err_msg || ' | DETAIL=' || COALESCE(v_err_detail, 'n.a.')
                  || ' | HINT='   || COALESCE(v_err_hint, 'n.a.'),
        'ERROR'
    );

    RAISE;
END;
$$;

SQL

CREATE PROCEDURE


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CREATE OR REPLACE PROCEDURE bl_cl.master_transactions_monthly_load()
LANGUAGE plpgsql
AS $$
DECLARE
    v_proc             TEXT := 'bl_cl.master_transactions_monthly_load';
    v_last_loaded_date DATE;
    v_min_source_date  DATE;
    v_max_source_date  DATE;
    v_start_loop_date  DATE;
    v_months_loaded    INTEGER := 0;
    v_err_msg          TEXT;
    v_err_detail       TEXT;
    v_err_hint         TEXT;
BEGIN
    SELECT MIN(DATE(transaction_dt)), MAX(DATE(transaction_dt))
    INTO v_min_source_date, v_max_source_date
    FROM bl_3nf.ce_transactions
    WHERE transaction_id <> 'n.a.';

    IF v_min_source_date IS NULL OR v_max_source_date IS NULL THEN
        RAISE NOTICE 'No source rows found in bl_3nf.ce_transactions';
        PERFORM bl_cl.log_etl_event(
            v_proc,
            'bl_dm.fct_transactions',
            0,
            'NO_CHANGE',
            'No source rows found. Nothing to load.'
        );
        RETURN;
    END IF;

    SELECT MAX(transaction_date)
    INTO v_last_loaded_date
    FROM bl_dm.fct_transactions;

    IF v_last_loaded_date IS NULL THEN
        v_start_loop_date := date_trunc('month', v_min_source_date)::DATE;
        RAISE NOTICE 'Fact is empty. Starting from first source month: %', to_char(v_start_loop_date, 'YYYY-MM');
    ELSE
        v_start_loop_date := (date_trunc('month', v_last_loaded_date)::DATE + INTERVAL '1 month')::DATE;
        RAISE NOTICE 'Last loaded date: %', v_last_loaded_date;
        RAISE NOTICE 'Next month to load: %', to_char(v_start_loop_date, 'YYYY-MM');
    END IF;

    IF v_start_loop_date > v_max_source_date THEN
        RAISE NOTICE 'No new months to load.';
        PERFORM bl_cl.log_etl_event(
            v_proc,
            'bl_dm.fct_transactions',
            0,
            'NO_CHANGE',
            'No new months to load.'
        );
        RETURN;
    END IF;

    RAISE NOTICE 'Loading months from % to %',
        to_char(v_start_loop_date, 'YYYY-MM'),
        to_char(v_max_source_date, 'YYYY-MM');

    WHILE v_start_loop_date <= v_max_source_date LOOP
        RAISE NOTICE 'Calling monthly loader for %', to_char(v_start_loop_date, 'YYYY-MM');

        CALL bl_dm.load_fct_transactions_by_month(
            EXTRACT(YEAR  FROM v_start_loop_date)::INT,
            EXTRACT(MONTH FROM v_start_loop_date)::INT
        );

        v_months_loaded := v_months_loaded + 1;
        v_start_loop_date := (v_start_loop_date + INTERVAL '1 month')::DATE;
    END LOOP;

    RAISE NOTICE 'Master monthly load completed. Months loaded=%', v_months_loaded;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_dm.fct_transactions',
        v_months_loaded,
        'SUCCESS',
        'Master monthly load completed. Months loaded=' || v_months_loaded || '.'
    );

EXCEPTION
WHEN OTHERS THEN
    GET STACKED DIAGNOSTICS
        v_err_detail = PG_EXCEPTION_DETAIL,
        v_err_hint   = PG_EXCEPTION_HINT;

    v_err_msg := SQLERRM;

    PERFORM bl_cl.log_etl_event(
        v_proc,
        'bl_dm.fct_transactions',
        0,
        'FAILED',
        'Master monthly load failed',
        v_err_msg || ' | DETAIL=' || COALESCE(v_err_detail, 'n.a.')
                  || ' | HINT='   || COALESCE(v_err_hint, 'n.a.'),
        'ERROR'
    );

    RAISE;
END;
$$;

SQL

CREATE PROCEDURE


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'
CALL bl_cl.master_transactions_monthly_load();


SQL


CALL


NOTICE:  Fact is empty. Starting from first source month: 2020-01
NOTICE:  Loading months from 2020-01 to 2021-12
NOTICE:  Calling monthly loader for 2020-01
NOTICE:  --------------------------------------------------
NOTICE:  Processing month: 2020-01
NOTICE:  Partition name: fct_transactions_202001
NOTICE:  No existing partition to drop for fct_transactions_202001
NOTICE:  Created new partition: fct_transactions_202001
NOTICE:  Loaded rows into fct_transactions_202001 : 40444
NOTICE:  Finished month: 2020-01
NOTICE:  Calling monthly loader for 2020-02
NOTICE:  --------------------------------------------------
NOTICE:  Processing month: 2020-02
NOTICE:  Partition name: fct_transactions_202002
NOTICE:  No existing partition to drop for fct_transactions_202002
NOTICE:  Created new partition: fct_transactions_202002
NOTICE:  Loaded rows into fct_transactions_202002 : 37939
NOTICE:  Finished month: 2020-02
NOTICE:  Calling monthly loader for 2020-03
NOTICE:  -----------------------------

In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'
CALL bl_cl.master_transactions_monthly_load();

-- to test only CALL bl_dm.load_fct_transactions_by_month(2020, 1);

SQL


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

/* =========================================================
   A) CE_TRANSACTIONS: HIGH-LEVEL ROW COUNTS
   ========================================================= */

SELECT
    source_system,
    COUNT(*) AS row_count
FROM bl_3nf.ce_transactions
WHERE row_sig <> 'n.a.'
GROUP BY source_system
ORDER BY source_system;


/* =========================================================
   B) CE_TRANSACTIONS: ENTITY COVERAGE CHECK
   Expected:
   - online  : engagement filled, store=-1, employee=-1
   - offline : engagement=-1, store filled, employee filled
   ========================================================= */

SELECT
    source_system,

    COUNT(*) AS total_rows,

    COUNT(*) FILTER (WHERE engagement_id <> -1) AS engagement_found,
    COUNT(*) FILTER (WHERE engagement_id = -1)  AS engagement_missing,

    COUNT(*) FILTER (WHERE store_id <> -1) AS store_found,
    COUNT(*) FILTER (WHERE store_id = -1)  AS store_missing,

    COUNT(*) FILTER (WHERE employee_id <> -1) AS employee_found,
    COUNT(*) FILTER (WHERE employee_id = -1)  AS employee_missing,

    COUNT(*) FILTER (WHERE customer_id <> -1) AS customer_found,
    COUNT(*) FILTER (WHERE customer_id = -1)  AS customer_missing,

    COUNT(*) FILTER (WHERE product_id <> -1) AS product_found,
    COUNT(*) FILTER (WHERE product_id = -1)  AS product_missing,

    COUNT(*) FILTER (WHERE promotion_id <> -1) AS promotion_found,
    COUNT(*) FILTER (WHERE promotion_id = -1)  AS promotion_missing,

    COUNT(*) FILTER (WHERE delivery_id <> -1) AS delivery_found,
    COUNT(*) FILTER (WHERE delivery_id = -1)  AS delivery_missing,

    COUNT(*) FILTER (WHERE city_id <> -1) AS city_found,
    COUNT(*) FILTER (WHERE city_id = -1)  AS city_missing

FROM bl_3nf.ce_transactions
WHERE row_sig <> 'n.a.'
GROUP BY source_system
ORDER BY source_system;


/* =========================================================
   C) CE_TRANSACTIONS: BUSINESS-RULE VALIDATION
   This shows exactly whether online/offline behavior matches expectation
   ========================================================= */

SELECT
    'online_should_have_engagement' AS check_name,
    COUNT(*) AS bad_rows
FROM bl_3nf.ce_transactions
WHERE source_system = 'sa_online_retail'
  AND row_sig <> 'n.a.'
  AND engagement_id = -1

UNION ALL
SELECT
    'online_should_not_have_store',
    COUNT(*)
FROM bl_3nf.ce_transactions
WHERE source_system = 'sa_online_retail'
  AND row_sig <> 'n.a.'
  AND store_id <> -1

UNION ALL
SELECT
    'online_should_not_have_employee',
    COUNT(*)
FROM bl_3nf.ce_transactions
WHERE source_system = 'sa_online_retail'
  AND row_sig <> 'n.a.'
  AND employee_id <> -1

UNION ALL
SELECT
    'offline_should_not_have_engagement',
    COUNT(*)
FROM bl_3nf.ce_transactions
WHERE source_system = 'sa_offline_retail'
  AND row_sig <> 'n.a.'
  AND engagement_id <> -1

UNION ALL
SELECT
    'offline_should_have_store',
    COUNT(*)
FROM bl_3nf.ce_transactions
WHERE source_system = 'sa_offline_retail'
  AND row_sig <> 'n.a.'
  AND store_id = -1

UNION ALL
SELECT
    'offline_should_have_employee',
    COUNT(*)
FROM bl_3nf.ce_transactions
WHERE source_system = 'sa_offline_retail'
  AND row_sig <> 'n.a.'
  AND employee_id = -1
ORDER BY check_name;


/* =========================================================
   D) SAMPLE BAD ROWS FROM CE_TRANSACTIONS
   ========================================================= */

/* offline rows where store_id is unexpectedly missing */
SELECT
    transaction_id,
    transaction_dt,
    source_system,
    source_table,
    store_id,
    employee_id,
    engagement_id,
    customer_id,
    product_id,
    promotion_id,
    delivery_id,
    city_id,
    row_sig
FROM bl_3nf.ce_transactions
WHERE source_system = 'sa_offline_retail'
  AND row_sig <> 'n.a.'
  AND store_id = -1
LIMIT 20;

/* offline rows where employee_id is unexpectedly missing */
SELECT
    transaction_id,
    transaction_dt,
    source_system,
    source_table,
    store_id,
    employee_id,
    engagement_id,
    customer_id,
    product_id,
    promotion_id,
    delivery_id,
    city_id,
    row_sig
FROM bl_3nf.ce_transactions
WHERE source_system = 'sa_offline_retail'
  AND row_sig <> 'n.a.'
  AND employee_id = -1
LIMIT 20;

/* online rows where engagement_id is unexpectedly missing */
SELECT
    transaction_id,
    transaction_dt,
    source_system,
    source_table,
    store_id,
    employee_id,
    engagement_id,
    customer_id,
    product_id,
    promotion_id,
    delivery_id,
    city_id,
    row_sig
FROM bl_3nf.ce_transactions
WHERE source_system = 'sa_online_retail'
  AND row_sig <> 'n.a.'
  AND engagement_id = -1
LIMIT 20;


/* =========================================================
   E) T_MAP_TRANSACTIONS: ARE DERIVED KEYS PRESENT?
   This tells us whether the problem starts already in t_map_transactions
   ========================================================= */

SELECT
    source_system,
    COUNT(*) AS total_rows,

    COUNT(*) FILTER (WHERE customer_src_id IS NOT NULL AND customer_src_id <> 'n.a.') AS customer_src_present,
    COUNT(*) FILTER (WHERE customer_src_id IS NULL OR customer_src_id = 'n.a.') AS customer_src_missing,

    COUNT(*) FILTER (WHERE product_src_id IS NOT NULL AND product_src_id <> 'n.a.') AS product_src_present,
    COUNT(*) FILTER (WHERE product_src_id IS NULL OR product_src_id = 'n.a.') AS product_src_missing,

    COUNT(*) FILTER (WHERE promotion_src_id IS NOT NULL AND promotion_src_id <> 'n.a.') AS promotion_src_present,
    COUNT(*) FILTER (WHERE promotion_src_id IS NULL OR promotion_src_id = 'n.a.') AS promotion_src_missing,

    COUNT(*) FILTER (WHERE store_src_id IS NOT NULL AND store_src_id <> 'n.a.') AS store_src_present,
    COUNT(*) FILTER (WHERE store_src_id IS NULL OR store_src_id = 'n.a.') AS store_src_missing,

    COUNT(*) FILTER (WHERE city_src_id IS NOT NULL AND city_src_id <> 'n.a.') AS city_src_present,
    COUNT(*) FILTER (WHERE city_src_id IS NULL OR city_src_id = 'n.a.') AS city_src_missing,

    COUNT(*) FILTER (WHERE employee_src_id IS NOT NULL AND employee_src_id <> 'n.a.') AS employee_src_present,
    COUNT(*) FILTER (WHERE employee_src_id IS NULL OR employee_src_id = 'n.a.') AS employee_src_missing,

    COUNT(*) FILTER (WHERE delivery_id IS NOT NULL AND delivery_id <> 'n.a.') AS delivery_raw_present,
    COUNT(*) FILTER (WHERE delivery_id IS NULL OR delivery_id = 'n.a.') AS delivery_raw_missing,

    COUNT(*) FILTER (WHERE engagement_id IS NOT NULL AND engagement_id <> 'n.a.') AS engagement_raw_present,
    COUNT(*) FILTER (WHERE engagement_id IS NULL OR engagement_id = 'n.a.') AS engagement_raw_missing

FROM bl_cl.t_map_transactions
WHERE row_sig <> 'n.a.'
GROUP BY source_system
ORDER BY source_system;


/* =========================================================
   F) T_MAP_TRANSACTIONS: BUSINESS-RULE VALIDATION
   ========================================================= */

SELECT
    'offline_store_src_should_exist' AS check_name,
    COUNT(*) AS bad_rows
FROM bl_cl.t_map_transactions
WHERE source_system = 'sa_offline_retail'
  AND row_sig <> 'n.a.'
  AND (store_src_id IS NULL OR store_src_id = 'n.a.')

UNION ALL
SELECT
    'offline_employee_src_should_exist',
    COUNT(*)
FROM bl_cl.t_map_transactions
WHERE source_system = 'sa_offline_retail'
  AND row_sig <> 'n.a.'
  AND (employee_src_id IS NULL OR employee_src_id = 'n.a.')

UNION ALL
SELECT
    'offline_engagement_should_be_na',
    COUNT(*)
FROM bl_cl.t_map_transactions
WHERE source_system = 'sa_offline_retail'
  AND row_sig <> 'n.a.'
  AND COALESCE(engagement_id, 'n.a.') <> 'n.a.'

UNION ALL
SELECT
    'online_engagement_should_exist',
    COUNT(*)
FROM bl_cl.t_map_transactions
WHERE source_system = 'sa_online_retail'
  AND row_sig <> 'n.a.'
  AND (engagement_id IS NULL OR engagement_id = 'n.a.')

UNION ALL
SELECT
    'online_store_src_should_be_na',
    COUNT(*)
FROM bl_cl.t_map_transactions
WHERE source_system = 'sa_online_retail'
  AND row_sig <> 'n.a.'
  AND COALESCE(store_src_id, 'n.a.') <> 'n.a.'

UNION ALL
SELECT
    'online_employee_src_should_be_na',
    COUNT(*)
FROM bl_cl.t_map_transactions
WHERE source_system = 'sa_online_retail'
  AND row_sig <> 'n.a.'
  AND COALESCE(employee_src_id, 'n.a.') <> 'n.a.'
ORDER BY check_name;


/* =========================================================
   G) ARE MAP TABLES LOADED?
   ========================================================= */

SELECT 'bl_cl.t_map_customers'   AS table_name, COUNT(*) AS row_count FROM bl_cl.t_map_customers
UNION ALL
SELECT 'bl_cl.t_map_stores',     COUNT(*) FROM bl_cl.t_map_stores
UNION ALL
SELECT 'bl_cl.t_map_products',   COUNT(*) FROM bl_cl.t_map_products
UNION ALL
SELECT 'bl_cl.t_map_promotions', COUNT(*) FROM bl_cl.t_map_promotions
UNION ALL
SELECT 'bl_cl.t_map_deliveries', COUNT(*) FROM bl_cl.t_map_deliveries
UNION ALL
SELECT 'bl_cl.t_map_engagements',COUNT(*) FROM bl_cl.t_map_engagements
UNION ALL
SELECT 'bl_cl.t_map_employees',  COUNT(*) FROM bl_cl.t_map_employees
UNION ALL
SELECT 'bl_cl.t_map_transactions', COUNT(*) FROM bl_cl.t_map_transactions
ORDER BY table_name;


/* =========================================================
   H) ARE 3NF TABLES LOADED?
   ========================================================= */

SELECT 'bl_3nf.ce_customers'        AS table_name, COUNT(*) AS row_count FROM bl_3nf.ce_customers
UNION ALL
SELECT 'bl_3nf.ce_stores',          COUNT(*) FROM bl_3nf.ce_stores
UNION ALL
SELECT 'bl_3nf.ce_products',        COUNT(*) FROM bl_3nf.ce_products
UNION ALL
SELECT 'bl_3nf.ce_promotions',      COUNT(*) FROM bl_3nf.ce_promotions
UNION ALL
SELECT 'bl_3nf.ce_deliveries',      COUNT(*) FROM bl_3nf.ce_deliveries
UNION ALL
SELECT 'bl_3nf.ce_engagements',     COUNT(*) FROM bl_3nf.ce_engagements
UNION ALL
SELECT 'bl_3nf.ce_employees_scd',   COUNT(*) FROM bl_3nf.ce_employees_scd
UNION ALL
SELECT 'bl_3nf.ce_transactions',    COUNT(*) FROM bl_3nf.ce_transactions
ORDER BY table_name;


/* =========================================================
   I) KEY LOOKUP TABLES: DISTINCT BUSINESS KEYS
   Useful to understand whether the 3NF tables are suspiciously too small
   ========================================================= */

SELECT
    't_map_customers_distinct_customer_src_id' AS metric,
    COUNT(DISTINCT customer_src_id)::BIGINT AS value
FROM bl_cl.t_map_customers

UNION ALL
SELECT
    'ce_customers_distinct_customer_src_id',
    COUNT(DISTINCT customer_src_id)::BIGINT
FROM bl_3nf.ce_customers

UNION ALL
SELECT
    't_map_stores_distinct_store_src_id',
    COUNT(DISTINCT store_src_id)::BIGINT
FROM bl_cl.t_map_stores

UNION ALL
SELECT
    'ce_stores_distinct_store_src_id',
    COUNT(DISTINCT store_src_id)::BIGINT
FROM bl_3nf.ce_stores

UNION ALL
SELECT
    't_map_products_distinct_product_id',
    COUNT(DISTINCT product_id)::BIGINT
FROM bl_cl.t_map_products

UNION ALL
SELECT
    'ce_products_distinct_product_src_id',
    COUNT(DISTINCT product_src_id)::BIGINT
FROM bl_3nf.ce_products

UNION ALL
SELECT
    't_map_promotions_distinct_promotion_id',
    COUNT(DISTINCT promotion_id)::BIGINT
FROM bl_cl.t_map_promotions

UNION ALL
SELECT
    'ce_promotions_distinct_promotion_src_id',
    COUNT(DISTINCT promotion_src_id)::BIGINT
FROM bl_3nf.ce_promotions

UNION ALL
SELECT
    't_map_deliveries_distinct_delivery_id',
    COUNT(DISTINCT delivery_id)::BIGINT
FROM bl_cl.t_map_deliveries

UNION ALL
SELECT
    'ce_deliveries_distinct_delivery_src_id',
    COUNT(DISTINCT delivery_src_id)::BIGINT
FROM bl_3nf.ce_deliveries

UNION ALL
SELECT
    't_map_engagements_distinct_engagement_id',
    COUNT(DISTINCT engagement_id)::BIGINT
FROM bl_cl.t_map_engagements

UNION ALL
SELECT
    'ce_engagements_distinct_engagement_src_id',
    COUNT(DISTINCT engagement_src_id)::BIGINT
FROM bl_3nf.ce_engagements

UNION ALL
SELECT
    't_map_employees_distinct_employee_src_id',
    COUNT(DISTINCT employee_src_id)::BIGINT
FROM bl_cl.t_map_employees

UNION ALL
SELECT
    'ce_employees_scd_distinct_employee_src_id',
    COUNT(DISTINCT employee_src_id)::BIGINT
FROM bl_3nf.ce_employees_scd
ORDER BY metric;


/* =========================================================
   J) DIRECT JOIN DIAGNOSTICS FROM T_MAP_TRANSACTIONS TO 3NF
   This isolates which join is failing before insert
   ========================================================= */

SELECT
    t.source_system,
    COUNT(*) AS total_rows,

    COUNT(cu.customer_id)  AS matched_customer,
    COUNT(pr.product_id)   AS matched_product,
    COUNT(pm.promotion_id) AS matched_promotion,
    COUNT(dv.delivery_id)  AS matched_delivery,
    COUNT(eg.engagement_id) AS matched_engagement,
    COUNT(ci.city_id)      AS matched_city,
    COUNT(st.store_id)     AS matched_store,
    COUNT(emp.employee_id) AS matched_employee

FROM bl_cl.t_map_transactions t
LEFT JOIN bl_3nf.ce_customers cu
    ON cu.customer_src_id = t.customer_src_id
LEFT JOIN bl_3nf.ce_products pr
    ON pr.product_src_id = t.product_src_id
LEFT JOIN bl_3nf.ce_promotions pm
    ON pm.promotion_src_id = t.promotion_src_id
LEFT JOIN bl_3nf.ce_deliveries dv
    ON dv.delivery_src_id = t.delivery_id
LEFT JOIN bl_3nf.ce_engagements eg
    ON eg.engagement_src_id = t.engagement_id
LEFT JOIN bl_3nf.ce_cities ci
    ON ci.city_src_id = t.city_src_id
LEFT JOIN bl_3nf.ce_stores st
    ON st.store_src_id = t.store_src_id
LEFT JOIN bl_3nf.ce_employees_scd emp
    ON emp.employee_src_id = t.employee_src_id
   AND t.transaction_dt >= emp.start_dt
   AND t.transaction_dt <  emp.end_dt
WHERE t.row_sig <> 'n.a.'
GROUP BY t.source_system
ORDER BY t.source_system;


/* =========================================================
   K) SAMPLES OF FAILED LOOKUPS FROM T_MAP_TRANSACTIONS
   ========================================================= */

/* offline rows whose store_src_id cannot find ce_stores */
SELECT
    t.transaction_id,
    t.transaction_dt,
    t.store_src_id,
    t.store_location,
    t.store_city,
    t.store_state,
    t.source_system
FROM bl_cl.t_map_transactions t
LEFT JOIN bl_3nf.ce_stores st
    ON st.store_src_id = t.store_src_id
WHERE t.source_system = 'sa_offline_retail'
  AND t.row_sig <> 'n.a.'
  AND COALESCE(t.store_src_id, 'n.a.') <> 'n.a.'
  AND st.store_id IS NULL
LIMIT 20;

/* offline rows whose employee_src_id cannot find ce_employees_scd */
SELECT
    t.transaction_id,
    t.transaction_dt,
    t.employee_src_id,
    t.employee_name,
    t.employee_hire_date,
    t.source_system
FROM bl_cl.t_map_transactions t
LEFT JOIN bl_3nf.ce_employees_scd emp
    ON emp.employee_src_id = t.employee_src_id
   AND t.transaction_dt >= emp.start_dt
   AND t.transaction_dt <  emp.end_dt
WHERE t.source_system = 'sa_offline_retail'
  AND t.row_sig <> 'n.a.'
  AND COALESCE(t.employee_src_id, 'n.a.') <> 'n.a.'
  AND emp.employee_id IS NULL
LIMIT 20;

/* online rows whose engagement_id cannot find ce_engagements */
SELECT
    t.transaction_id,
    t.transaction_dt,
    t.engagement_id,
    t.source_system
FROM bl_cl.t_map_transactions t
LEFT JOIN bl_3nf.ce_engagements eg
    ON eg.engagement_src_id = t.engagement_id
WHERE t.source_system = 'sa_online_retail'
  AND t.row_sig <> 'n.a.'
  AND COALESCE(t.engagement_id, 'n.a.') <> 'n.a.'
  AND eg.engagement_id IS NULL
LIMIT 20;

/* offline rows whose product_src_id cannot find ce_products */
SELECT
    t.transaction_id,
    t.transaction_dt,
    t.product_id,
    t.product_src_id,
    t.source_system
FROM bl_cl.t_map_transactions t
LEFT JOIN bl_3nf.ce_products pr
    ON pr.product_src_id = t.product_src_id
WHERE t.source_system = 'sa_offline_retail'
  AND t.row_sig <> 'n.a.'
  AND COALESCE(t.product_src_id, 'n.a.') <> 'n.a.'
  AND pr.product_id IS NULL
LIMIT 20;

/* offline rows whose promotion_src_id cannot find ce_promotions */
SELECT
    t.transaction_id,
    t.transaction_dt,
    t.promotion_id,
    t.promotion_src_id,
    t.source_system
FROM bl_cl.t_map_transactions t
LEFT JOIN bl_3nf.ce_promotions pm
    ON pm.promotion_src_id = t.promotion_src_id
WHERE t.source_system = 'sa_offline_retail'
  AND t.row_sig <> 'n.a.'
  AND COALESCE(t.promotion_src_id, 'n.a.') <> 'n.a.'
  AND pm.promotion_id IS NULL
LIMIT 20;

SQL

   source_system   | row_count 
-------------------+-----------
 sa_offline_retail |    475000
 sa_online_retail  |    475000
(2 rows)

   source_system   | total_rows | engagement_found | engagement_missing | store_found | store_missing | employee_found | employee_missing | customer_found | customer_missing | product_found | product_missing | promotion_found | promotion_missing | delivery_found | delivery_missing | city_found | city_missing 
-------------------+------------+------------------+--------------------+-------------+---------------+----------------+------------------+----------------+------------------+---------------+-----------------+-----------------+-------------------+----------------+------------------+------------+--------------
 sa_offline_retail |     475000 |                0 |             475000 |      475000 |             0 |         475000 |                0 |         475000 |                0 |        475000 |               0 |          475000 |               

In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'


/* =========================================================
   0) EXPECTED FACT VIEW FROM CE + DM LOOKUPS
   ========================================================= */
DROP TABLE IF EXISTS tmp_expected_fct;

CREATE TEMP TABLE tmp_expected_fct AS
SELECT
    ce.transaction_id                                        AS transaction_src_id,
    ce.total_sales,
    ce.quantity,
    ce.unit_price,
    ce.discount_applied,
    ce.payment_method,

    COALESCE(dp.product_surr_id, -1)                        AS product_surr_id,
    COALESCE(dpr.promotion_surr_id, -1)                     AS promotion_surr_id,
    COALESCE(ddv.delivery_surr_id, -1)                      AS delivery_surr_id,
    COALESCE(den.engagement_surr_id, -1)                    AS engagement_surr_id,
    COALESCE(ds.store_surr_id, -1)                          AS store_surr_id,
    COALESCE(dc.customer_surr_id, -1)                       AS customer_surr_id,

    COALESCE(
        /* 1) exact range */
        (
            SELECT de.employee_surr_id
            FROM bl_dm.dim_employees_scd de
            WHERE de.employee_src_id = ce.employee_id
              AND ce.transaction_dt >= de.start_dt
              AND ce.transaction_dt <  de.end_dt
            ORDER BY de.start_dt DESC
            LIMIT 1
        ),
        /* 2) closest prior */
        (
            SELECT de.employee_surr_id
            FROM bl_dm.dim_employees_scd de
            WHERE de.employee_src_id = ce.employee_id
              AND de.start_dt <= ce.transaction_dt
            ORDER BY de.start_dt DESC
            LIMIT 1
        ),
        /* 3) earliest version */
        (
            SELECT de.employee_surr_id
            FROM bl_dm.dim_employees_scd de
            WHERE de.employee_src_id = ce.employee_id
            ORDER BY de.start_dt ASC
            LIMIT 1
        ),
        -1
    )                                                       AS employee_surr_id,

    dd.date_surr_id                                         AS transaction_date_sk,
    DATE(ce.transaction_dt)::timestamp                      AS transaction_date,

    ce.source_system,
    ce.source_table
FROM bl_3nf.ce_transactions ce
JOIN bl_dm.dim_dates dd
  ON dd.full_date = DATE(ce.transaction_dt)
LEFT JOIN bl_dm.dim_products    dp
  ON dp.product_src_id = ce.product_id
LEFT JOIN bl_dm.dim_promotions  dpr
  ON dpr.promotion_src_id = ce.promotion_id
LEFT JOIN bl_dm.dim_deliveries  ddv
  ON ddv.delivery_src_id = ce.delivery_id
LEFT JOIN bl_dm.dim_engagements den
  ON den.engagement_src_id = ce.engagement_id
LEFT JOIN bl_dm.dim_stores      ds
  ON ds.store_src_id = ce.store_id
LEFT JOIN bl_dm.dim_customers   dc
  ON dc.customer_src_id = ce.customer_id
WHERE ce.row_sig <> 'n.a.';

CREATE INDEX idx_tmp_expected_fct_key
ON tmp_expected_fct (
    transaction_src_id,
    transaction_date,
    product_surr_id,
    promotion_surr_id,
    delivery_surr_id,
    engagement_surr_id,
    store_surr_id,
    customer_surr_id,
    employee_surr_id
);

/* =========================================================
   1) HIGH LEVEL COUNT CHECK
   ========================================================= */
SELECT
    'ce_transactions_rows' AS metric,
    COUNT(*)::BIGINT       AS value
FROM bl_3nf.ce_transactions
WHERE row_sig <> 'n.a.'

UNION ALL

SELECT
    'expected_fact_rows',
    COUNT(*)::BIGINT
FROM tmp_expected_fct

UNION ALL

SELECT
    'actual_fact_rows',
    COUNT(*)::BIGINT
FROM bl_dm.fct_transactions;


/* =========================================================
   2) MONTHLY COUNT CHECK
   same month counts should match
   ========================================================= */
SELECT
    COALESCE(e.ym, f.ym) AS ym,
    COALESCE(e.expected_rows, 0) AS expected_rows,
    COALESCE(f.fact_rows, 0)     AS fact_rows,
    COALESCE(f.fact_rows, 0) - COALESCE(e.expected_rows, 0) AS diff
FROM (
    SELECT
        TO_CHAR(transaction_date, 'YYYY-MM') AS ym,
        COUNT(*) AS expected_rows
    FROM tmp_expected_fct
    GROUP BY 1
) e
FULL OUTER JOIN (
    SELECT
        TO_CHAR(transaction_date, 'YYYY-MM') AS ym,
        COUNT(*) AS fact_rows
    FROM bl_dm.fct_transactions
    GROUP BY 1
) f
ON e.ym = f.ym
ORDER BY ym;


/* =========================================================
   3) ONLINE / OFFLINE DISTRIBUTION CHECK IN FACT
   Expected:
   - offline: employee filled, store filled, engagement = -1
   - online : engagement filled, employee = -1, store = -1
   ========================================================= */
SELECT
    source_system,
    COUNT(*) AS total_rows,

    COUNT(*) FILTER (WHERE engagement_surr_id <> -1) AS engagement_found,
    COUNT(*) FILTER (WHERE engagement_surr_id = -1)  AS engagement_missing,

    COUNT(*) FILTER (WHERE store_surr_id <> -1) AS store_found,
    COUNT(*) FILTER (WHERE store_surr_id = -1)  AS store_missing,

    COUNT(*) FILTER (WHERE employee_surr_id <> -1) AS employee_found,
    COUNT(*) FILTER (WHERE employee_surr_id = -1)  AS employee_missing
FROM bl_dm.fct_transactions
GROUP BY source_system
ORDER BY source_system;


/* =========================================================
   4) BUSINESS RULE VALIDATION
   should all be 0 bad rows
   ========================================================= */
SELECT
    'offline_should_have_employee' AS check_name,
    COUNT(*) AS bad_rows
FROM bl_dm.fct_transactions
WHERE source_system = 'sa_offline_retail'
  AND employee_surr_id = -1

UNION ALL
SELECT
    'offline_should_have_store',
    COUNT(*)
FROM bl_dm.fct_transactions
WHERE source_system = 'sa_offline_retail'
  AND store_surr_id = -1

UNION ALL
SELECT
    'offline_should_not_have_engagement',
    COUNT(*)
FROM bl_dm.fct_transactions
WHERE source_system = 'sa_offline_retail'
  AND engagement_surr_id <> -1

UNION ALL
SELECT
    'online_should_have_engagement',
    COUNT(*)
FROM bl_dm.fct_transactions
WHERE source_system = 'sa_online_retail'
  AND engagement_surr_id = -1

UNION ALL
SELECT
    'online_should_not_have_employee',
    COUNT(*)
FROM bl_dm.fct_transactions
WHERE source_system = 'sa_online_retail'
  AND employee_surr_id <> -1

UNION ALL
SELECT
    'online_should_not_have_store',
    COUNT(*)
FROM bl_dm.fct_transactions
WHERE source_system = 'sa_online_retail'
  AND store_surr_id <> -1
ORDER BY check_name;


/* =========================================================
   5) EXPECTED BUT MISSING IN FACT
   should be 0
   ========================================================= */
SELECT COUNT(*) AS expected_but_missing_in_fact
FROM (
    SELECT
        transaction_src_id,
        total_sales,
        quantity,
        unit_price,
        discount_applied,
        payment_method,
        product_surr_id,
        promotion_surr_id,
        delivery_surr_id,
        engagement_surr_id,
        store_surr_id,
        customer_surr_id,
        employee_surr_id,
        transaction_date_sk,
        transaction_date,
        source_system,
        source_table
    FROM tmp_expected_fct

    EXCEPT

    SELECT
        transaction_src_id,
        total_sales,
        quantity,
        unit_price,
        discount_applied,
        payment_method,
        product_surr_id,
        promotion_surr_id,
        delivery_surr_id,
        engagement_surr_id,
        store_surr_id,
        customer_surr_id,
        employee_surr_id,
        transaction_date_sk,
        transaction_date,
        source_system,
        source_table
    FROM bl_dm.fct_transactions
) x;


/* =========================================================
   6) FACT BUT NOT EXPECTED
   should be 0
   ========================================================= */
SELECT COUNT(*) AS fact_but_not_expected
FROM (
    SELECT
        transaction_src_id,
        total_sales,
        quantity,
        unit_price,
        discount_applied,
        payment_method,
        product_surr_id,
        promotion_surr_id,
        delivery_surr_id,
        engagement_surr_id,
        store_surr_id,
        customer_surr_id,
        employee_surr_id,
        transaction_date_sk,
        transaction_date,
        source_system,
        source_table
    FROM bl_dm.fct_transactions

    EXCEPT

    SELECT
        transaction_src_id,
        total_sales,
        quantity,
        unit_price,
        discount_applied,
        payment_method,
        product_surr_id,
        promotion_surr_id,
        delivery_surr_id,
        engagement_surr_id,
        store_surr_id,
        customer_surr_id,
        employee_surr_id,
        transaction_date_sk,
        transaction_date,
        source_system,
        source_table
    FROM tmp_expected_fct
) x;


/* =========================================================
   7) SAMPLE MISSING ROWS
   if count > 0, inspect these
   ========================================================= */
SELECT *
FROM (
    SELECT
        transaction_src_id,
        total_sales,
        quantity,
        unit_price,
        discount_applied,
        payment_method,
        product_surr_id,
        promotion_surr_id,
        delivery_surr_id,
        engagement_surr_id,
        store_surr_id,
        customer_surr_id,
        employee_surr_id,
        transaction_date_sk,
        transaction_date,
        source_system,
        source_table
    FROM tmp_expected_fct

    EXCEPT

    SELECT
        transaction_src_id,
        total_sales,
        quantity,
        unit_price,
        discount_applied,
        payment_method,
        product_surr_id,
        promotion_surr_id,
        delivery_surr_id,
        engagement_surr_id,
        store_surr_id,
        customer_surr_id,
        employee_surr_id,
        transaction_date_sk,
        transaction_date,
        source_system,
        source_table
    FROM bl_dm.fct_transactions
) x
LIMIT 20;


/* =========================================================
   8) SAMPLE EXTRA ROWS IN FACT
   if count > 0, inspect these
   ========================================================= */
SELECT *
FROM (
    SELECT
        transaction_src_id,
        total_sales,
        quantity,
        unit_price,
        discount_applied,
        payment_method,
        product_surr_id,
        promotion_surr_id,
        delivery_surr_id,
        engagement_surr_id,
        store_surr_id,
        customer_surr_id,
        employee_surr_id,
        transaction_date_sk,
        transaction_date,
        source_system,
        source_table
    FROM bl_dm.fct_transactions

    EXCEPT

    SELECT
        transaction_src_id,
        total_sales,
        quantity,
        unit_price,
        discount_applied,
        payment_method,
        product_surr_id,
        promotion_surr_id,
        delivery_surr_id,
        engagement_surr_id,
        store_surr_id,
        customer_surr_id,
        employee_surr_id,
        transaction_date_sk,
        transaction_date,
        source_system,
        source_table
    FROM tmp_expected_fct
) x
LIMIT 20;

SQL


DROP TABLE
SELECT 950000
CREATE INDEX
        metric        | value  
----------------------+--------
 ce_transactions_rows | 950000
 expected_fact_rows   | 950000
 actual_fact_rows     | 950000
(3 rows)

   ym    | expected_rows | fact_rows | diff 
---------+---------------+-----------+------
 2020-01 |         40444 |     40444 |    0
 2020-02 |         37939 |     37939 |    0
 2020-03 |         40127 |     40127 |    0
 2020-04 |         38927 |     38927 |    0
 2020-05 |         40518 |     40518 |    0
 2020-06 |         38842 |     38842 |    0
 2020-07 |         40304 |     40304 |    0
 2020-08 |         40150 |     40150 |    0
 2020-09 |         38957 |     38957 |    0
 2020-10 |         40879 |     40879 |    0
 2020-11 |         39010 |     39010 |    0
 2020-12 |         39893 |     39893 |    0
 2021-01 |         40301 |     40301 |    0
 2021-02 |         36368 |     36368 |    0
 2021-03 |         40335 |     40335 |    0
 2021-04 |         39010 |     39010 |    0
 

NOTICE:  table "tmp_expected_fct" does not exist, skipping


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CREATE OR REPLACE PROCEDURE bl_cl.master_full_load(
    p_source_system   VARCHAR DEFAULT NULL,
    p_file_id         BIGINT DEFAULT NULL,
    p_trigger_type    VARCHAR DEFAULT 'MANUAL'
)
LANGUAGE plpgsql
AS $$
DECLARE
    v_batch_id BIGINT;
    v_step_id  BIGINT;
BEGIN
    /* =========================================================
       1) OPEN BATCH
       ========================================================= */
    INSERT INTO bl_cl.etl_batch_run (
        pipeline_name,
        trigger_type,
        status,
        source_system
    )
    VALUES (
        'retail_dwh_master_full_load',
        p_trigger_type,
        'STARTED',
        p_source_system
    )
    RETURNING batch_id INTO v_batch_id;

    PERFORM bl_cl.log_etl_event(
        'bl_cl.master_full_load',
        'MASTER',
        0,
        'STARTED',
        'Master load started. source_system=' || COALESCE(p_source_system,'n.a.')
            || ', file_id=' || COALESCE(p_file_id::TEXT,'n.a.'),
        NULL,
        'INFO',
        0,0,0,0,
        v_batch_id
    );

    /* =========================================================
       2) MAP LAYER
       ========================================================= */

    -- load_map_customers
    INSERT INTO bl_cl.etl_step_run(batch_id, step_name, step_order, target_object, status)
    VALUES (v_batch_id, 'load_map_customers', 10, 'bl_cl.t_map_customers', 'STARTED')
    RETURNING step_run_id INTO v_step_id;

    CALL bl_cl.load_map_customers();

    UPDATE bl_cl.etl_step_run
       SET status = 'SUCCESS',
           end_ts = CURRENT_TIMESTAMP
     WHERE step_run_id = v_step_id;

    -- load_map_stores
    INSERT INTO bl_cl.etl_step_run(batch_id, step_name, step_order, target_object, status)
    VALUES (v_batch_id, 'load_map_stores', 20, 'bl_cl.t_map_stores', 'STARTED')
    RETURNING step_run_id INTO v_step_id;

    CALL bl_cl.load_map_stores();

    UPDATE bl_cl.etl_step_run
       SET status = 'SUCCESS',
           end_ts = CURRENT_TIMESTAMP
     WHERE step_run_id = v_step_id;

    -- load_map_products
    INSERT INTO bl_cl.etl_step_run(batch_id, step_name, step_order, target_object, status)
    VALUES (v_batch_id, 'load_map_products', 30, 'bl_cl.t_map_products', 'STARTED')
    RETURNING step_run_id INTO v_step_id;

    CALL bl_cl.load_map_products();

    UPDATE bl_cl.etl_step_run
       SET status = 'SUCCESS',
           end_ts = CURRENT_TIMESTAMP
     WHERE step_run_id = v_step_id;

    -- load_map_promotions
    INSERT INTO bl_cl.etl_step_run(batch_id, step_name, step_order, target_object, status)
    VALUES (v_batch_id, 'load_map_promotions', 40, 'bl_cl.t_map_promotions', 'STARTED')
    RETURNING step_run_id INTO v_step_id;

    CALL bl_cl.load_map_promotions();

    UPDATE bl_cl.etl_step_run
       SET status = 'SUCCESS',
           end_ts = CURRENT_TIMESTAMP
     WHERE step_run_id = v_step_id;

    -- load_map_deliveries
    INSERT INTO bl_cl.etl_step_run(batch_id, step_name, step_order, target_object, status)
    VALUES (v_batch_id, 'load_map_deliveries', 50, 'bl_cl.t_map_deliveries', 'STARTED')
    RETURNING step_run_id INTO v_step_id;

    CALL bl_cl.load_map_deliveries();

    UPDATE bl_cl.etl_step_run
       SET status = 'SUCCESS',
           end_ts = CURRENT_TIMESTAMP
     WHERE step_run_id = v_step_id;

    -- load_map_engagements
    INSERT INTO bl_cl.etl_step_run(batch_id, step_name, step_order, target_object, status)
    VALUES (v_batch_id, 'load_map_engagements', 60, 'bl_cl.t_map_engagements', 'STARTED')
    RETURNING step_run_id INTO v_step_id;

    CALL bl_cl.load_map_engagements();

    UPDATE bl_cl.etl_step_run
       SET status = 'SUCCESS',
           end_ts = CURRENT_TIMESTAMP
     WHERE step_run_id = v_step_id;

    -- load_map_employees
    INSERT INTO bl_cl.etl_step_run(batch_id, step_name, step_order, target_object, status)
    VALUES (v_batch_id, 'load_map_employees', 70, 'bl_cl.t_map_employees', 'STARTED')
    RETURNING step_run_id INTO v_step_id;

    CALL bl_cl.load_map_employees();

    UPDATE bl_cl.etl_step_run
       SET status = 'SUCCESS',
           end_ts = CURRENT_TIMESTAMP
     WHERE step_run_id = v_step_id;

    -- load_map_transactions
    INSERT INTO bl_cl.etl_step_run(batch_id, step_name, step_order, target_object, status)
    VALUES (v_batch_id, 'load_map_transactions', 80, 'bl_cl.t_map_transactions', 'STARTED')
    RETURNING step_run_id INTO v_step_id;

    CALL bl_cl.load_map_transactions();

    UPDATE bl_cl.etl_step_run
       SET status = 'SUCCESS',
           end_ts = CURRENT_TIMESTAMP
     WHERE step_run_id = v_step_id;

    /* =========================================================
       3) 3NF REFERENCE ENTITIES
       ========================================================= */

    INSERT INTO bl_cl.etl_step_run(batch_id, step_name, step_order, target_object, status)
    VALUES (v_batch_id, 'load_ce_states', 90, 'bl_3nf.ce_states', 'STARTED')
    RETURNING step_run_id INTO v_step_id;
    CALL bl_cl.load_ce_states();
    UPDATE bl_cl.etl_step_run SET status='SUCCESS', end_ts=CURRENT_TIMESTAMP WHERE step_run_id=v_step_id;

    INSERT INTO bl_cl.etl_step_run(batch_id, step_name, step_order, target_object, status)
    VALUES (v_batch_id, 'load_ce_cities', 100, 'bl_3nf.ce_cities', 'STARTED')
    RETURNING step_run_id INTO v_step_id;
    CALL bl_cl.load_ce_cities();
    UPDATE bl_cl.etl_step_run SET status='SUCCESS', end_ts=CURRENT_TIMESTAMP WHERE step_run_id=v_step_id;

    INSERT INTO bl_cl.etl_step_run(batch_id, step_name, step_order, target_object, status)
    VALUES (v_batch_id, 'load_ce_addresses', 110, 'bl_3nf.ce_addresses', 'STARTED')
    RETURNING step_run_id INTO v_step_id;
    CALL bl_cl.load_ce_addresses();
    UPDATE bl_cl.etl_step_run SET status='SUCCESS', end_ts=CURRENT_TIMESTAMP WHERE step_run_id=v_step_id;

    INSERT INTO bl_cl.etl_step_run(batch_id, step_name, step_order, target_object, status)
    VALUES (v_batch_id, 'load_ce_product_categories', 120, 'bl_3nf.ce_product_categories', 'STARTED')
    RETURNING step_run_id INTO v_step_id;
    CALL bl_cl.load_ce_product_categories();
    UPDATE bl_cl.etl_step_run SET status='SUCCESS', end_ts=CURRENT_TIMESTAMP WHERE step_run_id=v_step_id;

    INSERT INTO bl_cl.etl_step_run(batch_id, step_name, step_order, target_object, status)
    VALUES (v_batch_id, 'load_ce_promotion_types', 130, 'bl_3nf.ce_promotion_types', 'STARTED')
    RETURNING step_run_id INTO v_step_id;
    CALL bl_cl.load_ce_promotion_types();
    UPDATE bl_cl.etl_step_run SET status='SUCCESS', end_ts=CURRENT_TIMESTAMP WHERE step_run_id=v_step_id;

    INSERT INTO bl_cl.etl_step_run(batch_id, step_name, step_order, target_object, status)
    VALUES (v_batch_id, 'load_ce_shipping_partners', 140, 'bl_3nf.ce_shipping_partners', 'STARTED')
    RETURNING step_run_id INTO v_step_id;
    CALL bl_cl.load_ce_shipping_partners();
    UPDATE bl_cl.etl_step_run SET status='SUCCESS', end_ts=CURRENT_TIMESTAMP WHERE step_run_id=v_step_id;

    /* =========================================================
       4) 3NF BUSINESS ENTITIES
       ========================================================= */

    INSERT INTO bl_cl.etl_step_run(batch_id, step_name, step_order, target_object, status)
    VALUES (v_batch_id, 'load_ce_customers', 150, 'bl_3nf.ce_customers', 'STARTED')
    RETURNING step_run_id INTO v_step_id;
    CALL bl_cl.load_ce_customers();
    UPDATE bl_cl.etl_step_run SET status='SUCCESS', end_ts=CURRENT_TIMESTAMP WHERE step_run_id=v_step_id;

    INSERT INTO bl_cl.etl_step_run(batch_id, step_name, step_order, target_object, status)
    VALUES (v_batch_id, 'load_ce_stores', 160, 'bl_3nf.ce_stores', 'STARTED')
    RETURNING step_run_id INTO v_step_id;
    CALL bl_cl.load_ce_stores();
    UPDATE bl_cl.etl_step_run SET status='SUCCESS', end_ts=CURRENT_TIMESTAMP WHERE step_run_id=v_step_id;

    INSERT INTO bl_cl.etl_step_run(batch_id, step_name, step_order, target_object, status)
    VALUES (v_batch_id, 'load_ce_products', 170, 'bl_3nf.ce_products', 'STARTED')
    RETURNING step_run_id INTO v_step_id;
    CALL bl_cl.load_ce_products();
    UPDATE bl_cl.etl_step_run SET status='SUCCESS', end_ts=CURRENT_TIMESTAMP WHERE step_run_id=v_step_id;

    INSERT INTO bl_cl.etl_step_run(batch_id, step_name, step_order, target_object, status)
    VALUES (v_batch_id, 'load_ce_promotions', 180, 'bl_3nf.ce_promotions', 'STARTED')
    RETURNING step_run_id INTO v_step_id;
    CALL bl_cl.load_ce_promotions();
    UPDATE bl_cl.etl_step_run SET status='SUCCESS', end_ts=CURRENT_TIMESTAMP WHERE step_run_id=v_step_id;

    INSERT INTO bl_cl.etl_step_run(batch_id, step_name, step_order, target_object, status)
    VALUES (v_batch_id, 'load_ce_deliveries', 190, 'bl_3nf.ce_deliveries', 'STARTED')
    RETURNING step_run_id INTO v_step_id;
    CALL bl_cl.load_ce_deliveries();
    UPDATE bl_cl.etl_step_run SET status='SUCCESS', end_ts=CURRENT_TIMESTAMP WHERE step_run_id=v_step_id;

    INSERT INTO bl_cl.etl_step_run(batch_id, step_name, step_order, target_object, status)
    VALUES (v_batch_id, 'load_ce_engagements', 200, 'bl_3nf.ce_engagements', 'STARTED')
    RETURNING step_run_id INTO v_step_id;
    CALL bl_cl.load_ce_engagements();
    UPDATE bl_cl.etl_step_run SET status='SUCCESS', end_ts=CURRENT_TIMESTAMP WHERE step_run_id=v_step_id;

    INSERT INTO bl_cl.etl_step_run(batch_id, step_name, step_order, target_object, status)
    VALUES (v_batch_id, 'load_ce_employees_scd', 210, 'bl_3nf.ce_employees_scd', 'STARTED')
    RETURNING step_run_id INTO v_step_id;
    CALL bl_cl.load_ce_employees_scd();
    UPDATE bl_cl.etl_step_run SET status='SUCCESS', end_ts=CURRENT_TIMESTAMP WHERE step_run_id=v_step_id;

    INSERT INTO bl_cl.etl_step_run(batch_id, step_name, step_order, target_object, status)
    VALUES (v_batch_id, 'load_ce_transactions', 220, 'bl_3nf.ce_transactions', 'STARTED')
    RETURNING step_run_id INTO v_step_id;
    CALL bl_cl.load_ce_transactions();
    UPDATE bl_cl.etl_step_run SET status='SUCCESS', end_ts=CURRENT_TIMESTAMP WHERE step_run_id=v_step_id;

    /* =========================================================
       5) DIMENSIONS
       ========================================================= */

    INSERT INTO bl_cl.etl_step_run(batch_id, step_name, step_order, target_object, status)
    VALUES (v_batch_id, 'load_dim_customers', 230, 'bl_dm.dim_customers', 'STARTED')
    RETURNING step_run_id INTO v_step_id;
    CALL bl_cl.load_dim_customers();
    UPDATE bl_cl.etl_step_run SET status='SUCCESS', end_ts=CURRENT_TIMESTAMP WHERE step_run_id=v_step_id;

    INSERT INTO bl_cl.etl_step_run(batch_id, step_name, step_order, target_object, status)
    VALUES (v_batch_id, 'load_dim_stores', 240, 'bl_dm.dim_stores', 'STARTED')
    RETURNING step_run_id INTO v_step_id;
    CALL bl_cl.load_dim_stores();
    UPDATE bl_cl.etl_step_run SET status='SUCCESS', end_ts=CURRENT_TIMESTAMP WHERE step_run_id=v_step_id;

    INSERT INTO bl_cl.etl_step_run(batch_id, step_name, step_order, target_object, status)
    VALUES (v_batch_id, 'load_dim_products', 250, 'bl_dm.dim_products', 'STARTED')
    RETURNING step_run_id INTO v_step_id;
    CALL bl_cl.load_dim_products();
    UPDATE bl_cl.etl_step_run SET status='SUCCESS', end_ts=CURRENT_TIMESTAMP WHERE step_run_id=v_step_id;

    INSERT INTO bl_cl.etl_step_run(batch_id, step_name, step_order, target_object, status)
    VALUES (v_batch_id, 'load_dim_promotions', 260, 'bl_dm.dim_promotions', 'STARTED')
    RETURNING step_run_id INTO v_step_id;
    CALL bl_cl.load_dim_promotions();
    UPDATE bl_cl.etl_step_run SET status='SUCCESS', end_ts=CURRENT_TIMESTAMP WHERE step_run_id=v_step_id;

    INSERT INTO bl_cl.etl_step_run(batch_id, step_name, step_order, target_object, status)
    VALUES (v_batch_id, 'load_dim_deliveries', 270, 'bl_dm.dim_deliveries', 'STARTED')
    RETURNING step_run_id INTO v_step_id;
    CALL bl_cl.load_dim_deliveries();
    UPDATE bl_cl.etl_step_run SET status='SUCCESS', end_ts=CURRENT_TIMESTAMP WHERE step_run_id=v_step_id;

    INSERT INTO bl_cl.etl_step_run(batch_id, step_name, step_order, target_object, status)
    VALUES (v_batch_id, 'load_dim_engagements', 280, 'bl_dm.dim_engagements', 'STARTED')
    RETURNING step_run_id INTO v_step_id;
    CALL bl_cl.load_dim_engagements();
    UPDATE bl_cl.etl_step_run SET status='SUCCESS', end_ts=CURRENT_TIMESTAMP WHERE step_run_id=v_step_id;

    INSERT INTO bl_cl.etl_step_run(batch_id, step_name, step_order, target_object, status)
    VALUES (v_batch_id, 'load_dim_employees_scd', 290, 'bl_dm.dim_employees_scd', 'STARTED')
    RETURNING step_run_id INTO v_step_id;
    CALL bl_cl.load_dim_employees_scd();
    UPDATE bl_cl.etl_step_run SET status='SUCCESS', end_ts=CURRENT_TIMESTAMP WHERE step_run_id=v_step_id;

    INSERT INTO bl_cl.etl_step_run(batch_id, step_name, step_order, target_object, status)
    VALUES (v_batch_id, 'load_dim_dates', 300, 'bl_dm.dim_dates', 'STARTED')
    RETURNING step_run_id INTO v_step_id;
    CALL bl_cl.load_dim_dates('2024-01-01'::DATE, '2030-12-31'::DATE);
    UPDATE bl_cl.etl_step_run SET status='SUCCESS', end_ts=CURRENT_TIMESTAMP WHERE step_run_id=v_step_id;

    /* =========================================================
       6) FACT
       ========================================================= */
    INSERT INTO bl_cl.etl_step_run(batch_id, step_name, step_order, target_object, status)
    VALUES (v_batch_id, 'master_transactions_monthly_load', 310, 'bl_dm.fct_transactions', 'STARTED')
    RETURNING step_run_id INTO v_step_id;

    CALL bl_cl.master_transactions_monthly_load();

    UPDATE bl_cl.etl_step_run
       SET status = 'SUCCESS',
           end_ts = CURRENT_TIMESTAMP
     WHERE step_run_id = v_step_id;

    /* =========================================================
       7) CLOSE BATCH
       ========================================================= */
    UPDATE bl_cl.etl_batch_run
       SET status = 'SUCCESS',
           end_ts = CURRENT_TIMESTAMP
     WHERE batch_id = v_batch_id;

    PERFORM bl_cl.log_etl_event(
        'bl_cl.master_full_load',
        'MASTER',
        1,
        'SUCCESS',
        'Master load completed successfully.',
        NULL,
        'INFO',
        0,0,0,0,
        v_batch_id
    );

EXCEPTION
    WHEN OTHERS THEN

        UPDATE bl_cl.etl_step_run
           SET status = 'FAILED',
               end_ts = CURRENT_TIMESTAMP,
               error_message = SQLERRM
         WHERE step_run_id = v_step_id;

        UPDATE bl_cl.etl_batch_run
           SET status = 'FAILED',
               end_ts = CURRENT_TIMESTAMP,
               error_message = SQLERRM
         WHERE batch_id = v_batch_id;

        PERFORM bl_cl.log_etl_event(
            'bl_cl.master_full_load',
            'MASTER',
            0,
            'FAILED',
            'Master load failed.',
            SQLERRM,
            'ERROR',
            0,0,0,0,
            v_batch_id
        );

        RAISE;
END;
$$;

SQL

In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'
CALL bl_cl.master_full_load(
    p_source_system => 'bulk_95_demo',
    p_file_id       => NULL,
    p_trigger_type  => 'MANUAL'
);
SQL

In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

CALL bl_cl.master_full_load();

SQL

In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'


SQL

In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'


SQL



```
INCREMENTAL```



In [ ]:
%%bash
set -e
DB="retail_dw"

sudo chmod o+rx /content
sudo chmod o+rx /content/data
sudo chmod o+r /content/data/03_empty_5_off.csv
sudo chmod o+r /content/data/04_empty_5_on.csv

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

ALTER FOREIGN TABLE sa_offline_retail.ext_offline_retail
  OPTIONS (SET filename '/content/data/03_empty_5_off.csv');

ALTER FOREIGN TABLE sa_online_retail.ext_online_retail
  OPTIONS (SET filename '/content/data/04_empty_5_on.csv');

INSERT INTO sa_offline_retail.src_offline_retail_raw (
  customer_id, gender, marital_status, transaction_id, transaction_date,
  product_id, product_category, quantity, unit_price, discount_applied,
  day_of_week, week_of_year, month_of_year, product_name, product_brand,
  product_stock, product_material, promotion_id, promotion_type,
  promotion_start_date, promotion_end_date, customer_zip_code,
  customer_city, customer_state, store_zip_code, store_city, store_state,
  date_of_birth, payment_method, delivery_id, delivery_type,
  delivery_status, shipping_partner, employee_salary, membership_date,
  store_location, last_purchase_date, total_sales, product_manufacture_date,
  product_expiry_date, promotion_channel, employee_name, employee_position,
  employee_hire_date,
  batch_id, load_type, source_file_name, load_dts, source_row_num
)
SELECT
  customer_id, gender, marital_status, transaction_id, transaction_date,
  product_id, product_category, quantity, unit_price, discount_applied,
  day_of_week, week_of_year, month_of_year, product_name, product_brand,
  product_stock, product_material, promotion_id, promotion_type,
  promotion_start_date, promotion_end_date, customer_zip_code,
  customer_city, customer_state, store_zip_code, store_city, store_state,
  date_of_birth, payment_method, delivery_id, delivery_type,
  delivery_status, shipping_partner, employee_salary, membership_date,
  store_location, last_purchase_date, total_sales, product_manufacture_date,
  product_expiry_date, promotion_channel, employee_name, employee_position,
  employee_hire_date,
  2 AS batch_id,
  'INCREMENTAL' AS load_type,
  '03_empty_5_off.csv' AS source_file_name,
  CURRENT_TIMESTAMP AS load_dts,
  ROW_NUMBER() OVER () AS source_row_num
FROM sa_offline_retail.ext_offline_retail;

INSERT INTO sa_online_retail.src_online_retail_raw (
  customer_id, gender, marital_status, transaction_id, transaction_date,
  product_id, product_category, quantity, unit_price, discount_applied,
  day_of_week, week_of_year, month_of_year, product_name, product_brand,
  product_stock, product_material, promotion_id, promotion_type,
  promotion_start_date, promotion_end_date, customer_zip_code,
  customer_city, customer_state, customer_support_calls, date_of_birth,
  payment_method, delivery_id, delivery_type, delivery_status,
  shipping_partner, membership_date, website_address, order_channel,
  customer_support_method, issue_status, product_manufacture_date,
  product_expiry_date, total_sales, promotion_channel, last_purchase_date,
  app_usage, website_visits, social_media_engagement, engagement_id,
  batch_id, load_type, source_file_name, load_dts, source_row_num
)
SELECT
  customer_id, gender, marital_status, transaction_id, transaction_date,
  product_id, product_category, quantity, unit_price, discount_applied,
  day_of_week, week_of_year, month_of_year, product_name, product_brand,
  product_stock, product_material, promotion_id, promotion_type,
  promotion_start_date, promotion_end_date, customer_zip_code,
  customer_city, customer_state, customer_support_calls, date_of_birth,
  payment_method, delivery_id, delivery_type, delivery_status,
  shipping_partner, membership_date, website_address, order_channel,
  customer_support_method, issue_status, product_manufacture_date,
  product_expiry_date, total_sales, promotion_channel, last_purchase_date,
  app_usage, website_visits, social_media_engagement, engagement_id,
  2 AS batch_id,
  'INCREMENTAL' AS load_type,
  '04_empty_5_on.csv' AS source_file_name,
  CURRENT_TIMESTAMP AS load_dts,
  ROW_NUMBER() OVER () AS source_row_num
FROM sa_online_retail.ext_online_retail;

SQL

In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

/* quick check */
SELECT source_file_name, load_type, COUNT(*) AS row_count
FROM sa_offline_retail.src_offline_retail_raw
GROUP BY source_file_name, load_type
ORDER BY source_file_name, load_type;
SQL

In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

/* =========================================================
   ONLINE CLEAN STAGING
   ========================================================= */

DROP TABLE IF EXISTS sa_online_retail.src_online_retail;
CREATE TABLE sa_online_retail.src_online_retail AS
SELECT
    /* ===================== CUSTOMER ===================== */
    COALESCE(NULLIF(LOWER(TRIM(customer_id)), ''), 'n.a.') AS customer_id,
    COALESCE(NULLIF(LOWER(TRIM(gender)), ''), 'n.a.') AS gender,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(marital_status),' ','_')), ''), 'n.a.') AS marital_status,

    CASE
        WHEN NULLIF(TRIM(date_of_birth), '') IS NOT NULL
        AND TRIM(date_of_birth) ~ '^\d{2}-\d{2}-\d{4}$'
        THEN TO_DATE(TRIM(date_of_birth), 'DD-MM-YYYY')
        WHEN NULLIF(TRIM(date_of_birth), '') IS NOT NULL
        AND TRIM(date_of_birth) ~ '^\d{2}/\d{2}/\d{4}$'
        THEN TO_DATE(TRIM(date_of_birth), 'DD/MM/YYYY')
    END AS birth_of_dt,

    CASE
        WHEN NULLIF(TRIM(membership_date), '') IS NOT NULL
        AND TRIM(membership_date) ~ '^\d{2}-\d{2}-\d{4}$'
        THEN TO_DATE(TRIM(membership_date), 'DD-MM-YYYY')
        WHEN NULLIF(TRIM(membership_date), '') IS NOT NULL
        AND TRIM(membership_date) ~ '^\d{2}/\d{2}/\d{4}$'
        THEN TO_DATE(TRIM(membership_date), 'DD/MM/YYYY')
    END AS membership_dt,

    CASE
        WHEN NULLIF(TRIM(last_purchase_date), '') IS NOT NULL
        AND TRIM(last_purchase_date) ~ '^\d{2}-\d{2}-\d{4} \d{2}:\d{2}$'
        THEN TO_TIMESTAMP(TRIM(last_purchase_date), 'DD-MM-YYYY HH24:MI')
        WHEN NULLIF(TRIM(last_purchase_date), '') IS NOT NULL
        AND TRIM(last_purchase_date) ~ '^\d{2}/\d{2}/\d{4} \d{2}:\d{2}$'
        THEN TO_TIMESTAMP(TRIM(last_purchase_date), 'DD/MM/YYYY HH24:MI')
    END AS last_purchase_dt,

    COALESCE(NULLIF(TRIM(customer_zip_code), ''), 'n.a.') AS customer_zip_code,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(customer_city),' ','_')), ''), 'n.a.') AS customer_city,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(customer_state),' ','_')), ''), 'n.a.') AS customer_state,


    /* ===================== TRANSACTION ===================== */
    COALESCE(NULLIF(LOWER(TRIM(transaction_id)), ''), 'n.a.') AS transaction_id,

    CASE
        WHEN NULLIF(TRIM(transaction_date), '') IS NOT NULL
        AND TRIM(transaction_date) ~ '^\d{2}-\d{2}-\d{4} \d{2}:\d{2}$'
        THEN TO_TIMESTAMP(TRIM(transaction_date), 'DD-MM-YYYY HH24:MI')
        WHEN NULLIF(TRIM(transaction_date), '') IS NOT NULL
        AND TRIM(transaction_date) ~ '^\d{2}/\d{2}/\d{4} \d{2}:\d{2}$'
        THEN TO_TIMESTAMP(TRIM(transaction_date), 'DD/MM/YYYY HH24:MI')
    END AS transaction_dt,

 /* ===================== FINANCIAL ===================== */

    CASE
      WHEN TRIM(quantity) ~ '^-?\d+$'
      THEN quantity::INT
    END  AS quantity,

      CASE
        WHEN TRIM(unit_price) ~ '^-?\d+(\.\d+)?$'
        THEN unit_price::NUMERIC(10,2)
      END AS unit_price,

    CASE
      WHEN TRIM(total_sales) ~ '^-?\d+(\.\d+)?$'
      THEN total_sales::NUMERIC(10,2)
    END AS total_sales,

    CASE
      WHEN TRIM(discount_applied) ~ '^-?\d+(\.\d+)?$'
      THEN discount_applied::NUMERIC(10,2)
    END AS discount_applied,

COALESCE(NULLIF(LOWER(REPLACE(TRIM(payment_method), ' ', '_')), ''), 'n.a.') AS payment_method,

    /* ===================== TIME ===================== */
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(day_of_week), ' ', '_')), ''), 'n.a.') AS day_of_week,

        CASE
            WHEN TRIM(week_of_year) ~ '^\d+$'
            THEN CAST(TRIM(week_of_year) AS INTEGER)
        END AS week_of_year,

        CASE
            WHEN TRIM(month_of_year) ~ '^\d+$'
            THEN CAST(TRIM(month_of_year) AS INTEGER)
        END AS month_of_year,


    /* ===================== PRODUCT ===================== */
    COALESCE(NULLIF(LOWER(TRIM(product_id)), ''), 'n.a.') AS product_id,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(product_category),' ','_')), ''), 'n.a.') AS product_category,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(product_name),' ','_')), ''), 'n.a.') AS product_name,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(product_brand),' ','_')), ''), 'n.a.') AS product_brand,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(product_material),' ','_')), ''), 'n.a.') AS product_material,

    CASE WHEN NULLIF(TRIM(product_stock), '') IS NOT NULL
         THEN product_stock::INT END AS product_stock,

    CASE
        WHEN NULLIF(TRIM(product_manufacture_date), '') IS NOT NULL
        AND TRIM(product_manufacture_date) ~ '^\d{2}-\d{2}-\d{4} \d{2}:\d{2}$'
        THEN TO_TIMESTAMP(TRIM(product_manufacture_date), 'DD-MM-YYYY HH24:MI')
        WHEN NULLIF(TRIM(product_manufacture_date), '') IS NOT NULL
        AND TRIM(product_manufacture_date) ~ '^\d{2}/\d{2}/\d{4} \d{2}:\d{2}$'
        THEN TO_TIMESTAMP(TRIM(product_manufacture_date), 'DD/MM/YYYY HH24:MI')
    END AS product_manufacture_dt,

    CASE
        WHEN NULLIF(TRIM(product_expiry_date), '') IS NOT NULL
        AND TRIM(product_expiry_date) ~ '^\d{2}-\d{2}-\d{4} \d{2}:\d{2}$'
        THEN TO_TIMESTAMP(TRIM(product_expiry_date), 'DD-MM-YYYY HH24:MI')
        WHEN NULLIF(TRIM(product_expiry_date), '') IS NOT NULL
        AND TRIM(product_expiry_date) ~ '^\d{2}/\d{2}/\d{4} \d{2}:\d{2}$'
        THEN TO_TIMESTAMP(TRIM(product_expiry_date), 'DD/MM/YYYY HH24:MI')
    END AS product_expiry_dt,

    /* ===================== PROMOTION ===================== */
    COALESCE(NULLIF(LOWER(TRIM(promotion_id)), ''), 'n.a.') AS promotion_id,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(promotion_type),' ','_')), ''), 'n.a.') AS promotion_type,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(promotion_channel),' ','_')), ''), 'n.a.') AS promotion_channel,

 CASE
    WHEN NULLIF(TRIM(promotion_start_date), '') IS NOT NULL
     AND TRIM(promotion_start_date) ~ '^\d{2}-\d{2}-\d{4} \d{2}:\d{2}$'
    THEN TO_TIMESTAMP(TRIM(promotion_start_date), 'DD-MM-YYYY HH24:MI')
    WHEN NULLIF(TRIM(promotion_start_date), '') IS NOT NULL
     AND TRIM(promotion_start_date) ~ '^\d{2}/\d{2}/\d{4} \d{2}:\d{2}$'
    THEN TO_TIMESTAMP(TRIM(promotion_start_date), 'DD/MM/YYYY HH24:MI')
END AS promotion_start_dt,

CASE
    WHEN NULLIF(TRIM(promotion_end_date), '') IS NOT NULL
     AND TRIM(promotion_end_date) ~ '^\d{2}-\d{2}-\d{4} \d{2}:\d{2}$'
    THEN TO_TIMESTAMP(TRIM(promotion_end_date), 'DD-MM-YYYY HH24:MI')
    WHEN NULLIF(TRIM(promotion_end_date), '') IS NOT NULL
     AND TRIM(promotion_end_date) ~ '^\d{2}/\d{2}/\d{4} \d{2}:\d{2}$'
    THEN TO_TIMESTAMP(TRIM(promotion_end_date), 'DD/MM/YYYY HH24:MI')
END AS promotion_end_dt,


    /* ===================== DELIVERY ===================== */
    COALESCE(NULLIF(LOWER(TRIM(delivery_id)), ''), 'n.a.') AS delivery_id,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(delivery_type), ' ', '_')), ''), 'n.a.') AS delivery_type,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(delivery_status), ' ', '_')), ''), 'n.a.') AS delivery_status,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(shipping_partner),' ','_')), ''), 'n.a.') AS shipping_partner,

   /* ===================== ENGAGEMENT ===================== */

    COALESCE(NULLIF(LOWER(TRIM(engagement_id)), ''), 'n.a.') AS engagement_id,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(website_address), ' ', '_')), ''), 'n.a.') AS website_address,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(order_channel), ' ', '_')), ''), 'n.a.') AS order_channel,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(customer_support_method), ' ', '_')), ''), 'n.a.') AS customer_support_method,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(issue_status), ' ', '_')), ''), 'n.a.') AS issue_status,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(app_usage), ' ', '_')), ''), 'n.a.') AS app_usage,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(social_media_engagement), ' ', '_')), ''), 'n.a.') AS social_media_engagement,

    /* SAFE INTEGER CAST */
    CASE
        WHEN TRIM(src.website_visits) ~ '^\d+$'
        THEN CAST(TRIM(src.website_visits) AS INTEGER)
    END AS website_visits,

    CASE
        WHEN TRIM(src.customer_support_calls) ~ '^\d+$'
        THEN CAST(TRIM(src.customer_support_calls) AS INTEGER)
    END AS customer_support_calls,

        /* ===================== META ===================== */
        insert_dt

    FROM sa_online_retail.src_online_retail_raw src;


/* =========================================================
   OFFLINE CLEAN STAGING
   ========================================================= */

DROP TABLE IF EXISTS sa_offline_retail.src_offline_retail;
CREATE TABLE sa_offline_retail.src_offline_retail AS
SELECT
    /* ===================== CUSTOMER ===================== */
    COALESCE(NULLIF(LOWER(TRIM(customer_id)), ''), 'n.a.') AS customer_id,
    COALESCE(NULLIF(LOWER(TRIM(gender)), ''), 'n.a.') AS gender,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(marital_status),' ','_')), ''), 'n.a.') AS marital_status,

    CASE
        WHEN NULLIF(TRIM(date_of_birth), '') IS NOT NULL
        AND TRIM(date_of_birth) ~ '^\d{2}-\d{2}-\d{4}$'
        THEN TO_DATE(TRIM(date_of_birth), 'DD-MM-YYYY')
        WHEN NULLIF(TRIM(date_of_birth), '') IS NOT NULL
        AND TRIM(date_of_birth) ~ '^\d{2}/\d{2}/\d{4}$'
        THEN TO_DATE(TRIM(date_of_birth), 'DD/MM/YYYY')
    END AS birth_of_dt,

    CASE
        WHEN NULLIF(TRIM(membership_date), '') IS NOT NULL
        AND TRIM(membership_date) ~ '^\d{2}-\d{2}-\d{4}$'
        THEN TO_DATE(TRIM(membership_date), 'DD-MM-YYYY')
        WHEN NULLIF(TRIM(membership_date), '') IS NOT NULL
        AND TRIM(membership_date) ~ '^\d{2}/\d{2}/\d{4}$'
        THEN TO_DATE(TRIM(membership_date), 'DD/MM/YYYY')
    END AS membership_dt,

    CASE
        WHEN NULLIF(TRIM(last_purchase_date), '') IS NOT NULL
        AND TRIM(last_purchase_date) ~ '^\d{2}-\d{2}-\d{4} \d{2}:\d{2}$'
        THEN TO_TIMESTAMP(TRIM(last_purchase_date), 'DD-MM-YYYY HH24:MI')
        WHEN NULLIF(TRIM(last_purchase_date), '') IS NOT NULL
        AND TRIM(last_purchase_date) ~ '^\d{2}/\d{2}/\d{4} \d{2}:\d{2}$'
        THEN TO_TIMESTAMP(TRIM(last_purchase_date), 'DD/MM/YYYY HH24:MI')
    END AS last_purchase_dt,

    COALESCE(NULLIF(TRIM(customer_zip_code), ''), 'n.a.') AS customer_zip_code,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(customer_city),' ','_')), ''), 'n.a.') AS customer_city,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(customer_state),' ','_')), ''), 'n.a.') AS customer_state,


    /* ===================== TRANSACTION ===================== */
    COALESCE(NULLIF(LOWER(TRIM(transaction_id)), ''), 'n.a.') AS transaction_id,

    CASE
        WHEN NULLIF(TRIM(transaction_date), '') IS NOT NULL
        AND TRIM(transaction_date) ~ '^\d{2}-\d{2}-\d{4} \d{2}:\d{2}$'
        THEN TO_TIMESTAMP(TRIM(transaction_date), 'DD-MM-YYYY HH24:MI')
        WHEN NULLIF(TRIM(transaction_date), '') IS NOT NULL
        AND TRIM(transaction_date) ~ '^\d{2}/\d{2}/\d{4} \d{2}:\d{2}$'
        THEN TO_TIMESTAMP(TRIM(transaction_date), 'DD/MM/YYYY HH24:MI')
    END AS transaction_dt,

/* ===================== FINANCIAL ===================== */

    CASE
      WHEN TRIM(quantity) ~ '^-?\d+$'
      THEN quantity::INT
    END  AS quantity,

      CASE
        WHEN TRIM(unit_price) ~ '^-?\d+(\.\d+)?$'
        THEN unit_price::NUMERIC(10,2)
      END AS unit_price,

    CASE
      WHEN TRIM(total_sales) ~ '^-?\d+(\.\d+)?$'
      THEN total_sales::NUMERIC(10,2)
    END AS total_sales,

    CASE
      WHEN TRIM(discount_applied) ~ '^-?\d+(\.\d+)?$'
      THEN discount_applied::NUMERIC(10,2)
    END AS discount_applied,

COALESCE(NULLIF(LOWER(REPLACE(TRIM(payment_method), ' ', '_')), ''), 'n.a.') AS payment_method,

   /* ===================== TIME ===================== */
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(day_of_week), ' ', '_')), ''), 'n.a.') AS day_of_week,

   CASE
    WHEN TRIM(week_of_year) ~ '^\d+$'
    THEN CAST(TRIM(week_of_year) AS INTEGER)
END AS week_of_year,

CASE
    WHEN TRIM(month_of_year) ~ '^\d+$'
    THEN CAST(TRIM(month_of_year) AS INTEGER)
END AS month_of_year,


    /* ===================== STORE ===================== */
    COALESCE(NULLIF(TRIM(store_zip_code), ''), 'n.a.') AS store_zip_code,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(store_city),' ','_')), ''), 'n.a.') AS store_city,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(store_state),' ','_')), ''), 'n.a.') AS store_state,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(store_location),' ','_')), ''), 'n.a.') AS store_location,

    /* ===================== EMPLOYEE ===================== */
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(employee_name),' ','_')), ''), 'n.a.') AS employee_name,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(employee_position),' ','_')), ''), 'n.a.') AS employee_position,

        CASE
            WHEN TRIM(employee_salary) ~ '^-?\d+(\.\d+)?$'
            THEN CAST(TRIM(employee_salary) AS NUMERIC(10,2))
        END AS employee_salary,
          CASE
              WHEN NULLIF(TRIM(employee_hire_date), '') IS NOT NULL
              AND TRIM(employee_hire_date) ~ '^\d{2}-\d{2}-\d{4}$'
              THEN TO_DATE(TRIM(employee_hire_date), 'DD-MM-YYYY')
              WHEN NULLIF(TRIM(employee_hire_date), '') IS NOT NULL
              AND TRIM(employee_hire_date) ~ '^\d{2}/\d{2}/\d{4}$'
              THEN TO_DATE(TRIM(employee_hire_date), 'DD/MM/YYYY')
          END AS employee_hire_date,

    /* ===================== PRODUCT ===================== */
    COALESCE(NULLIF(TRIM(product_id), ''), 'n.a.') AS product_id,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(product_category),' ','_')), ''), 'n.a.') AS product_category,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(product_name),' ','_')), ''), 'n.a.') AS product_name,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(product_brand),' ','_')), ''), 'n.a.') AS product_brand,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(product_material),' ','_')), ''), 'n.a.') AS product_material,

        CASE
            WHEN TRIM(product_stock) ~ '^\d+$'
            THEN CAST(TRIM(product_stock) AS INTEGER)
        END AS product_stock,
    CASE
        WHEN NULLIF(TRIM(product_manufacture_date), '') IS NOT NULL
        AND TRIM(product_manufacture_date) ~ '^\d{2}-\d{2}-\d{4} \d{2}:\d{2}$'
        THEN TO_TIMESTAMP(TRIM(product_manufacture_date), 'DD-MM-YYYY HH24:MI')
        WHEN NULLIF(TRIM(product_manufacture_date), '') IS NOT NULL
        AND TRIM(product_manufacture_date) ~ '^\d{2}/\d{2}/\d{4} \d{2}:\d{2}$'
        THEN TO_TIMESTAMP(TRIM(product_manufacture_date), 'DD/MM/YYYY HH24:MI')
    END AS product_manufacture_dt,

    CASE
        WHEN NULLIF(TRIM(product_expiry_date), '') IS NOT NULL
        AND TRIM(product_expiry_date) ~ '^\d{2}-\d{2}-\d{4} \d{2}:\d{2}$'
        THEN TO_TIMESTAMP(TRIM(product_expiry_date), 'DD-MM-YYYY HH24:MI')
        WHEN NULLIF(TRIM(product_expiry_date), '') IS NOT NULL
        AND TRIM(product_expiry_date) ~ '^\d{2}/\d{2}/\d{4} \d{2}:\d{2}$'
        THEN TO_TIMESTAMP(TRIM(product_expiry_date), 'DD/MM/YYYY HH24:MI')
    END AS product_expiry_dt,

    /* ===================== PROMOTION ===================== */
    COALESCE(NULLIF(TRIM(promotion_id), ''), 'n.a.') AS promotion_id,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(promotion_type),' ','_')), ''), 'n.a.') AS promotion_type,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(promotion_channel),' ','_')), ''), 'n.a.') AS promotion_channel,

 CASE
    WHEN NULLIF(TRIM(promotion_start_date), '') IS NOT NULL
     AND TRIM(promotion_start_date) ~ '^\d{2}-\d{2}-\d{4} \d{2}:\d{2}$'
    THEN TO_TIMESTAMP(TRIM(promotion_start_date), 'DD-MM-YYYY HH24:MI')
    WHEN NULLIF(TRIM(promotion_start_date), '') IS NOT NULL
     AND TRIM(promotion_start_date) ~ '^\d{2}/\d{2}/\d{4} \d{2}:\d{2}$'
    THEN TO_TIMESTAMP(TRIM(promotion_start_date), 'DD/MM/YYYY HH24:MI')
END AS promotion_start_dt,

CASE
    WHEN NULLIF(TRIM(promotion_end_date), '') IS NOT NULL
     AND TRIM(promotion_end_date) ~ '^\d{2}-\d{2}-\d{4} \d{2}:\d{2}$'
    THEN TO_TIMESTAMP(TRIM(promotion_end_date), 'DD-MM-YYYY HH24:MI')
    WHEN NULLIF(TRIM(promotion_end_date), '') IS NOT NULL
     AND TRIM(promotion_end_date) ~ '^\d{2}/\d{2}/\d{4} \d{2}:\d{2}$'
    THEN TO_TIMESTAMP(TRIM(promotion_end_date), 'DD/MM/YYYY HH24:MI')
END AS promotion_end_dt,

 /* ===================== DELIVERY ===================== */
    COALESCE(NULLIF(TRIM(delivery_id), ''), 'n.a.') AS delivery_id,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(delivery_type), ' ', '_')), ''), 'n.a.') AS delivery_type,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(delivery_status), ' ', '_')), ''), 'n.a.') AS delivery_status,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(shipping_partner),' ','_')), ''), 'n.a.') AS shipping_partner,


    /* ===================== META ===================== */
    insert_dt

FROM sa_offline_retail.src_offline_retail_raw;

SQL

In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'
CALL bl_cl.master_full_load(
    p_source_system => 'incremental_5_demo',
    p_file_id       => NULL,
    p_trigger_type  => 'MANUAL'
);
SQL

In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -P pager=off <<'SQL'

SELECT
    source_file_name,
    DATE_TRUNC('month',
        CASE
            WHEN transaction_date ~ '^\d{2}-\d{2}-\d{4} \d{2}:\d{2}$'
            THEN TO_TIMESTAMP(transaction_date, 'DD-MM-YYYY HH24:MI')
            WHEN transaction_date ~ '^\d{2}/\d{2}/\d{4} \d{2}:\d{2}$'
            THEN TO_TIMESTAMP(transaction_date, 'DD/MM/YYYY HH24:MI')
        END
    )::DATE AS month_start,
    COUNT(*) AS row_count
FROM sa_offline_retail.src_offline_retail_raw
WHERE source_file_name = '03_empty_5_off.csv'
GROUP BY source_file_name, month_start

UNION ALL

SELECT
    source_file_name,
    DATE_TRUNC('month',
        CASE
            WHEN transaction_date ~ '^\d{2}-\d{2}-\d{4} \d{2}:\d{2}$'
            THEN TO_TIMESTAMP(transaction_date, 'DD-MM-YYYY HH24:MI')
            WHEN transaction_date ~ '^\d{2}/\d{2}/\d{4} \d{2}:\d{2}$'
            THEN TO_TIMESTAMP(transaction_date, 'DD/MM/YYYY HH24:MI')
        END
    )::DATE AS month_start,
    COUNT(*) AS row_count
FROM sa_online_retail.src_online_retail_raw
WHERE source_file_name = '04_empty_5_on.csv'
GROUP BY source_file_name, month_start
ORDER BY month_start;

SQL

In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -P pager=off <<'SQL'

CALL bl_dm.load_fct_transactions_by_month(2024, 5);
CALL bl_dm.load_fct_transactions_by_month(2024, 6);
SQL

CSV EMPLOYEE OZEL CREATION

In [ ]:
import pandas as pd

# Kaynak olarak bulk offline dosyasını oku
df = pd.read_csv('/content/data/01_empty_95_off.csv')

# Employee bilgisi boş olmayan bir satır seç

cand = df[
    df['employee_name'].notna() &
    df['employee_position'].notna() &
    df['employee_hire_date'].notna() &
    df['transaction_date'].notna()
].copy()

# İlk uygun satırı al
row = cand.iloc[[0]].copy()

# SCD2 değişimini göstermek için pozisyonu değiştir
old_position = str(row.iloc[0]['employee_position'])
row.loc[:, 'employee_position'] = old_position + '_senior'

# observed_ts mantığını tetiklemek için transaction_date'i ileri taşı
# veri formatın DD-MM-YYYY HH:MM veya DD/MM/YYYY HH:MM olabilir
old_trx = str(row.iloc[0]['transaction_date'])

if '-' in old_trx:
    dt = pd.to_datetime(old_trx, format='%d-%m-%Y %H:%M', errors='coerce')
else:
    dt = pd.to_datetime(old_trx, format='%d/%m/%Y %H:%M', errors='coerce')

new_dt = dt + pd.DateOffset(months=2)

# aynı formatı koru
if '-' in old_trx:
    row.loc[:, 'transaction_date'] = new_dt.strftime('%d-%m-%Y %H:%M')
else:
    row.loc[:, 'transaction_date'] = new_dt.strftime('%d/%m/%Y %H:%M')

# İstersen salary de değiştir
if 'employee_salary' in row.columns:
    try:
        row.loc[:, 'employee_salary'] = float(row.iloc[0]['employee_salary']) + 1000
    except:
        pass

output_path = '/content/data/src_offline_retail_employee_inc.csv'
row.to_csv(output_path, index=False)

print("Created:", output_path)
print(row[['employee_name', 'employee_position', 'employee_hire_date', 'transaction_date', 'employee_salary']])

In [ ]:
%%bash
set -e
DB="retail_dw"

sudo chmod o+r /content/data/src_offline_retail_employee_inc.csv

sudo -u postgres psql -d "$DB" -v ON_ERROR_STOP=1 -P pager=off <<'SQL'

DROP FOREIGN TABLE IF EXISTS sa_offline_retail.ext_offline_retail_employee_inc;
DROP TABLE IF EXISTS sa_offline_retail.src_offline_retail_employee_inc;

/* foreign table over the 1-row incremental employee file */
CREATE FOREIGN TABLE sa_offline_retail.ext_offline_retail_employee_inc (
  customer_id              VARCHAR(1000),
  gender                   VARCHAR(1000),
  marital_status           VARCHAR(1000),
  transaction_id           VARCHAR(1000),
  transaction_date         VARCHAR(1000),
  product_id               VARCHAR(1000),
  product_category         VARCHAR(1000),
  quantity                 VARCHAR(1000),
  unit_price               VARCHAR(1000),
  discount_applied         VARCHAR(1000),
  day_of_week              VARCHAR(1000),
  week_of_year             VARCHAR(1000),
  month_of_year            VARCHAR(1000),
  product_name             VARCHAR(1000),
  product_brand            VARCHAR(1000),
  product_stock            VARCHAR(1000),
  product_material         VARCHAR(1000),
  promotion_id             VARCHAR(1000),
  promotion_type           VARCHAR(1000),
  promotion_start_date     VARCHAR(1000),
  promotion_end_date       VARCHAR(1000),
  customer_zip_code        VARCHAR(1000),
  customer_city            VARCHAR(1000),
  customer_state           VARCHAR(1000),
  store_zip_code           VARCHAR(1000),
  store_city               VARCHAR(1000),
  store_state              VARCHAR(1000),
  date_of_birth            VARCHAR(1000),
  payment_method           VARCHAR(1000),
  delivery_id              VARCHAR(1000),
  delivery_type            VARCHAR(1000),
  delivery_status          VARCHAR(1000),
  shipping_partner         VARCHAR(1000),
  employee_salary          VARCHAR(1000),
  membership_date          VARCHAR(1000),
  store_location           VARCHAR(1000),
  last_purchase_date       VARCHAR(1000),
  total_sales              VARCHAR(1000),
  product_manufacture_date VARCHAR(1000),
  product_expiry_date      VARCHAR(1000),
  promotion_channel        VARCHAR(1000),
  employee_name            VARCHAR(1000),
  employee_position        VARCHAR(1000),
  employee_hire_date       VARCHAR(1000)
)
SERVER csv_server
OPTIONS (
  filename  '/content/data/src_offline_retail_employee_inc.csv',
  format    'csv',
  header    'true',
  delimiter ',',
  quote     '"',
  escape    '"'
);

/* typed table compatible with load_map_employees() */
CREATE TABLE sa_offline_retail.src_offline_retail_employee_inc AS
SELECT
    COALESCE(NULLIF(LOWER(TRIM(customer_id)), ''), 'n.a.') AS customer_id,
    COALESCE(NULLIF(LOWER(TRIM(gender)), ''), 'n.a.') AS gender,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(marital_status),' ','_')), ''), 'n.a.') AS marital_status,

    CASE
        WHEN NULLIF(TRIM(transaction_date), '') IS NOT NULL
         AND TRIM(transaction_date) ~ '^\d{2}-\d{2}-\d{4} \d{2}:\d{2}$'
        THEN TO_TIMESTAMP(TRIM(transaction_date), 'DD-MM-YYYY HH24:MI')
        WHEN NULLIF(TRIM(transaction_date), '') IS NOT NULL
         AND TRIM(transaction_date) ~ '^\d{2}/\d{2}/\d{4} \d{2}:\d{2}$'
        THEN TO_TIMESTAMP(TRIM(transaction_date), 'DD/MM/YYYY HH24:MI')
    END AS transaction_dt,

    COALESCE(NULLIF(LOWER(REPLACE(TRIM(employee_name),' ','_')), ''), 'n.a.') AS employee_name,
    COALESCE(NULLIF(LOWER(REPLACE(TRIM(employee_position),' ','_')), ''), 'n.a.') AS employee_position,

    CASE
      WHEN TRIM(employee_salary) ~ '^-?\d+(\.\d+)?$'
      THEN employee_salary::NUMERIC(10,2)
    END AS employee_salary,

    CASE
        WHEN NULLIF(TRIM(employee_hire_date), '') IS NOT NULL
         AND TRIM(employee_hire_date) ~ '^\d{2}-\d{2}-\d{4}$'
        THEN TO_DATE(TRIM(employee_hire_date), 'DD-MM-YYYY')
        WHEN NULLIF(TRIM(employee_hire_date), '') IS NOT NULL
         AND TRIM(employee_hire_date) ~ '^\d{2}/\d{2}/\d{4}$'
        THEN TO_DATE(TRIM(employee_hire_date), 'DD/MM/YYYY')
    END AS employee_hire_date

FROM sa_offline_retail.ext_offline_retail_employee_inc;

/* check */
SELECT * FROM sa_offline_retail.src_offline_retail_employee_inc;

SQL

In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -P pager=off <<'SQL'
CALL bl_cl.load_map_employees();
CALL bl_cl.load_ce_employees_scd();
CALL bl_cl.load_dim_employees_scd();
SQL

In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -P pager=off <<'SQL'
SELECT
    employee_src_id,
    COUNT(*) AS version_count,
    MIN(start_dt) AS first_start_dt,
    MAX(end_dt) AS last_end_dt
FROM bl_3nf.ce_employees_scd
WHERE employee_src_id <> 'n.a.'
GROUP BY employee_src_id
HAVING COUNT(*) > 1
ORDER BY version_count DESC, employee_src_id
LIMIT 20;
SQL

In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -P pager=off <<'SQL'
SELECT
    employee_id,
    employee_src_id,
    employee_name,
    employee_position,
    employee_salary,
    employee_hire_date,
    start_dt,
    end_dt,
    is_active,
    source_system,
    source_table
FROM bl_3nf.ce_employees_scd
WHERE employee_src_id IN (
    SELECT employee_name || '-' || employee_hire_date::TEXT
    FROM sa_offline_retail.src_offline_retail_employee_inc
)
ORDER BY employee_src_id, start_dt;
SQL

In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -P pager=off <<'SQL'
SELECT
    employee_surr_id,
    employee_id,
    employee_src_id,
    employee_name,
    employee_position,
    employee_salary,
    employee_hire_date,
    start_dt,
    end_dt,
    is_active
FROM bl_dm.dim_employees_scd
WHERE employee_src_id IN (
    SELECT employee_name || '-' || employee_hire_date::TEXT
    FROM sa_offline_retail.src_offline_retail_employee_inc
)
ORDER BY employee_src_id, start_dt;
SQL

GRANT-REVOKE PERMISSIONS


In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -P pager=off <<'SQL'

-- ============================================================================
-- MINIMAL RBAC / GRANT-REVOKE SCRIPT
-- Tailored for the current notebook structure
-- Schemas assumed:
--   sa_online_retail
--   sa_offline_retail
--   bl_cl
--   bl_3nf
--   bl_dm
-- ============================================================================


-- ============================================================================
-- 1) SUPPORTING SCHEMAS
-- These are not data layers. They are governance / presentation helper schemas.
-- ============================================================================

CREATE SCHEMA IF NOT EXISTS reporting;
CREATE SCHEMA IF NOT EXISTS analytics_sandbox;
CREATE SCHEMA IF NOT EXISTS dq_audit;


-- ============================================================================
-- 2) CREATE ROLES
-- NOLOGIN roles are permission containers.
-- Later, real login users can be granted these roles.
-- ============================================================================

DO $$
BEGIN
    IF NOT EXISTS (SELECT 1 FROM pg_roles WHERE rolname = 'dw_platform_admin') THEN
        CREATE ROLE dw_platform_admin NOLOGIN;
    END IF;

    IF NOT EXISTS (SELECT 1 FROM pg_roles WHERE rolname = 'dw_data_engineer') THEN
        CREATE ROLE dw_data_engineer NOLOGIN;
    END IF;

    IF NOT EXISTS (SELECT 1 FROM pg_roles WHERE rolname = 'dw_analytics_engineer') THEN
        CREATE ROLE dw_analytics_engineer NOLOGIN;
    END IF;

    IF NOT EXISTS (SELECT 1 FROM pg_roles WHERE rolname = 'dw_bi_analyst') THEN
        CREATE ROLE dw_bi_analyst NOLOGIN;
    END IF;

    IF NOT EXISTS (SELECT 1 FROM pg_roles WHERE rolname = 'dw_data_quality_engineer') THEN
        CREATE ROLE dw_data_quality_engineer NOLOGIN;
    END IF;

    IF NOT EXISTS (SELECT 1 FROM pg_roles WHERE rolname = 'svc_etl_loader') THEN
        CREATE ROLE svc_etl_loader NOLOGIN;
    END IF;

    IF NOT EXISTS (SELECT 1 FROM pg_roles WHERE rolname = 'zara_sales_user') THEN
        CREATE ROLE zara_sales_user NOLOGIN;
    END IF;

    IF NOT EXISTS (SELECT 1 FROM pg_roles WHERE rolname = 'zara_finance_user') THEN
        CREATE ROLE zara_finance_user NOLOGIN;
    END IF;

    IF NOT EXISTS (SELECT 1 FROM pg_roles WHERE rolname = 'zara_budget_user') THEN
        CREATE ROLE zara_budget_user NOLOGIN;
    END IF;

    IF NOT EXISTS (SELECT 1 FROM pg_roles WHERE rolname = 'zara_manager_user') THEN
        CREATE ROLE zara_manager_user NOLOGIN;
    END IF;

    IF NOT EXISTS (SELECT 1 FROM pg_roles WHERE rolname = 'zara_business_admin') THEN
        CREATE ROLE zara_business_admin NOLOGIN;
    END IF;

    IF NOT EXISTS (SELECT 1 FROM pg_roles WHERE rolname = 'zara_customer_portal_user') THEN
        CREATE ROLE zara_customer_portal_user NOLOGIN;
    END IF;
END $$;


-- ============================================================================
-- 3) REVOKE PUBLIC ACCESS
-- Good baseline hardening.
-- ============================================================================

REVOKE ALL ON SCHEMA sa_online_retail  FROM PUBLIC;
REVOKE ALL ON SCHEMA sa_offline_retail FROM PUBLIC;
REVOKE ALL ON SCHEMA bl_cl             FROM PUBLIC;
REVOKE ALL ON SCHEMA bl_3nf            FROM PUBLIC;
REVOKE ALL ON SCHEMA bl_dm             FROM PUBLIC;
REVOKE ALL ON SCHEMA reporting         FROM PUBLIC;
REVOKE ALL ON SCHEMA analytics_sandbox FROM PUBLIC;
REVOKE ALL ON SCHEMA dq_audit          FROM PUBLIC;


-- ============================================================================
-- 4) PLATFORM ADMIN
-- Full control. Use sparingly.
-- ============================================================================

GRANT USAGE, CREATE ON SCHEMA sa_online_retail  TO dw_platform_admin;
GRANT USAGE, CREATE ON SCHEMA sa_offline_retail TO dw_platform_admin;
GRANT USAGE, CREATE ON SCHEMA bl_cl             TO dw_platform_admin;
GRANT USAGE, CREATE ON SCHEMA bl_3nf            TO dw_platform_admin;
GRANT USAGE, CREATE ON SCHEMA bl_dm             TO dw_platform_admin;
GRANT USAGE, CREATE ON SCHEMA reporting         TO dw_platform_admin;
GRANT USAGE, CREATE ON SCHEMA analytics_sandbox TO dw_platform_admin;
GRANT USAGE, CREATE ON SCHEMA dq_audit          TO dw_platform_admin;

GRANT ALL PRIVILEGES ON ALL TABLES
IN SCHEMA sa_online_retail, sa_offline_retail, bl_cl, bl_3nf, bl_dm, reporting, analytics_sandbox, dq_audit
TO dw_platform_admin;

GRANT ALL PRIVILEGES ON ALL SEQUENCES
IN SCHEMA sa_online_retail, sa_offline_retail, bl_cl, bl_3nf, bl_dm, reporting, analytics_sandbox, dq_audit
TO dw_platform_admin;

GRANT ALL PRIVILEGES ON ALL FUNCTIONS
IN SCHEMA bl_cl, bl_dm
TO dw_platform_admin;

GRANT ALL PRIVILEGES ON ALL ROUTINES
IN SCHEMA bl_cl, bl_dm
TO dw_platform_admin;


-- ============================================================================
-- 5) DATA ENGINEER
-- Can manage all warehouse objects inside DB, but this does NOT mean
-- modifying original landed source files outside PostgreSQL.
-- ============================================================================

GRANT USAGE ON SCHEMA sa_online_retail, sa_offline_retail, bl_cl, bl_3nf, bl_dm, reporting
TO dw_data_engineer;

GRANT CREATE ON SCHEMA reporting TO dw_data_engineer;

GRANT SELECT, INSERT, UPDATE, DELETE, TRUNCATE
ON ALL TABLES IN SCHEMA sa_online_retail, sa_offline_retail, bl_cl, bl_3nf, bl_dm
TO dw_data_engineer;

GRANT USAGE, SELECT, UPDATE
ON ALL SEQUENCES IN SCHEMA sa_online_retail, sa_offline_retail, bl_cl, bl_3nf, bl_dm
TO dw_data_engineer;

GRANT EXECUTE ON ALL FUNCTIONS IN SCHEMA bl_cl, bl_dm TO dw_data_engineer;
GRANT EXECUTE ON ALL ROUTINES  IN SCHEMA bl_cl, bl_dm TO dw_data_engineer;

GRANT SELECT, INSERT, UPDATE, DELETE
ON ALL TABLES IN SCHEMA reporting
TO dw_data_engineer;


-- ============================================================================
-- 6) ANALYTICS ENGINEER
-- Read all layers, but create only in reporting / sandbox.
-- Should not mutate core warehouse tables.
-- ============================================================================

GRANT USAGE ON SCHEMA sa_online_retail, sa_offline_retail, bl_cl, bl_3nf, bl_dm, reporting, analytics_sandbox
TO dw_analytics_engineer;

GRANT CREATE ON SCHEMA reporting, analytics_sandbox
TO dw_analytics_engineer;

GRANT SELECT
ON ALL TABLES IN SCHEMA sa_online_retail, sa_offline_retail, bl_cl, bl_3nf, bl_dm
TO dw_analytics_engineer;

GRANT USAGE, SELECT
ON ALL SEQUENCES IN SCHEMA sa_online_retail, sa_offline_retail, bl_cl, bl_3nf, bl_dm
TO dw_analytics_engineer;

GRANT SELECT, INSERT, UPDATE, DELETE
ON ALL TABLES IN SCHEMA reporting, analytics_sandbox
TO dw_analytics_engineer;

GRANT USAGE, SELECT, UPDATE
ON ALL SEQUENCES IN SCHEMA reporting, analytics_sandbox
TO dw_analytics_engineer;


-- ============================================================================
-- 7) BI ANALYST
-- Read-only on datamart and reporting layer.
-- ============================================================================

GRANT USAGE ON SCHEMA bl_dm, reporting
TO dw_bi_analyst;

GRANT SELECT
ON ALL TABLES IN SCHEMA bl_dm, reporting
TO dw_bi_analyst;

GRANT USAGE, SELECT
ON ALL SEQUENCES IN SCHEMA bl_dm, reporting
TO dw_bi_analyst;


-- ============================================================================
-- 8) DATA QUALITY ENGINEER
-- Read all layers + can write DQ results to dq_audit.
-- ============================================================================

GRANT USAGE ON SCHEMA sa_online_retail, sa_offline_retail, bl_cl, bl_3nf, bl_dm, dq_audit
TO dw_data_quality_engineer;

GRANT CREATE ON SCHEMA dq_audit
TO dw_data_quality_engineer;

GRANT SELECT
ON ALL TABLES IN SCHEMA sa_online_retail, sa_offline_retail, bl_cl, bl_3nf, bl_dm
TO dw_data_quality_engineer;

GRANT USAGE, SELECT
ON ALL SEQUENCES IN SCHEMA sa_online_retail, sa_offline_retail, bl_cl, bl_3nf, bl_dm
TO dw_data_quality_engineer;

GRANT SELECT, INSERT, UPDATE, DELETE
ON ALL TABLES IN SCHEMA dq_audit
TO dw_data_quality_engineer;

GRANT USAGE, SELECT, UPDATE
ON ALL SEQUENCES IN SCHEMA dq_audit
TO dw_data_quality_engineer;


-- ============================================================================
-- 9) ETL SERVICE ROLE
-- For scheduled / automated loads.
-- ============================================================================

GRANT USAGE ON SCHEMA sa_online_retail, sa_offline_retail, bl_cl, bl_3nf, bl_dm
TO svc_etl_loader;

GRANT SELECT, INSERT, UPDATE, DELETE, TRUNCATE
ON ALL TABLES IN SCHEMA sa_online_retail, sa_offline_retail, bl_cl, bl_3nf, bl_dm
TO svc_etl_loader;

GRANT USAGE, SELECT, UPDATE
ON ALL SEQUENCES IN SCHEMA sa_online_retail, sa_offline_retail, bl_cl, bl_3nf, bl_dm
TO svc_etl_loader;

GRANT EXECUTE ON ALL FUNCTIONS IN SCHEMA bl_cl, bl_dm TO svc_etl_loader;
GRANT EXECUTE ON ALL ROUTINES  IN SCHEMA bl_cl, bl_dm TO svc_etl_loader;


-- ============================================================================
-- 10) CLIENT-SIDE BUSINESS USERS
-- Simplified approach: reporting-only access.
-- They should not see raw/core layers.
-- ============================================================================

GRANT USAGE ON SCHEMA reporting TO zara_sales_user;
GRANT USAGE ON SCHEMA reporting TO zara_finance_user;
GRANT USAGE ON SCHEMA reporting TO zara_budget_user;
GRANT USAGE ON SCHEMA reporting TO zara_manager_user;
GRANT USAGE ON SCHEMA reporting TO zara_business_admin;

GRANT SELECT ON ALL TABLES IN SCHEMA reporting TO zara_sales_user;
GRANT SELECT ON ALL TABLES IN SCHEMA reporting TO zara_finance_user;
GRANT SELECT ON ALL TABLES IN SCHEMA reporting TO zara_budget_user;
GRANT SELECT ON ALL TABLES IN SCHEMA reporting TO zara_manager_user;
GRANT SELECT ON ALL TABLES IN SCHEMA reporting TO zara_business_admin;


-- ============================================================================
-- 11) CUSTOMER PORTAL USER
-- Minimal version for now: grant nothing on raw/core.
-- Later, this role should be pointed to a secure reporting view with RLS.
-- ============================================================================

GRANT USAGE ON SCHEMA reporting TO zara_customer_portal_user;

-- Example future pattern:
-- GRANT SELECT ON reporting.v_customer_orders_secure TO zara_customer_portal_user;


-- ============================================================================
-- 12) EXPLICITLY BLOCK CLIENT ROLES FROM RAW / CORE LAYERS
-- ============================================================================

REVOKE ALL ON SCHEMA sa_online_retail, sa_offline_retail, bl_cl, bl_3nf, bl_dm FROM
    zara_sales_user,
    zara_finance_user,
    zara_budget_user,
    zara_manager_user,
    zara_business_admin,
    zara_customer_portal_user;

REVOKE ALL ON ALL TABLES IN SCHEMA sa_online_retail, sa_offline_retail, bl_cl, bl_3nf, bl_dm FROM
    zara_sales_user,
    zara_finance_user,
    zara_budget_user,
    zara_manager_user,
    zara_business_admin,
    zara_customer_portal_user;


-- ============================================================================
-- 13) DEFAULT PRIVILEGES FOR FUTURE OBJECTS
-- Assumes postgres owns the new objects in your notebook.
-- ============================================================================

-- sa_online_retail
ALTER DEFAULT PRIVILEGES FOR ROLE postgres IN SCHEMA sa_online_retail
GRANT SELECT, INSERT, UPDATE, DELETE, TRUNCATE ON TABLES TO dw_data_engineer, svc_etl_loader;

ALTER DEFAULT PRIVILEGES FOR ROLE postgres IN SCHEMA sa_online_retail
GRANT SELECT ON TABLES TO dw_analytics_engineer, dw_data_quality_engineer;

ALTER DEFAULT PRIVILEGES FOR ROLE postgres IN SCHEMA sa_online_retail
GRANT USAGE, SELECT, UPDATE ON SEQUENCES TO dw_data_engineer, svc_etl_loader;

ALTER DEFAULT PRIVILEGES FOR ROLE postgres IN SCHEMA sa_online_retail
GRANT USAGE, SELECT ON SEQUENCES TO dw_analytics_engineer, dw_data_quality_engineer;


-- sa_offline_retail
ALTER DEFAULT PRIVILEGES FOR ROLE postgres IN SCHEMA sa_offline_retail
GRANT SELECT, INSERT, UPDATE, DELETE, TRUNCATE ON TABLES TO dw_data_engineer, svc_etl_loader;

ALTER DEFAULT PRIVILEGES FOR ROLE postgres IN SCHEMA sa_offline_retail
GRANT SELECT ON TABLES TO dw_analytics_engineer, dw_data_quality_engineer;

ALTER DEFAULT PRIVILEGES FOR ROLE postgres IN SCHEMA sa_offline_retail
GRANT USAGE, SELECT, UPDATE ON SEQUENCES TO dw_data_engineer, svc_etl_loader;

ALTER DEFAULT PRIVILEGES FOR ROLE postgres IN SCHEMA sa_offline_retail
GRANT USAGE, SELECT ON SEQUENCES TO dw_analytics_engineer, dw_data_quality_engineer;


-- bl_cl
ALTER DEFAULT PRIVILEGES FOR ROLE postgres IN SCHEMA bl_cl
GRANT SELECT, INSERT, UPDATE, DELETE, TRUNCATE ON TABLES TO dw_data_engineer, svc_etl_loader;

ALTER DEFAULT PRIVILEGES FOR ROLE postgres IN SCHEMA bl_cl
GRANT SELECT ON TABLES TO dw_analytics_engineer, dw_data_quality_engineer;

ALTER DEFAULT PRIVILEGES FOR ROLE postgres IN SCHEMA bl_cl
GRANT USAGE, SELECT, UPDATE ON SEQUENCES TO dw_data_engineer, svc_etl_loader;

ALTER DEFAULT PRIVILEGES FOR ROLE postgres IN SCHEMA bl_cl
GRANT USAGE, SELECT ON SEQUENCES TO dw_analytics_engineer, dw_data_quality_engineer;

ALTER DEFAULT PRIVILEGES FOR ROLE postgres IN SCHEMA bl_cl
GRANT EXECUTE ON FUNCTIONS TO dw_platform_admin, dw_data_engineer, svc_etl_loader;

ALTER DEFAULT PRIVILEGES FOR ROLE postgres IN SCHEMA bl_cl
GRANT EXECUTE ON ROUTINES TO dw_platform_admin, dw_data_engineer, svc_etl_loader;


-- bl_3nf
ALTER DEFAULT PRIVILEGES FOR ROLE postgres IN SCHEMA bl_3nf
GRANT SELECT, INSERT, UPDATE, DELETE, TRUNCATE ON TABLES TO dw_data_engineer, svc_etl_loader;

ALTER DEFAULT PRIVILEGES FOR ROLE postgres IN SCHEMA bl_3nf
GRANT SELECT ON TABLES TO dw_analytics_engineer, dw_data_quality_engineer;

ALTER DEFAULT PRIVILEGES FOR ROLE postgres IN SCHEMA bl_3nf
GRANT USAGE, SELECT, UPDATE ON SEQUENCES TO dw_data_engineer, svc_etl_loader;

ALTER DEFAULT PRIVILEGES FOR ROLE postgres IN SCHEMA bl_3nf
GRANT USAGE, SELECT ON SEQUENCES TO dw_analytics_engineer, dw_data_quality_engineer;


-- bl_dm
ALTER DEFAULT PRIVILEGES FOR ROLE postgres IN SCHEMA bl_dm
GRANT SELECT, INSERT, UPDATE, DELETE, TRUNCATE ON TABLES TO dw_data_engineer, svc_etl_loader;

ALTER DEFAULT PRIVILEGES FOR ROLE postgres IN SCHEMA bl_dm
GRANT SELECT ON TABLES TO dw_analytics_engineer, dw_bi_analyst, dw_data_quality_engineer;

ALTER DEFAULT PRIVILEGES FOR ROLE postgres IN SCHEMA bl_dm
GRANT USAGE, SELECT, UPDATE ON SEQUENCES TO dw_data_engineer, svc_etl_loader;

ALTER DEFAULT PRIVILEGES FOR ROLE postgres IN SCHEMA bl_dm
GRANT USAGE, SELECT ON SEQUENCES TO dw_analytics_engineer, dw_bi_analyst, dw_data_quality_engineer;


-- reporting
ALTER DEFAULT PRIVILEGES FOR ROLE postgres IN SCHEMA reporting
GRANT SELECT, INSERT, UPDATE, DELETE ON TABLES TO dw_data_engineer, dw_analytics_engineer;

ALTER DEFAULT PRIVILEGES FOR ROLE postgres IN SCHEMA reporting
GRANT SELECT ON TABLES TO dw_bi_analyst, zara_sales_user, zara_finance_user, zara_budget_user, zara_manager_user, zara_business_admin, zara_customer_portal_user;

ALTER DEFAULT PRIVILEGES FOR ROLE postgres IN SCHEMA reporting
GRANT USAGE, SELECT, UPDATE ON SEQUENCES TO dw_data_engineer, dw_analytics_engineer;

ALTER DEFAULT PRIVILEGES FOR ROLE postgres IN SCHEMA reporting
GRANT USAGE, SELECT ON SEQUENCES TO dw_bi_analyst;


-- analytics_sandbox
ALTER DEFAULT PRIVILEGES FOR ROLE postgres IN SCHEMA analytics_sandbox
GRANT SELECT, INSERT, UPDATE, DELETE ON TABLES TO dw_analytics_engineer;

ALTER DEFAULT PRIVILEGES FOR ROLE postgres IN SCHEMA analytics_sandbox
GRANT USAGE, SELECT, UPDATE ON SEQUENCES TO dw_analytics_engineer;


-- dq_audit
ALTER DEFAULT PRIVILEGES FOR ROLE postgres IN SCHEMA dq_audit
GRANT SELECT, INSERT, UPDATE, DELETE ON TABLES TO dw_data_quality_engineer, dw_platform_admin;

ALTER DEFAULT PRIVILEGES FOR ROLE postgres IN SCHEMA dq_audit
GRANT USAGE, SELECT, UPDATE ON SEQUENCES TO dw_data_quality_engineer, dw_platform_admin;


-- ============================================================================
-- 14) OPTIONAL LOGIN USER EXAMPLES
-- Uncomment only if you want test users.
-- ============================================================================

/*
DO $$
BEGIN
    IF NOT EXISTS (SELECT 1 FROM pg_roles WHERE rolname = 'sumeyye_de') THEN
        CREATE ROLE sumeyye_de LOGIN PASSWORD 'change_me';
        GRANT dw_data_engineer TO sumeyye_de;
    END IF;

    IF NOT EXISTS (SELECT 1 FROM pg_roles WHERE rolname = 'sumeyye_ae') THEN
        CREATE ROLE sumeyye_ae LOGIN PASSWORD 'change_me';
        GRANT dw_analytics_engineer TO sumeyye_ae;
    END IF;

    IF NOT EXISTS (SELECT 1 FROM pg_roles WHERE rolname = 'sumeyye_bi') THEN
        CREATE ROLE sumeyye_bi LOGIN PASSWORD 'change_me';
        GRANT dw_bi_analyst TO sumeyye_bi;
    END IF;
END $$;
*/


-- ============================================================================
-- 15) QUICK CHECKS
-- ============================================================================

SELECT rolname
FROM pg_roles
WHERE rolname IN (
    'dw_platform_admin',
    'dw_data_engineer',
    'dw_analytics_engineer',
    'dw_bi_analyst',
    'dw_data_quality_engineer',
    'svc_etl_loader',
    'zara_sales_user',
    'zara_finance_user',
    'zara_budget_user',
    'zara_manager_user',
    'zara_business_admin',
    'zara_customer_portal_user'
)
ORDER BY rolname;

SQL

In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -P pager=off <<'SQL'

-- ============================================================================
-- RLS SETUP FOR CUSTOMER-LEVEL ACCESS
-- Purpose:
-- Show that customers can only see their own transactions by customer_surr_id.
-- This uses a session variable:
--     SET app.current_customer_surr_id = '123';
-- ============================================================================


-- 1) Customer portal role can use reporting schema
GRANT USAGE ON SCHEMA reporting TO zara_customer_portal_user;


-- 2) Enable RLS on fact table
ALTER TABLE bl_dm.fct_transactions ENABLE ROW LEVEL SECURITY;


-- 3) Drop old policy if you want to rerun safely
DROP POLICY IF EXISTS p_customer_portal_select ON bl_dm.fct_transactions;


-- 4) Create policy
CREATE POLICY p_customer_portal_select
ON bl_dm.fct_transactions
FOR SELECT
TO zara_customer_portal_user
USING (
    customer_surr_id = current_setting('app.current_customer_surr_id', true)::BIGINT
);


-- 5) Optional: force RLS for stronger enforcement
-- If table owner bypass behavior worries you, uncomment this:
-- ALTER TABLE bl_dm.fct_transactions FORCE ROW LEVEL SECURITY;


-- 6) Create secure customer-facing view
DROP VIEW IF EXISTS reporting.v_customer_orders_secure;

CREATE VIEW reporting.v_customer_orders_secure AS
SELECT
    date_id,
    customer_surr_id,
    product_surr_id,
    promotion_surr_id,
    delivery_surr_id,
    engagement_surr_id,
    employee_surr_id,
    transaction_src_id,
    transaction_dt,
    quantity,
    unit_price,
    total_sales,
    discount_applied,
    insert_dt
FROM bl_dm.fct_transactions;


-- 7) Grant only the secure view to customer portal role
GRANT SELECT ON reporting.v_customer_orders_secure TO zara_customer_portal_user;


-- 8) Optional hardening:
-- revoke direct select on fact table from customer role if previously granted
REVOKE ALL ON bl_dm.fct_transactions FROM zara_customer_portal_user;

SQL

In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -P pager=off <<'SQL'

-- TEST

SET app.current_customer_surr_id = '101';
SELECT * FROM reporting.v_customer_orders_secure;


SQL

In [ ]:
%%bash
set -e
DB="retail_dw"

sudo -u postgres psql -d "$DB" -P pager=off <<'SQL'

SET ROLE zara_customer_portal_user;
SET app.current_customer_surr_id = '101';

SELECT customer_surr_id, COUNT(*)
FROM reporting.v_customer_orders_secure
GROUP BY customer_surr_id;

SQL